## Variant region list:
- load the file of all sequences/oligo names (80215)
- make a list of these with a column for the name, sequence, type (if variant: ALT, if reference: REF, if region: region, control (true/false), gene association (all information about the gene (i.e. name with | and ~RAS ...))
- example table is shown [here](https://docs.google.com/spreadsheets/d/1YfepW_nv14024v8KwveGaJTbcBX6pRKRUohl7jZhltY/edit#gid=0)
### Questions:
- Some rows have ALT_, REF_, what does it mean if it has no REF_ and ALT_ is it a region then?
  - (1) id (not too long), (2) sequence, (3) category,  (4) class (test, variant (pos/neg) control), (5) source ? for our design ("candidate CRE nearby 536 cardiac, neuro, cava and random genes"), (6) ref_sequence ? CLEA controlls? what with the sequences (general controlls we took them actually from hg19, I think?,  (7) chrom (NA possible), (8) chrom_start (NA possible), (9) chrom_end (NA possible),  (10) variant_class (NA possible), (11) variant_pos (NA possible), (12) SPDI (NA possible), (13) allele (NA possible),  (14) info (free form)

### Process:
- We identified regions near TSS of genes (we looked for variants within these regions (centered in these regions (100bp from the center)))
- Total number of different regions: 80215
- First table:
  - for each row: one header with the informations it has ()


In [28]:
# imports 
import pandas as pd 
from Bio import SeqIO
# use the config file in yaml format
import yaml
import os
import re
import gzip # gzipped files
import hashlib


# load helpful functions
import sys
sys.path.append('/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/00_helpful_functions')
import helpful_functions as hf
# # reload helpful_functions
# from importlib import reload
# reload(hf)



# read config
# config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/config/config.yaml"
config_path = "../../../80K_analysis/global80K_config.yaml"
# config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)

In [29]:
def write_fasta(sequence_df, output_path, header=['header', 'sequence']):
    """
    Write the fasta file with the header and sequence
    """
    with open(output_path, 'w') as f:
        for index, row in sequence_df.iterrows():
            f.write('>' + row[header[0]] + '\n' + row[header[1]] + '\n')
    return True

In [31]:
# load the data
design_fasta = '/fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa'
design_fasta = 'resources/design_no_duplicates_sequence_and_header.fa'
design_fasta = config['files']['final_design']['design_fasta']

# read the fasta file with the sequences and prepare a tsv with header and sequence using biopython
records = list(SeqIO.parse(design_fasta, "fasta"))
design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

design_df['header'] = header
design_df['sequence'] = sequence

# label is the string in front of the first ":"
design_df['label'] = design_df['header'].str.split(':').str[0]
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

design_df

,header,sequence,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK


### Playground
- check number of specific labels in design


In [7]:
neuro_controls = ['C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK', 'C_positive_neuron_CD']
neuro_controls_design_df = design_df[design_df['label'].isin(neuro_controls)]
print(neuro_controls_design_df.shape[0])
neuro_controls_design_df['label'].value_counts()

728


label
C_negative_neuron_MK    222
C_negative_neuron_NP    217
C_positive_neuron_NP     99
C_positive_neuron_MK     96
C_positive_neuron_CD     94
Name: count, dtype: int64

### Trying to find unmerged and merged headers (does not work properly)

In [14]:
# read unmerged file
unmerged_file = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design.fa"
unmerged_file = config['files']['final_design']['design_with_duplicates']

# if file is gzipped, use the following
if unmerged_file.endswith(".gz"):
    file_handler = gzip.open(unmerged_file, "rt")
else:
    file_handler = open(unmerged_file, "r")

records = list(SeqIO.parse(file_handler, "fasta"))
unmer_design_df = pd.DataFrame(columns=['header', 'sequence'])
header = [] 
sequence = []
for record in records:
    header.append(record.id)
    sequence.append(str(record.seq))

unmer_design_df['header'] = header
unmer_design_df[col_sequence] = sequence
unmer_design_df["unmerged"] = True
unmer_design_df = unmer_design_df[['header', 'unmerged']]
unmer_design_df.head()

,header,unmerged
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,True


In [15]:
# merge unmer_design_df and design_df on header
# the lines without label are merged
complete_design_df = design_df.merge(unmer_design_df, on="header", how="left")
complete_design_df
# number of na values in "label"
merged_header_df = complete_design_df[complete_design_df.unmerged.isna()] #3278
merged_header_df.shape
merged_header_list = merged_header_df.header.to_list()
merged_header_list[100]
# example 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E1378368~NRAS|ENSG00000213281.5|EH38E1378368_rev_tile1-1'

'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E1378368~NRAS|ENSG00000213281.5|EH38E1378368_rev_tile1-1'

In [16]:
merged_header_df

,header,sequence,label,unmerged
708,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTAAGGAAGGGAGGGAGGGAGGGAGCGATCCCT...,cardiac_neuro_cava_random,NaN
709,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTGGACCACCTCCACCAACTGTCAGCTCACATC...,cardiac_neuro_cava_random,NaN
710,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACTGTCCTTCCTGAGGCCTCCAGCATTATTG...,cardiac_neuro_cava_random,NaN
711,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTTTACTTATTTATTTATTTTTTGAGACAGGGT...,cardiac_neuro_cava_random,NaN
712,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACCCCATAGTATGCCTCCCTCCCTCTCTCCT...,cardiac_neuro_cava_random,NaN
...,...,...,...,...
77523,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTGGTCGCCCATTCCCAAGCTCCCACCTTGACG...,C_positive_heart_AB,NaN
77524,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTGGGCCCAACCAAGGAACCTGGCCTGGTCTCA...,C_positive_heart_AB,NaN
77525,C_positive_heart_AB:FLNA-ENST00000420627.5:Oli...,AGGACCGGATCAACTCCACTCGCCCTGAGTCCACACAAGTTCCTGG...,C_positive_heart_AB,NaN
77526,C_positive_heart_AB:FLNA-ENST00000360319.9:Oli...,AGGACCGGATCAACTGTAAAATTGCCCAGGAGCCCGGGACGGGTGC...,C_positive_heart_AB,NaN


In [17]:
# count number of rows per label
# split based on "|" number in order to find pattern
merged_header_df["pipe_count"] = merged_header_df['header'].str.count(r'\|')
merged_header_df_cardiac = merged_header_df[merged_header_df["label"] == "cardiac_neuro_cava_random"]
merged_header_df_cardiac #1717 / 3278 => 52%
merged_header_df_cardiac.pipe_count.value_counts()
# pipe_count
# 8     596
# 7     471
# 4     341
# 10    309

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-139/ipykernel_2189296/2263432361.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_header_df["pipe_count"] = merged_header_df['header'].str.count(r'\|')


pipe_count
8     596
7     471
4     341
10    309
Name: count, dtype: int64

### Split the number of headers from cardiac_neuro_cava_random label by the number of pipes to find patterns
- found patterns
  - alternate oligos: ALT_ (one snp each)
  - reference oligos: REF_ (some are merged) also regions
  - regions: no REF_ or ALT_ (some are merged)

#### Split of cardiac_neuro_cava_random by number of pipes:
- pipe_count
- 8     596
- 7     471
- 4     341
- 10    309

##### 8 pipes: (596)
- all alt
- do all have same number of "_" 
- do all have same number and position of "~"
- do all end with a variant
- example: # 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

In [ ]:
# 8: 596 rows
cardiac_8_pipe_df = merged_header_df_cardiac[merged_header_df_cardiac["pipe_count"] == 8]
cardiac_8_pipe_df # 596

# count number of "_"
cardiac_8_pipe_df.header.str.count(r"_")
# count number and look at position of "~"

# end pattern matches variant patter?
cardiac_8_pipe_list = cardiac_8_pipe_df["header"].to_list()
cardiac_8_pipe_list[0]


'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

#### Short cut ideas:
- filter all header with variants: |(last pipe)<num>-<num>
  - Pattern ? if variant is there there is ALT in the beginning
- Find variant pattern -> its variant
- Find reference pattern -> its reference
- Find nothing -> its region
- check the numbers
  - Number control sequences: 6275
  - Number tested sequences: 73940
```
-------Design Number Summary (for cardiac_neuro_cava_random group) --------
Number of references: 18582
Number of alternative sequences: 46458
Number of regions (without variants): 8900
```
- contorls:
  - looking for variant controls (especially positive ones)
    - control labels with "ALT" in header: `cat /home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/controll_header.tsv | grep -v "C_positive_neuron_CD"| grep "ALT" | awk -F ':' '{print $1}' | sort | uniq `
      - C_positive_heart_CAD
      - GC_Atrial_fib
      - GC_Kircher
      - GC_Liang
      - GC_Mendelian_variants
      - GC_Mohlke
      - GC_Selvarajan
      - MK
  - numbers of "ALT" in control header: 656 `cat /home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/controll_header.tsv | grep -v "C_positive_neuron_CD"| grep "ALT" | wc -l` (removed `C_positive_neuron_CD` because these are element controls)



In [ ]:
# helpful functions for checking reference, variants and regions
def check_variant(header, check_controls=False):
    """Checks if a header is a variant header"""
    # get the last part of the header
    last_part = header.split('|')[-1]
    # check if the last part if it matches the regex [\d]+-[\d]+
    if re.match(r'[\dA-Z]+-[\d]+', last_part): # is sufficient, because all these headers have ALT in their name
        return True
    else:
        return False

def check_reference(header):
    """Checks if a header is a reference header"""
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return True
    else:
        return False

def check_region(header):
    """
    Checks if a header is a region header
    A region header is here defined as a header without ALT_ or REF_ after the first ":" 
    """
    # check after the label if REF_ is in the header
    non_label_header = header.split(':')[1]
    if "REF_" in non_label_header.split('|')[0]:
        return False
    elif "ALT_" in non_label_header.split('|')[0]:
        return False
    else:
        return True

def check_region_variant_reference_numbers(header_list, check_controls=False):
    """Checks if a header is a variant, reference or region header"""
    ref_counter = 0
    var_counter = 0
    region_counter = 0
    unknown_counter = 0
    for header in header_list:
        if header.split(':')[0] != 'cardiac_neuro_cava_random':
            continue
            # TODO: add way to check if sequence is a alternative in controls
            if check_controls:
                continue
        if check_variant(header):
            var_counter += 1
            header_type = 'ALT'
        elif check_reference(header):
                ref_counter += 1
                header_type = 'REF'
        elif check_region(header):    
            region_counter += 1
            header_type = 'region'
        else:
            header_tpye = 'unknown'
            unknown_counter += 1
            print(f'Found unknown header: {header}')
    return ref_counter, var_counter, region_counter

def create_fasta_df_from_one_line_sequence_fasta(fasta_path, filter_cardiac=False):
    """Creates a dataframe with header and sequence from a fasta file which has one line per sequence"""
    records = list(SeqIO.parse(fasta_path, "fasta"))
    design_df = pd.DataFrame(columns=['header', col_sequence])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df[col_sequence] = sequence

    if filter_cardiac:
        design_df['label'] = design_df['header'].str.split(':').str[0]
        design_df = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

    return design_df

def get_gene_name_from_header(header):
    """Identify the gene name (e.g. MYH6) from the header: sequence after first ":" and before first "|" then crop the sequence after the first "_" if it exists"""
    gene_name = header.split(':')[1].split('|')[0]
    if '_' in gene_name:
        gene_name = gene_name.split('_')[1]
    return gene_name

def write_variants_fasta(design_fasta_path, fasta_out_directory):
    """Write all variants to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=False)
    design_df['is_variant'] = design_df['header'].apply(check_variant)
    design_df = design_df[design_df['is_variant'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_variants_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_region_fasta(design_fasta_path, fasta_out_directory):
    """Write all regions to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_region'] = design_df['header'].apply(check_region)
    design_df = design_df[design_df['is_region'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_regions_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

def write_reference_fasta(design_fasta_path, fasta_out_directory):
    """Write all references to a fasta file with the header and sequence"""
    design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta_path, filter_cardiac=True)
    design_df['is_reference'] = design_df['header'].apply(check_reference)
    design_df = design_df[design_df['is_reference'] == True]
    # write the fasta file
    output_path = os.path.join(fasta_out_directory, f'identified_references_{design_df.shape[0]}.fa')
    with open(output_path, 'w') as f:
        for index, row in design_df.iterrows():
            f.write('>' + row['header'] + '\n' + row['sequence'] + '\n')
    return design_df

In [ ]:
# iterate all headers and get the gene name of interest
gene_names = set()
not_cardiac = 0
header_list = design_df['header'].to_list()
for hdr in header_list:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        not_cardiac += 1
        continue
    # put gene name into set
    gene_name = get_gene_name_from_header(hdr)
    gene_names.add(gene_name)

# check the number of the gene_name set
print(f'Number of unique "associated" genes: {len(gene_names)}')  # => Found all (same number as in summary presentation) 525 "associated" genes

Number of unique "assiciated" genes: 525


In [ ]:
print('Number control sequences: %s'%(not_cardiac)) # 6275

num_cardiac = design_df.shape[0] - not_cardiac # 80215 - 6275 => 73940 (can be checked with grep and are the correct numbers)
print('Number tested sequences: %s'%(num_cardiac))


Number control sequences: 6275
Number tested sequences: 73940


In [ ]:

ref_counter, var_counter, region_counter = check_region_variant_reference_numbers(header_list)

if ref_counter + var_counter + region_counter != num_cardiac:
    raise ValueError("The numbers do not match")
else:
    print("The numbers of tested sequences and region_variant_reference function match")

print('-------Design Number Summary --------\nNumber of references: %s\nNumber of alternative sequences: %s\nNumber of regions (without variants): %s'%(ref_counter, var_counter, region_counter)) # 02.04.2024: 18582 + 46458 + 8900 = 73940

# we want: 28000 cCREs we got 8900
design_fasta_file = config['files']['final_design']['design_fasta']
output_fasta_directory = 'resources/'
# write_variants_fasta(design_fasta_file, output_fasta_directory)
# write_reference_fasta(design_fasta_file, output_fasta_directory)
# write_region_fasta(design_fasta_file, output_fasta_directory)
# Problem: we are not sure about the exact numbers and the formats might be different
  # - how do I check if the number of variants (currently 46458) is correct?

The numbers of tested sequences and region_variant_reference function match
-------Design Number Summary --------
Number of references: 18582
Number of alternative sequences: 46458
Number of regions (without variants): 8900


In [ ]:
# read all headers and check for the regex pattern of variant info after the last pipe ("|")
# count the number of found variant patterns
# if the pattern is found, check if it has ALT_ after the first ":"
# count the number of found variant pattersn with ALT_
var_count = 0
var_count_alt = 0
for hdr in header_list:
    if hdr.split(':')[0] != 'cardiac_neuro_cava_random':
        continue
    if re.match('ALT_', hdr.split(':')[1]):
        # if it matches, count it
        var_count_alt += 1
        # print(hdr)
        # break
        # get the last part of the header
        last_part = hdr.split('|')[-1]
        # check if the last part if it matches the regex [\d]+-[\d]+
        if re.match(r'[\dA-Z]+-[\d]+', last_part):
            var_count += 1
        else:
            print(hdr)
        
# check if the number of variants is the same as the number of variants with ALT_
if var_count == var_count_alt:
    print("All variants have ALT_ in the header")

All variants have ALT_ in the header


In [ ]:
# list of all unique labels
unique_labels = design_df['label'].unique().tolist()
unique_labels

['cardiac_neuro_cava_random',
 'GC_Atrial_fib',
 'GC_Liang',
 'GC_Selvarajan',
 'GC_Mohlke',
 'GC_Kircher',
 'GC_Mendelian_variants',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Hon',
 'GC_Vista',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_negative_neuron_NP',
 'C_positive_heart_MK',
 'C_positive_neuron_CD',
 'C_positive_neuron_MK',
 'C_positive_neuron_NP',
 'C_positive_heart_AB',
 'C_SLEA',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled',
 'MK']

In [ ]:
# investigate the variants
for i in range(40, 48):
    print(design_df['header'].tolist()[i])


cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779571_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779574_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779580_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779583_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779643_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779653_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779655_fwd_tile1-1
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779714_fwd_tile1-1


In [ ]:
for lable in unique_labels:
    if lable == 'cardiac_neuro_cava_random':
        continue
    df = design_df[design_df['label'] == lable]
    print(lable)
    print(df['header'].str.count('\|').value_counts())

GC_Atrial_fib
header
2    34
4    11
Name: count, dtype: int64
GC_Liang
header
1    16
Name: count, dtype: int64
GC_Selvarajan
header
1    241
2    105
4     10
3      8
Name: count, dtype: int64
GC_Mohlke
header
4     26
12     5
8      3
Name: count, dtype: int64
GC_Kircher
header
400    203
Name: count, dtype: int64
GC_Mendelian_variants
header
2    161
1     48
Name: count, dtype: int64
C_positive_heart_CAD
header
0    97
Name: count, dtype: int64
GC_Cort_Chengyu
header
3    184
2      1
Name: count, dtype: int64
GC_GABA_Chengyu
header
3    85
Name: count, dtype: int64
GC_Glut_Chengyu
header
3    40
Name: count, dtype: int64
GC_Hon
header
1    6
Name: count, dtype: int64
GC_Vista
header
1    256
Name: count, dtype: int64
GC_DNase_positive
header
0    41
Name: count, dtype: int64
GC_DNase_negative_brain
header
0    15
Name: count, dtype: int64
GC_DNase_negative_blood
header
0    15
Name: count, dtype: int64
C_negative_heart_MK
header
0    243
Name: count, dtype: int64
C_negative_neu

### Metadata file: [document](https://docs.google.com/document/d/1ThHgLjMnS2r-vv_4ZHHO9y4K1Qbc2S2WKKUDw__iYXU/edit)
- I want to have a table of id, sequence, category, class, source, ref_sequence, chrom, chrom_start, chrom_end, variant_class, variant_pos, SPDI, allele, info

In [41]:
import os
import pandas as pd
import numpy as np
focusing_label = 'cardiac_neuro_cava_random'

variant_control_groups = ["GC_Selvarajan", "GC_Kircher", "GC_Mendelian_variants", "C_positive_heart_CAD", "GC_Atrial_fib", "GC_Mohlke", "GC_Liang"]
variant_groups = ["cardiac_neuro_cava_random"] + variant_control_groups

dnase_control_groups = ['GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled']

variant_map = pd.read_csv(config['files']['final_design']['final_design_variant_region_map'], sep="\t")
variant_map = pd.read_csv(config['files']['final_design']['variant_table_deduplicated'], sep="\t")
variant_map_reference_sequences = variant_map['REF_ID'].to_list()
variant_map_alternative_sequences = variant_map['ALT_ID'].to_list()

def get_category(header):
    """Get the category of the header
    if C_SLEA in tmp_label => synthetic
    elif scramble in header or is DNase group => scrambled
    elif ref_ or alt_ in header => variant 
    else element
    """
    
    label = get_label(header)
    if label in synthetic_control_groups:
        return 'synthetic'
    
    elif label in dnase_control_groups:
        # all DNase controls used coordinates of hg18
        return 'scrambled'
    
    elif 'scramble' in header:
        # Check if header is scrambled
        # Info: scrambled is in C_negative_neuron_NP and scramble MK 
        # Note: Other cases are not checked with this function
        return 'scrambled'
    else:
        if label in variant_groups and ref_or_alt_in_header(header):
            return 'variant'
        else:
            return 'element'


def get_class(header):
    """
    Get the class of the header
    A class is according to the IGVF metadata format: test, variant positive control, variant negative control, 
          element active control or element inactive control
    """
    # known pattern for cardiac_neuro_cava_random: all are "test"
    if 'cardiac_neuro_cava_random' in header:
        return 'test'
    elif is_positive_control(header):
        return get_positive_class(header)
    elif is_negative_control(header):
        return get_negative_class(header)
    else:
        return 'NA'
        
    
def get_source(header):
    """Get the source of the header"""
    # only known pattern: for cardiac_neuro_cava_random: "candidate CRE nearby 536 cardiac, neuro, cava and random genes"
    if 'cardiac_neuro_cava_random' in header:
        return 'candidate CRE nearby cardiac, neuro, cava and random genes'
    if 'GC_' in header:
        label = header.split(':')[0]
        return 'IGVF general controls (%s)'%(label)
    else:
        return 'NA'
    
    
def create_path(path):
    """Check if path exists if not create it"""
    dir_path = os.path.dirname(path)
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        
        import gzip
from Bio import SeqIO


def create_fasta_df_from_one_line_sequence_fasta(fasta_path, filter_cardiac=False):
    """Creates a dataframe with header and sequence from a fasta file which has one line per sequence"""
    # if file gzip:
    if fasta_path.split('.')[-1] == 'gz':
        with gzip.open(fasta_path, "rt") as fasta_handle:
            records = list(SeqIO.parse(fasta_handle, "fasta"))
    else:        
        records = list(SeqIO.parse(fasta_path, "fasta"))
    
    design_df = pd.DataFrame(columns=['header', 'sequence'])
    header = [] 
    sequence = []
    for record in records:
        header.append(record.id)
        sequence.append(str(record.seq))

    design_df['header'] = header
    design_df['sequence'] = sequence

    if filter_cardiac:
        design_df['label'] = design_df['header'].str.split(':').str[0]
        design_df = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

    return design_df


def get_info(header):
    """Set information: e.g. wrong coordinates used"""
    label = get_label(header)
    # wrong coordinates used
    if label in wrong_coordinates_groups:
        return '%s;wrong genome build (GRCh37) used for these sequences'%(label)
    else:
        return '%s'%(label)
    
    
def get_label(header):
    return header.split(':')[0]


def is_ref_sequence(header):
    """Check if header is in the REF_ID of the used variant map"""
    # assuming REF_ID holds label information need to split header first
    return header in variant_map_reference_sequences


def is_alt_sequence(header):
    """Check if header is in the ALT_ID of the used variant map"""
    # assuming ALT_ID holds label information need to split header first
    return header in variant_map_alternative_sequences


def ref_or_alt_in_header(header):
    """Checks if "ref_" or "_alt" is in the header"""
    if is_ref_sequence(header) or is_alt_sequence(header):
        return True
    return False


def ref_alt_decision(header):
    """
    returns for each variant row in the design if ref or alt
    """
    if is_ref_sequence(header):
        return 'ref'
    elif is_alt_sequence(header):
        return 'alt'
    else:
        return 'NA'


# def is_control_variant(header):
#     """Check entries of variant region map and returns if the header is a control"""
#     # first find the control groups meant to have variant controls (found from variant_region_map.tsv.gz in final_design/results/final_design/)
#     return get_label(header) in variant_control_groups 


### Decide if negative or positive controls
# important note: negative_control_groups and positive_control_groups are important and need to be created before 

def get_negative_class(header):
    if ref_or_alt_in_header(header):
        return 'variant negative control'
    return 'element inactive control'


def is_negative_control(header):
    """Checks if header is negative control"""
    label = get_label(header)
    if label in negative_control_groups:
        return True
    return False


def get_positive_class(header):
    if ref_or_alt_in_header(header):
        return 'variant positive control'
    return 'element active control'


def is_positive_control(header):
    """Checks if header is positive control"""
    label = get_label(header)
    if label in positive_control_groups:
        return True
    return False


def get_control_reference(row):
    """Get reference sequence for control"""
    if row['ref_sequence'] != 'NA':
        return row['ref_sequence']
    else:
        # Add additional specification if needed here
        return 'GRCh38'
    

def get_region_match_name(header): #TODO: change functionality header matching is more complex
    """
    Get the region match name from the header
    1. split by ':' and take the second part [1]
    2. split by '_' and take the second part [1]
    """
    if header.split(':')[0] == 'cardiac_neuro_cava_random':
        if 'ALT_' in header or 'REF_' in header:
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
                header = header.replace('~', ',')
            # easy types: 
            # cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            # cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            return ':'.join(header.split(':')[1:]).split('_')[1]
        else: # cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778480_fwd_tile1-1
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:APOL2|ENSG00000128335.14|EH38E3478577~APOL4|ENSG00000100336.18|EH38E3478577_rev_tile1-1
                header = header.replace('~', ',')
            return ':'.join(header.split(':')[1:]).split('_')[0]
    else:
        return header
    

def parse_regions_from_header(row):
    """
    Parse regions from header
    Only possible for the headers you see below
    Not possible for synthetic and scrambled sequences
    22.04: läuft durch getestet nur auf underscore_parsable_headers
    """
    if row[col_category] == 'synthetic' or row[col_category] == 'scrambled':
        return row
    header = row['header']
    label = get_label(header)
    underscore_parsable_headers = ['C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK', 'C_positive_heart_MK', 'C_negative_heart_MK']
    # C_negative_neuron_NP:GW18_PFC_ABC_chr15_89400286_89400556_0.830617698776558
    # C_positive_neuron_NP:GW18_PFC_ABC_chr5_68730744_68731014_6.25979643705865
    # C_positive_neuron_MK:tile_37639_chr6_112501812_112502081_reference_0.828839741594896
    # C_positive_neuron_MK:tile_37639_chr6_112501812_112502081_reference_0.828839741594896
    # C_positive_neuron_MK:tile_37243_chr6_97306567_97306836_A_T_149_0.750659896963247
    NP_group = ['C_negative_neuron_NP', 'C_positive_neuron_NP'] # underscore but without variants
    MK_group = ['C_negative_neuron_MK', 'C_positive_heart_MK', 'C_positive_neuron_MK',  'C_negative_neuron_MK'] # underscore ref: _reference_ alt: _char_char_position
    double_colon_variants = ['positive_neuron_CD']
    chrom,ref, alt = "", "", "" 
    start, end, variant_pos = 0, 0, 0
    if label in underscore_parsable_headers:
        # split header at '_chr'
        pre_position = header.split('_chr')
        pre_position = pre_position[1].split('_')
        if len(pre_position) < 3: # unexpected header: throw Value error
            raise ValueError('Unexpected header shape')
        # get chrom start end (group-specific)
        row[col_chr] = f"chr{pre_position[0]}"
        row[col_end] = int(pre_position[2])
        if label in MK_group:
            row[col_start] = int(pre_position[1]) - 1 # turn into 0-based
        elif label in NP_group:
            row[col_start] = int(pre_position[1])
        if len(pre_position) == 7: # variant in header
            row[col_variant_class] = 'SNP'
            row[my_col_ref_base] = pre_position[3]
            row[my_col_alt_base] = pre_position[4]
            row[col_variant_pos] = int(pre_position[5])
    return row
    

def add_design_style_header(header):
    """gets a design style header (with * instead of > and ~ instead of ,) and returns a matching header"""
    header = header.replace('>', '*')
    header = header.replace(',', '~')
    return header


def add_matching_header(header):
    """gets header with > and , and replaces it with * and ~"""
    header = header.replace('*', '>')
    header = header.replace('~', ',')
    return header
        
        

In [42]:
# load fasta (header, sequence) and add columns with NA values 
# design_fasta = config['files']['reference']
design_fasta = config['files']['final_design']['design_with_duplicates']
design_fasta = config['files']['final_design']['design_fasta']
pre_metadata_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta, filter_cardiac=False)

# column names
col_name = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class' # SNP
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'



# add temporary label column
pre_metadata_df['tmp_label'] = pre_metadata_df['header'].apply(get_label)

# which controls are considered positive and negative
controls = pre_metadata_df[pre_metadata_df['tmp_label'] != focusing_label]
# get all unique values of the tmp_label column 
all_control_groups = controls['tmp_label'].unique()

positive_control_groups = ['C_positive_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_CD']
synthetic_control_groups = ['C_SLEA']
wrong_coordinates_groups = [group for group in all_control_groups if "dnase" in group.lower()] # add information that these used wrong coordinates

negative_control_groups = [group for group in all_control_groups if not group in positive_control_groups]


# add empty columns name, category, class, source, ref_sequence, chrom, chrom_start, chrom_end, variant_class, variant_pos, SPDI, allele, info
pre_metadata_df[col_name] = 'NA' # will be the header: later: genome coordinates and for shuffled,scrambled and synthetic a clear prefix
pre_metadata_df[col_category] = 'NA' # for cardiac_neuro_cava_random: if "ALT_" or "REF_" in header, then "variant", else "element" otherwise leave NA
pre_metadata_df[col_class] = 'NA' # if cardiac_neuro_cava_random: "test", else leave NA (TODO: find pattern in controls for variants and elements and positive and negative controls)
pre_metadata_df[col_source] = 'NA' # if cardiac_neuro_cava_random: "candidate CRE nearby 536 cardiac, neuro, cava and random genes", if "GC" in header, then "IGVF general controls", else leave NA
pre_metadata_df[col_ref] = 'NA' # will be added later if "cardiac_neuro_cava_random" then "hg38", if "SLEA" in header, then "hg18", else leave NA
pre_metadata_df[col_chr] = 'NA' # leave NA (TODO: add this from the regions.bed file in the final_design/*/final_design directory)
pre_metadata_df[col_start] = 'NA' # leave NA (TODO: see above)
pre_metadata_df[col_end] = 'NA' # leave NA (TODO: see above)
pre_metadata_df[col_strand] = 'NA' # leave NA (TODO: see above)
pre_metadata_df[col_variant_class] = 'NA' # if "cardiac_neuro_cava_random" and category is "variant" then "SNV", else leave NA
pre_metadata_df[col_variant_pos] = 'NA' # leave NA (TODO: see above + compute from reference position (pos of variant - 1) - pos of reference = variant_pos (0-based))
pre_metadata_df[col_SPDI] = 'NA' # leave NA
pre_metadata_df[col_allele] = 'NA' # if cardiac_neuro_cava_random if REF_ in header then "ref", if ALT_ in header then "alt", else leave NA
pre_metadata_df[col_info] = 'NA' # leave NA

# name:
# pre_metadata_df[col_name] = pre_metadata_df[col_sequence].apply(lambda x: 'oligo_' + hashlib.md5(x.encode()).hexdigest())
pre_metadata_df[col_name] = pre_metadata_df['header']
# check for duplicates
pre_metadata_df[col_name].duplicated().sum() # 0


# class:
pre_metadata_df[col_class] = pre_metadata_df['header'].apply(get_class)

# category:
pre_metadata_df[col_category] = pre_metadata_df['header'].apply(get_category)


# source:
pre_metadata_df[col_source] = pre_metadata_df['header'].apply(get_source)

# ref_sequence:
pre_metadata_df[col_ref] = pre_metadata_df['header'].apply(lambda x: 'GRCh38' if 'cardiac_neuro_cava_random' in x else ('hg18' if 'SLEA' in x else 'GRCh38'))

# leave NA for seq_chr, seq_start, seq_end (add later by loading bed class-wise)


# variant_class:
pre_metadata_df[col_variant_class] = pre_metadata_df.apply(lambda x: 'SNP' if x[col_category] == 'variant' else 'NA', axis=1)

# leave NA for variant_pos, SPDI, 

# allele:
pre_metadata_df[col_allele] = pre_metadata_df['header'].apply(ref_alt_decision)

# info
pre_metadata_df[col_info] = pre_metadata_df['header'].apply(get_info)
# pre_metadata_df


#### Get variant positions of cardiac_neuro_cava_random with vcf file
- /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/variants.vcf.gz

In [20]:
bed_df_5k_path = config['files']['final_design']['region_bed_5k']
bed_df_5k = pd.read_csv(bed_df_5k_path, sep="\t", header=None)
bed_df_5k.columns = ['chrom', 'start', 'end', 'ID', 'score', 'strand']
bed_df_5k['sequence_length'] = bed_df_5k['end'] - bed_df_5k['start']
bed_df_5k

,chrom,start,end,ID,score,strand,sequence_length
0,chr1,2177692,2178006,SKI|ENSG00000157933.11|EH38E2778468,.,+,314
1,chr1,2179468,2179817,SKI|ENSG00000157933.11|EH38E2778471,.,+,349
2,chr1,2180360,2180705,SKI|ENSG00000157933.11|EH38E2778473,.,+,345
3,chr1,2181818,2182138,SKI|ENSG00000157933.11|EH38E2778476,.,+,320
4,chr1,2182410,2182738,SKI|ENSG00000157933.11|EH38E2778477,.,+,328
...,...,...,...,...,...,...,...
31087,chrX,154544972,154545298,G6PD|ENSG00000160211.20|EH38E3949733,.,-,326
31088,chrX,154545819,154546165,G6PD|ENSG00000160211.20|EH38E3949734,.,-,346
31089,chrX,154548990,154549196,G6PD|ENSG00000160211.20|EH38E3949743,.,-,206
31090,chrX,154549862,154550013,G6PD|ENSG00000160211.20|EH38E2774396,.,-,151


In [24]:
# pre_metadata_df.head()
pre_metadata_df['header'].to_list()

['cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778477_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778478_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778480_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778484_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778493_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778511_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778514_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778516_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778523_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778531_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778533_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778545_

In [74]:
# read vcf file and merge info with region file
import vcfpy
vcf_path = config['files']['final_design']['vcf_file_tested']
vcf_path = config['files']['final_design']['vcf_file']

# Open file, this will read in the header
reader = vcfpy.Reader.from_path(vcf_path)


/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr1>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr10>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr11>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr12>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not found in header line contig=<ID=chr13>
  warnings.warn(
/home/kisa/miniforge3/envs/mobil/lib/python3.9/site-packages/vcfpy/header.py:583: FieldInfoNotFound: Field "length" not f

In [8]:
def region_id_to_variant_region(id):
    """
    variant region file (variant.vcf) has no group information in the region ids
    => remove the group info from the idea for matching
    """
    if ":" in id:
        return id.split(":")[1]
    else: 
        raise ValueError('No group information in id')
    

def get_region(id):
    """
    return chrom start end of region with this id
    """
    empty = False
    row = bed_df.loc[bed_df['variant_region'] == id]
    if row.shape[0] < 1:
        empty = True
        return "chrom", "start", "end", empty
    chrom = row['chrom'].to_list()[0]
    start = row['start'].to_list()[0]
    end = row['end'].to_list()[0]
    return chrom, start, end, empty

In [9]:
# vcf_file_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/variants.vcf.gz'
# read vcf file with commment #
bed_path = config['files']['final_design']['region_bed']
# get bed in dataframe
bed_df = pd.read_csv(bed_path, sep="\t", header=None)
bed_df.columns = ['chrom', 'start', 'end', 'ID', 'score', 'strand']
# bed_df['sequence_length'] = bed_df['end'] - bed_df['start']
# bed_df['variant_region'] = bed_df['ID'].apply(region_id_to_variant_region)
bed_df

,chrom,start,end,ID,score,strand
0,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
1,chr1,2181843,2182113,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
2,chr1,2182439,2182709,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
3,chr1,2182830,2183100,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
4,chr1,2185027,2185297,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
...,...,...,...,...,...,...
28385,chrX,154531979,154532249,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28386,chrX,154539055,154539325,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28387,chrX,154545000,154545270,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28388,chrX,154549802,154550072,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-


##### Find example where multiple regions and we cannot find the region in the second bed file

In [106]:
# create tsv from vcf file: with chrom pos ref alt id qual ... + header
counter = 0
for record in reader:
    # get region
    # print(record)
    region_list = record.INFO['Region']
    ref_list = record.INFO['REF_ID']
    # region is list object: more than one region per variant?
    # region has no group information => change bed file header
    if len(region_list) > 1:
        print(record)
        counter += 1
    if len(region_list) > 1:
        # hopefully first region is there
        chrom, start, end, empty = get_region(region_list[0])
        print(chrom)
        print(start)
        print(end)
        for region in region_list[1:]:
            # find region in bed file
            # second region might not be there
            cur_chrom, cur_start, cur_end, empty = get_region(region)
            if not ((chrom == cur_chrom) and (start == cur_start) and (end == cur_end)):
                print('regions are not equal')
        
    # check if region 
    
    # # design header style 
    # region = region.replace('>','*')
    # region = region.replace(',', '~')
    # print(region)
    # print(ref)
    # # line = [record.CHROM, record.POS, record.ID, record.REF, record.ALT, record.QUAL, record.FILTER, record.INFO]
    # if counter > 4:
    #     break
    # counter += 1
    
print('Number of variants with more than one region: ', counter)

Record('chr1', 231021494, ['cardiac_neuro_cava_random:ARV1|ENSG00000173409.14|EH38E2873164|1-231021494-T-C'], 'T', [Substitution(type_='SNV', value='C')], None, ['PASS'], {'Region': ['ARV1|ENSG00000173409.14|EH38E2873164_fwd_tile1-1', 'ARV1|ENSG00000173409.14|EH38E2873165_fwd_tile1-1'], 'REF_ID': ['REF_ARV1|ENSG00000173409.14|EH38E2873164_fwd_tile1-1', 'REF_ARV1|ENSG00000173409.14|EH38E2873165_fwd_tile1-1'], 'ALT_ID': ['ALT_ARV1|ENSG00000173409.14|EH38E2873164_fwd_tile1-1_ARV1|ENSG00000173409.14|EH38E2873164|1-231021494-T-C', 'ALT_ARV1|ENSG00000173409.14|EH38E2873165_fwd_tile1-1_ARV1|ENSG00000173409.14|EH38E2873164|1-231021494-T-C'], 'AF': [6.56849e-06]}, [], [])
chr1
231021287
231021557


IndexError: list index out of range

#### Check how many cases do not have a region id we can find in the region bed
- 136 for all sequences 
- 2 for tested sequences

In [116]:
empty_counter, counter = 0, 0
for record in reader:
    # get region
    # print(record)
    region_list = record.INFO['Region']
    ref_list = record.INFO['REF_ID']
    # region is list object: more than one region per variant?
    # region has no group information => change bed file header
    if len(region_list) > 1:
        # print(record)
        counter += 1
    if len(region_list) > 1:
        # hopefully first region is there
        chrom, start, end, empty = get_region(region_list[0])
        # print(chrom)
        # print(start)
        # print(end)
        for region in region_list[1:]:
            # find region in bed file
            # second region might not be there
            cur_chrom, cur_start, cur_end, empty = get_region(region)
            if empty:
                print(record)
                empty_counter += 1

print('Not finable regions: ', empty_counter)

Record('chr1', 231021494, ['cardiac_neuro_cava_random:ARV1|ENSG00000173409.14|EH38E2873164|1-231021494-T-C'], 'T', [Substitution(type_='SNV', value='C')], None, ['PASS'], {'Region': ['ARV1|ENSG00000173409.14|EH38E2873164_fwd_tile1-1', 'ARV1|ENSG00000173409.14|EH38E2873165_fwd_tile1-1'], 'REF_ID': ['REF_ARV1|ENSG00000173409.14|EH38E2873164_fwd_tile1-1', 'REF_ARV1|ENSG00000173409.14|EH38E2873165_fwd_tile1-1'], 'ALT_ID': ['ALT_ARV1|ENSG00000173409.14|EH38E2873164_fwd_tile1-1_ARV1|ENSG00000173409.14|EH38E2873164|1-231021494-T-C', 'ALT_ARV1|ENSG00000173409.14|EH38E2873165_fwd_tile1-1_ARV1|ENSG00000173409.14|EH38E2873164|1-231021494-T-C'], 'AF': [6.56849e-06]}, [], [])
Record('chr18', 55421931, ['cardiac_neuro_cava_random:TCF4|ENSG00000196628.20|EH38E3270136|18-55421931-C-T'], 'C', [Substitution(type_='SNV', value='T')], None, ['PASS'], {'Region': ['TCF4|ENSG00000196628.20|EH38E3270136_rev_tile1-1', 'TCF4|ENSG00000196628.20|EH38E1918394_rev_tile1-1'], 'REF_ID': ['REF_TCF4|ENSG00000196628.20|

#### Only consider one region as truth and get this region
- does it work? yes but what is with the second region if here is one?

In [129]:
counter = 0
empty_region_list = 0
count_no_region_found = 0
count_skipped_variant = 0
for record in reader:
    # get region
    # print(record)
    variant_position = record.POS
    region_list = record.INFO['Region']
    ref_list = record.INFO['REF_ID']
    # region is list object: more than one region per variant?
    # region has no group information => change bed file header
    
    # skip cases where multiple regions are assigned
    if len(region_list) > 1:
        count_skipped_variant += 1
        continue
    elif len(region_list) == 1:
        chrom, start, end, empty = get_region(region_list[0])
        if empty: # not possible
            print(record)
            count_no_region_found += 1
            # print('could not be found')
    else:
        empty_region_list += 1
    
    # # compute variant position
    # relative_variant_position = variant_position - start
    # # if counter > 4:
    # #     break
    # # counter += 1
    # print(record)
    # print(relative_variant_position)
print('Empty region_list: ', empty_region_list)
print('Region not findable: ', count_no_region_found)
print('Skipped, because multiple regions', count_skipped_variant)

Record('chr1', 21564170, ['GC_Mendelian_variants:chr1:21564170G>A|ALPL'], 'G', [Substitution(type_='SNV', value='A')], None, ['PASS'], {'gene': 'ALPL', 'PMID': 10679946, 'Enhancer': True, 'Region': ['chr1:21564170G>A|ALPL'], 'REF_ID': ['REF_chr1:21564170G>A|ALPL'], 'ALT_ID': ['ALT_chr1:21564170G>A|ALPL_chr1:21564170G>A|ALPL']}, [], [])
Record('chr1', 56497149, ['C_positive_heart_CAD:rs17114036'], 'A', [Substitution(type_='SNV', value='G')], None, ['PASS'], {'Region': ['rs17114036'], 'REF_ID': ['REF_rs17114036'], 'ALT_ID': ['ALT_rs17114036_rs17114036']}, [], [])
Record('chr1', 156478417, ['C_positive_heart_CAD:rs4450010'], 'T', [Substitution(type_='SNV', value='G')], None, ['PASS'], {'Region': ['rs4450010'], 'REF_ID': ['REF_rs4450010'], 'ALT_ID': ['ALT_rs4450010_rs4450010']}, [], [])
Record('chr1', 201917641, ['C_positive_heart_CAD:rs34091558'], 'T', [Substitution(type_='INS', value='TA')], None, ['PASS'], {'Region': ['rs34091558'], 'REF_ID': ['REF_rs34091558'], 'ALT_ID': ['ALT_rs340915

##### Duplicated design: Investigate if all variants from cardiac_neuro_cava_random can be matched with regions
- Rows in files:
    - variant region map (only tested): 46374 (all: 47044)
    - bed file (only tested): 27556 (all: 28390)
    - design file (only tested): 73940 (all: 80806 (deduplicated: 80215))
    - variant vcf (only tested): 45737 (all: 46143 (might include duplicates))
    - higher number of variant region map then variants in vcf because?
      - 46374 unique alternative sequences but only 45737 unique variant ids
      - how is it possible that two alt are connected? (Only possible with tiling which was not done according to Mohan) 
- for all sequences: 
    - matched variant region:  46821
- only for tested: 
    matched variant region:  46374 (means all from)
- only tested matched variant rows:  73940
- ref shape: 18331 (all of them are matched with pre_metadata)
- alt shape: 45000 (all of them are matched with pre_metadata)
- region shape: 10609 (only 8900 are matched with pre_metadata with matching header)
- all ref and alt can be found but not all regions


- variant region map: 47044 (only tested: 46374)
- headers are different between deduplicated and duplicated (1717 are different)
- regionen: easy left join
- var und ref: 
    - variant region map (add design_style header)

- First merge variant region map with regions
- Secondly check how many of our sequences pre_metada_df (with deduplicated design) header we can match
- if I match with the matching headers: can I match anything?

# todo: put variant position into meta data


In [43]:
# add matching header to pre_metadata_df
pre_metadata_df['tmp_matching_header'] = pre_metadata_df['header'].apply(add_matching_header) 
pre_metadata_df

# load variant region map
variant_region_map_path = config['files']['final_design']['final_design_variant_region_map']
variant_region_map = pd.read_csv(variant_region_map_path, sep="\t")
variant_region_map

bed_path = config['files']['final_design']['region_bed']
# get bed in dataframe
bed_df = pd.read_csv(bed_path, sep="\t", header=None)
bed_df.columns = ['chrom', 'start', 'end', 'ID', 'score', 'strand']
# bed_df['sequence_length'] = bed_df['end'] - bed_df['start']
# bed_df['variant_region'] = bed_df['ID'].apply(region_id_to_variant_region)
bed_df


,chrom,start,end,ID,score,strand
0,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
1,chr1,2181843,2182113,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
2,chr1,2182439,2182709,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
3,chr1,2182830,2183100,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
4,chr1,2185027,2185297,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
...,...,...,...,...,...,...
28385,chrX,154531979,154532249,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28386,chrX,154539055,154539325,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28387,chrX,154545000,154545270,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28388,chrX,154549802,154550072,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-


In [44]:
# consider only tested sequences
tested_variant_region_map = variant_region_map[variant_region_map['Variant'].str.startswith('cardiac_neuro_cava_random')]
print('only tested variants: ', tested_variant_region_map.shape[0])
tested_bed_df = bed_df[bed_df['ID'].str.startswith('cardiac_neuro_cava_random')]
print('only tested region rows: ', tested_bed_df.shape[0])
tested_pre_metadata_df = pre_metadata_df[pre_metadata_df['tmp_label'] == 'cardiac_neuro_cava_random']
print('only tested metadata rows: ', tested_pre_metadata_df.shape[0])
# variant_region_map_file = tested_variant_region_map.merge(bed_df, left_on="Region", right_on="ID", how="left")

# only tested variants:  46374
# only tested region rows:  27556
# only tested metadata rows:  73940

only tested variants:  46374
only tested region rows:  27556
only tested metadata rows:  73940


In [47]:
tested_variant_region_map_file = tested_variant_region_map.merge(tested_bed_df, left_on="Region", right_on="ID", how="left")
tested_matched_variant_region = tested_variant_region_map_file.loc[~tested_variant_region_map_file['chrom'].isna()]
tested_matched_variant_region.shape[0] # 46374 found all regions

46374

In [ ]:
# check if the ref and alt id are matchable between the tested_pre_metadata_df
# with grep unique alt in duplicated and deduplicated 47834
# unique only in deduplicated: 46458 => 1374 different
# with grep unique ref in duplicated and deduplicated 18824
# combined 72223 unique headers without "~" or "," which are also unique in only deduplicated => 1.717 are different

In [53]:
# ref
ref_tested_pre_metadata = tested_pre_metadata_df.loc[tested_pre_metadata_df['allele'] == 'ref']
print('ref shape: ', ref_tested_pre_metadata.shape[0])
# alt
alt_tested_pre_metadata = tested_pre_metadata_df.loc[tested_pre_metadata_df['allele'] == 'alt']
print('alt shape: ', alt_tested_pre_metadata.shape[0])

ref shape:  18572
alt shape:  46374


In [68]:
ref_tested_metadata_region = ref_tested_pre_metadata.merge(tested_matched_variant_region[['Region', 'REF_ID', 'chrom', 'start', 'end', 'score', 'strand']].drop_duplicates(), left_on="tmp_matching_header", right_on="REF_ID", how="left")
merged_ref_tested_metadata_region = ref_tested_metadata_region[~ref_tested_metadata_region['Region'].isna()]
print("matched ref: ", merged_ref_tested_metadata_region.shape[0]) # 18572 all could be found

alt_tested_pre_metadata_variant = alt_tested_pre_metadata.merge(tested_matched_variant_region[['Region', 'ALT_ID', 'chrom', 'start', 'end', 'score', 'strand']].drop_duplicates(), left_on="tmp_matching_header", right_on="ALT_ID", how="left")
merged_alt_tested_metadata_region = alt_tested_pre_metadata_variant[~alt_tested_pre_metadata_variant['Region'].isna()]
print("matched alt: ", merged_alt_tested_metadata_region.shape[0]) # 46374 all could be matched


matched ref:  18331
matched alt:  45000


In [48]:
# region
region_tested_pre_metadata = tested_pre_metadata_df.loc[tested_pre_metadata_df['category'] == 'element']
print('region shape: ', region_tested_pre_metadata.shape[0])

ref shape:  18572
alt shape:  46374
region shape:  8994


In [63]:
# match region_tested_pre_metadata directly with region id
tested_element_region = region_tested_pre_metadata.merge(tested_bed_df, left_on="tmp_matching_header", right_on="ID", how="left")
tested_matched_element_region = tested_element_region.loc[~tested_element_region['chrom'].isna()]
tested_matched_element_region.shape[0] # 8900 (94 are missing)

8900

##### Investigate not matching headers of regions

In [61]:
# seems like all start with ref or alt
def ref_or_alt(header):
    if ':REF_' in header:
        return 'ref'
    elif ':ALT_' in header:
        return 'alt'
    else:
        return "nothing"

In [75]:
not_merged_alt_tested_metadata_region = tested_element_region[tested_element_region['chrom'].isna()]
not_merged_alt_tested_metadata_region['ref_or_alt'] = not_merged_alt_tested_metadata_region['header'].apply(ref_or_alt)
print(not_merged_alt_tested_metadata_region.shape[0])
not_merged_alt_tested_metadata_region['header'].to_list()

/tmp/ipykernel_377269/3049616686.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  not_merged_alt_tested_metadata_region['ref_or_alt'] = not_merged_alt_tested_metadata_region['header'].apply(ref_or_alt)


['cardiac_neuro_cava_random:REF_FBXO28|ENSG00000143756.12|EH38E2868786_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_MYL2|ENSG00000111245.17|EH38E3040936_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_MEIS2|ENSG00000134138.22|EH38E3127800_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_CHD3|ENSG00000170004.19|EH38E3207402~KDM6B|ENSG00000132510.11|EH38E3207402_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_PIGS|ENSG00000087111.22|EH38E3215655_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_CIC|ENSG00000079432.9|EH38E3307613_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_GIGYF2|ENSG00000204120.16|EH38E3407187_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_WDFY3|ENSG00000163625.17|EH38E2309079_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_GATAD1|ENSG00000157259.8|EH38E3785709_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_NLGN3|ENSG00000196338.15|EH38E3937045_fwd_tile1-1',
 'cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2779927_fwd_tile1-1_PRDM16|ENSG00000142611.17|EH38E2779927|1-33

In [65]:
not_merged_alt_tested_metadata_region['ref_or_alt'].value_counts()

ref_or_alt
alt    84
ref    10
Name: count, dtype: int64

In [72]:
# split in ref and alt 
alt_not_merged_alt_tested_metadata_region = not_merged_alt_tested_metadata_region.loc[not_merged_alt_tested_metadata_region['ref_or_alt'] == 'alt']
print(alt_not_merged_alt_tested_metadata_region.shape[0])
ref_not_merged_alt_tested_metadata_region = not_merged_alt_tested_metadata_region.loc[not_merged_alt_tested_metadata_region['ref_or_alt'] == 'ref']
print(ref_not_merged_alt_tested_metadata_region.shape[0])

84
10


In [73]:
ref_element_tested_metadata_region = ref_not_merged_alt_tested_metadata_region.merge(tested_matched_variant_region[['Region', 'REF_ID', 'chrom', 'start', 'end', 'score', 'strand']].drop_duplicates(), left_on="header", right_on="REF_ID", how="left")
merged_ref_element_tested = ref_element_tested_metadata_region[~ref_element_tested_metadata_region['Region'].isna()]
print("matched ref: ", merged_ref_element_tested.shape[0])

alt_element_tested_pre_metadata_variant = alt_not_merged_alt_tested_metadata_region.merge(tested_matched_variant_region[['Region', 'ALT_ID', 'chrom', 'start', 'end', 'score', 'strand']].drop_duplicates(), left_on="header", right_on="ALT_ID", how="left")
merged_alt_element_tested = alt_element_tested_pre_metadata_variant[~alt_element_tested_pre_metadata_variant['Region'].isna()]
print("matched alt: ", merged_alt_element_tested.shape[0])

matched ref:  0
matched alt:  0


##### Ultimative check: check in vcf file by hand the 94 headers

##### Compare final design (duplicated) and deduplicated
- Difference of 591 sequences because of duplicates in the control sequences => I can use the tested sequences and their regions
- duplicated (80806):
  - sequences: 
    - 73940 tested (unique)
    - 6866 controls
      - 6275 controls (unique)
- deduplicated (80215):
  - 73940 tested (unique)
  - 6275 conrols (unique)


In [ ]:
# Deduplicated: 
# -----------Summary statics of the current version of metadata file: -------------
# The number of unique rows in the table is: 80215
# Number of tested (73940) and control sequences (6275)
# Number of class distribution:
#                       class  count
# 0                      test  73940
# 1  element inactive control   5986
# 2    element active control    289
# Number distribution of category from tested rows:
#   category  count
# 0  variant  64946
# 1  element   8994
# Number of controls variants and elements:
#     category  count
# 0    element   5408
# 1  scrambled    667
# 2  synthetic    200
# Number distribution of controls between class:
# class
# element inactive control    5986
# element active control       289
# Name: count, dtype: int64
# Number of ref and alt for tested:
# allele
# alt    46374
# ref    18572
# NA      8994
# Name: count, dtype: int64
# Number of ref and alt for controls:
# allele
# NA    6275
# Name: count, dtype: int64

In [ ]:
# Summary of the duplicated design (duplication not in the tested sequences)
# -----------Summary statics of the current version of metadata file: -------------
# The number of unique rows in the table is: 80084
# Number of tested (73940) and control sequences (6866)
# Number of class distribution:
#                       class  count
# 0                      test  73940
# 1  element inactive control   5587
# 2  variant negative control    984
# 3    element active control    295
# Number distribution of category from tested rows:
#   category  count
# 0  variant  64946
# 1  element   8994
# Number of controls variants and elements:
#     category  count
# 0    element   5015
# 1    variant    984
# 2  scrambled    667
# 3  synthetic    200
# Number distribution of controls between class:
# class
# element inactive control    5587
# variant negative control     984
# element active control       295
# Name: count, dtype: int64
# Number of ref and alt for tested:
# allele
# alt    46374
# ref    18572
# NA      8994
# Name: count, dtype: int64
# Number of ref and alt for controls:
# allele
# NA     5882
# alt     670
# ref     314
# Name: count, dtype: int64

In [ ]:
# only tested
# -----------Summary statics of the current version of metadata file: -------------
# The number of unique rows in the table is: 73940
# Number of tested (73940) and control sequences (0)
# Number of class distribution:
#   class  count
# 0  test  73940
# Number distribution of category from tested rows:
#   category  count
# 0  variant  64946
# 1  element   8994
# Number of controls variants and elements:
# Empty DataFrame
# Columns: [category, count]
# Index: []
# Number distribution of controls between class:
# Series([], Name: count, dtype: int64)
# Number of ref and alt for tested:
# allele
# alt    46374
# ref    18572
# NA      8994
# Name: count, dtype: int64
# Number of ref and alt for controls:
# Series([], Name: count, dtype: int64)

In [19]:
# tested_variant_region_map
tested_bed_df = bed_df[bed_df['ID'].str.startswith('cardiac_neuro_cava_random')]
print('Region shape: ', tested_bed_df.shape[0])
print('all regions: ', bed_df.shape[0])

Region shape:  27556
all regions:  28390


In [12]:
variant_region_map_file = tested_variant_region_map.merge(bed_df, left_on="Region", right_on="ID", how="left")
matched_variant_region = variant_region_map_file.loc[~variant_region_map_file['chrom'].isna()]
matched_variant_region # 46821 
print('matched variant region: ', matched_variant_region.shape[0])
tested_matched_variant_region = matched_variant_region.loc[matched_variant_region['Variant'].str.startswith('cardiac_neuro_cava_random')]
tested_matched_variant_region # 46374 all regions in variant map could be matched
tested_pre_metadata_df = pre_metadata_df[pre_metadata_df['tmp_label'] == 'cardiac_neuro_cava_random']
print('only tested matched variant rows: ', tested_pre_metadata_df.shape[0])
# devide the metadata_df 

# ref
ref_tested_pre_metadata = tested_pre_metadata_df.loc[tested_pre_metadata_df['allele'] == 'ref']
print('ref shape: ', ref_tested_pre_metadata.shape[0])
# alt
alt_tested_pre_metadata = tested_pre_metadata_df.loc[tested_pre_metadata_df['allele'] == 'alt']
print('alt shape: ', alt_tested_pre_metadata.shape[0])

# region
region_tested_pre_metadata = tested_pre_metadata_df.loc[tested_pre_metadata_df['category'] == 'element']
print('region shape: ', region_tested_pre_metadata.shape[0])


matched variant region:  46374
only tested matched variant rows:  73940
ref shape:  18331
alt shape:  45000
region shape:  10609


In [179]:
region_tested_pre_metadata['header'].to_list()

['cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778477_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778478_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778480_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778484_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778493_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778511_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778514_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778516_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778523_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778531_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778533_fwd_tile1-1',
 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778545_

In [41]:
# bed_df[~bed_df['ID'].str.startswith('cardiac_neuro_cava_random')]
# check label 
bed_df['label'] = bed_df['ID'].apply(hf.get_label)
bed_df['label'].value_counts()

label
cardiac_neuro_cava_random    27556
GC_Vista                       256
GC_Cort_Chengyu                185
GC_Selvarajan                  166
GC_GABA_Chengyu                 85
GC_Glut_Chengyu                 83
GC_Atrial_fib                   22
GC_Mohlke                       18
GC_Liang                         8
GC_Hon                           6
GC_Kircher                       5
Name: count, dtype: int64

In [31]:
# TODO: investigate reason for not matched
# first combine bed and vcf information
test_position_not_matched

,header,sequence,tmp_label,name,category,class,source,ref,chr,start_x,...,SPDI,allele,info,chrom,start_y,end_y,ID,score,strand_y,sequence_length
708,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTAAGGAAGGGAGGGAGGGAGGGAGCGATCCCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
709,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTGGACCACCTCCACCAACTGTCAGCTCACATC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
710,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACTGTCCTTCCTGAGGCCTCCAGCATTATTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
711,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTTTACTTATTTATTTATTTTTTGAGACAGGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
712,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACCCCATAGTATGCCTCCCTCCCTCTCTCCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,alt,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,alt,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,alt,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,alt,cardiac_neuro_cava_random,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# put regions into meta data
tested_sequences = pre_metadata_df[pre_metadata_df['tmp_label'] == 'cardiac_neuro_cava_random']
print(tested_sequences.shape[0])
tested_position = tested_sequences.merge(bed_df, left_on=col_name, right_on='ID', how="left")
print(tested_position.shape[0])
test_position_merged = tested_position[~tested_position['chrom'].isna()]
test_position_not_matched = tested_position[tested_position['chrom'].isna()]

print(test_position_merged.shape[0]) # 8801


73940
73940
8801


,header,sequence,tmp_label,name,category,class,source,ref,chr,start_x,...,SPDI,allele,info,chrom,start_y,end_y,ID,score,strand_y,sequence_length
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chr1,2181843.0,2182113.0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270.0
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chr1,2182439.0,2182709.0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270.0
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chr1,2182830.0,2183100.0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270.0
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chr1,2185027.0,2185297.0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270.0
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chr1,2188389.0,2188659.0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8895,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGACTCTGGGCTGCTCAGAGGCTGCCTTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chrX,154376832.0,154377102.0,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,.,-,270.0
8896,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGAGCCCTGGGGAACGCCATGAGCCCTCAGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chrX,154383755.0,154384025.0,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,.,-,270.0
8897,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGGACTTAAACCCCAGCCTCCCCCGTCCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chrX,154402403.0,154402673.0,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,.,-,270.0
8898,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGCCCATAATTTATTGATTTTTTAAAATTTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,...,NA,NA,cardiac_neuro_cava_random,chrX,154409693.0,154409963.0,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,.,-,270.0


,chrom,start,end,ID,score,strand,sequence_length
0,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270
1,chr1,2181843,2182113,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270
2,chr1,2182439,2182709,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270
3,chr1,2182830,2183100,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270
4,chr1,2185027,2185297,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+,270
...,...,...,...,...,...,...,...
28385,chrX,154531979,154532249,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-,270
28386,chrX,154539055,154539325,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-,270
28387,chrX,154545000,154545270,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-,270
28388,chrX,154549802,154550072,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-,270


In [14]:
bed_df

,chrom,start,end,ID,score,strand
0,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
1,chr1,2181843,2182113,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
2,chr1,2182439,2182709,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
3,chr1,2182830,2183100,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
4,chr1,2185027,2185297,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
...,...,...,...,...,...,...
28385,chrX,154531979,154532249,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28386,chrX,154539055,154539325,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28387,chrX,154545000,154545270,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28388,chrX,154549802,154550072,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-


#### Parsing from blat results for cardiac_neuro_cava_random
- ref_interesting_region_control_df.head()

#### Start parsing regions from header
- for intersting neuro without chengyu

In [21]:
pre_metadata_df.apply(parse_regions_from_header, axis=1)

C_negative_neuron_MK:tile_14444_chr15_67066278_67066547_reference__1.1385203581298
C_negative_neuron_MK:tile_9599_chr12_65742272_65742541_reference__1.12348297546129
C_negative_neuron_MK:tile_9256_chr12_26114182_26114451_reference__1.06883601881989
C_negative_neuron_MK:tile_44145_chr8_80578146_80578415_reference__1.04321190480237
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_G_C_19__0.955831785228529
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_A_T_2__0.94196907866527
C_negative_neuron_MK:tile_33520_chr5_77645003_77645272_reference__0.91419962800252
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_reference__0.877705254146922
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_A_G_23__0.876434621875319
C_negative_neuron_MK:rdhs_667570_chr8_22044451_22044720_reference__0.869708675963775
C_negative_neuron_MK:tile_34755_chr5_139668319_139668588_reference__0.816007668785099
C_negative_neuron_MK:tile_1997_chr1_88462325_88462594_C_G_161__0.742187250156855
C_negative_n

,SPDI,allele,category,chr,class,end,header,info,name,ref,sequence,source,start,strand,tmp_alt_base,tmp_label,tmp_ref_base,variant_class,variant_pos
0,NA,NA,element,NA,test,NA,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,GRCh38,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NaN,cardiac_neuro_cava_random,NaN,NA,NA
1,NA,NA,element,NA,test,NA,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,GRCh38,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NaN,cardiac_neuro_cava_random,NaN,NA,NA
2,NA,NA,element,NA,test,NA,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,GRCh38,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NaN,cardiac_neuro_cava_random,NaN,NA,NA
3,NA,NA,element,NA,test,NA,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,GRCh38,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NaN,cardiac_neuro_cava_random,NaN,NA,NA
4,NA,NA,element,NA,test,NA,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,GRCh38,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NaN,cardiac_neuro_cava_random,NaN,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,NA,NA,scrambled,NA,element inactive control,NA,MK:tile_2240|chr1-116244322+116244591|scramble...,MK,MK:tile_2240|chr1-116244322+116244591|scramble...,GRCh38,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,NA,NA,NA,NaN,MK,NaN,NA,NA
80211,NA,NA,scrambled,NA,element inactive control,NA,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,MK,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,GRCh38,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,NA,NA,NA,NaN,MK,NaN,NA,NA
80212,NA,NA,scrambled,NA,element inactive control,NA,MK:tile_18415|chr17-71181691+71181960|scramble...,MK,MK:tile_18415|chr17-71181691+71181960|scramble...,GRCh38,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,NA,NA,NA,NaN,MK,NaN,NA,NA
80213,NA,NA,scrambled,NA,element inactive control,NA,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,MK:tile_14356|chr15-67031618+67031887|scramble...,GRCh38,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,NA,NA,NA,NaN,MK,NaN,NA,NA


In [20]:
# add regions with parser to 
interesting_neuro = ['C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK']
print(pre_metadata_df[pre_metadata_df['tmp_label'].isin(interesting_neuro)].shape[0])
interesting_neuro_pre_metadata_df = pre_metadata_df[pre_metadata_df['tmp_label'].isin(interesting_neuro)].apply(parse_regions_from_header, axis=1)

interesting_neuro_pre_metadata_df # 634 

# 

634
C_negative_neuron_MK:tile_14444_chr15_67066278_67066547_reference__1.1385203581298
C_negative_neuron_MK:tile_9599_chr12_65742272_65742541_reference__1.12348297546129
C_negative_neuron_MK:tile_9256_chr12_26114182_26114451_reference__1.06883601881989
C_negative_neuron_MK:tile_44145_chr8_80578146_80578415_reference__1.04321190480237
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_G_C_19__0.955831785228529
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_A_T_2__0.94196907866527
C_negative_neuron_MK:tile_33520_chr5_77645003_77645272_reference__0.91419962800252
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_reference__0.877705254146922
C_negative_neuron_MK:tile_1999_chr1_88462556_88462825_A_G_23__0.876434621875319
C_negative_neuron_MK:rdhs_667570_chr8_22044451_22044720_reference__0.869708675963775
C_negative_neuron_MK:tile_34755_chr5_139668319_139668588_reference__0.816007668785099
C_negative_neuron_MK:tile_1997_chr1_88462325_88462594_C_G_161__0.742187250156855
C_negati

,SPDI,allele,category,chr,class,end,header,info,name,ref,sequence,source,start,strand,tmp_alt_base,tmp_label,tmp_ref_base,variant_class,variant_pos
75794,NA,NA,element,chr15,element inactive control,67066547,C_negative_neuron_MK:tile_14444_chr15_67066278...,C_negative_neuron_MK,C_negative_neuron_MK:tile_14444_chr15_67066278...,GRCh38,AGGACCGGATCAACTCTCAAGAAGACGGGGCAGCTGGACGAGCTGG...,NA,67066277,NA,NaN,C_negative_neuron_MK,NaN,NA,NA
75795,NA,NA,element,chr12,element inactive control,65742541,C_negative_neuron_MK:tile_9599_chr12_65742272_...,C_negative_neuron_MK,C_negative_neuron_MK:tile_9599_chr12_65742272_...,GRCh38,AGGACCGGATCAACTAGCCCGCCCGCGCCGCACGGTGGCCCCTTGC...,NA,65742271,NA,NaN,C_negative_neuron_MK,NaN,NA,NA
75796,NA,NA,element,chr12,element inactive control,26114451,C_negative_neuron_MK:tile_9256_chr12_26114182_...,C_negative_neuron_MK,C_negative_neuron_MK:tile_9256_chr12_26114182_...,GRCh38,AGGACCGGATCAACTGGGTGTCGACCTTCGCGTAACCCGCGCCCAG...,NA,26114181,NA,NaN,C_negative_neuron_MK,NaN,NA,NA
75797,NA,NA,element,chr8,element inactive control,80578415,C_negative_neuron_MK:tile_44145_chr8_80578146_...,C_negative_neuron_MK,C_negative_neuron_MK:tile_44145_chr8_80578146_...,GRCh38,AGGACCGGATCAACTTGCCGCTTCCGAGGGCTCCTCTCGCCAAGGG...,NA,80578145,NA,NaN,C_negative_neuron_MK,NaN,NA,NA
75798,NA,NA,element,chr1,element inactive control,88462825,C_negative_neuron_MK:tile_1999_chr1_88462556_8...,C_negative_neuron_MK,C_negative_neuron_MK:tile_1999_chr1_88462556_8...,GRCh38,AGGACCGGATCAACTCATCAAGCCGCGCTCTCTCATTAATCTGCTC...,NA,88462555,NA,NaN,C_negative_neuron_MK,NaN,NA,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76614,NA,NA,element,chr12,element active control,121536370,C_positive_neuron_NP:GW18_PFC_ABC_chr12_121536...,C_positive_neuron_NP,C_positive_neuron_NP:GW18_PFC_ABC_chr12_121536...,GRCh38,AGGACCGGATCAACTGCAGCGCGCCGGCACGAGTGGAGATAATGCG...,NA,121536100,NA,NaN,C_positive_neuron_NP,NaN,NA,NA
76615,NA,NA,element,chr4,element active control,85029288,C_positive_neuron_NP:NGN2_iPSC_ABC_chr4_850290...,C_positive_neuron_NP,C_positive_neuron_NP:NGN2_iPSC_ABC_chr4_850290...,GRCh38,AGGACCGGATCAACTTAACAGATGAAGCCTCAATTAGGTAGCTAGC...,NA,85029018,NA,NaN,C_positive_neuron_NP,NaN,NA,NA
76616,NA,NA,element,chr4,element active control,782020,C_positive_neuron_NP:GW18_PFC_ABC_NGN2_iPSC_AB...,C_positive_neuron_NP,C_positive_neuron_NP:GW18_PFC_ABC_NGN2_iPSC_AB...,GRCh38,AGGACCGGATCAACTCCGAGACCCGCACTTAGTTACCTGAGAAGGC...,NA,781750,NA,NaN,C_positive_neuron_NP,NaN,NA,NA
76617,NA,NA,element,chr1,element active control,110538548,C_positive_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,C_positive_neuron_NP,C_positive_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,GRCh38,AGGACCGGATCAACTCAGCTCCAGCGGCATGAAATATTGATGCCCT...,NA,110538278,NA,NaN,C_positive_neuron_NP,NaN,NA,NA


In [6]:
pre_metadata_df


,header,sequence,tmp_label,name,category,class,source,variant_class,variant_pos,SPDI,allele,info,ref_seq
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK,MK:tile_2240|chr1-116244322+116244591|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK,MK:tile_18415|chr17-71181691+71181960|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK,MK:tile_14356|chr15-67031618+67031887|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38


#### Generate input for blat

In [7]:
# tmp: code blocks for getting the regions for all controls with blat (currently focusing on important controls)
# first check if the control regions of important controls are usable (check sequences in ucsc)
interesting_control_list = ['C_negative_neuron_MK', 'C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_CD']

In [8]:
important_control_pre_metadata_df = pre_metadata_df[pre_metadata_df['tmp_label'].isin(interesting_control_list)] # 728   
# important_control_pre_metadata_df['tmp_label'].value_counts()
ref_interesting_region_control_df = important_control_pre_metadata_df[important_control_pre_metadata_df[col_allele] != 'alt'] # 728 (all elements) 
ref_interesting_region_control_df = ref_interesting_region_control_df[ref_interesting_region_control_df[col_category] != 'scrambled']  # 722 5 scrambled
ref_interesting_region_control_df = ref_interesting_region_control_df[ref_interesting_region_control_df[col_category] != 'synthetic'] # 722 no synthetic
# # ref_interesting_region_control_df 
# # # remove adapters
ref_interesting_region_control_df['no_adapter_sequence'] = ref_interesting_region_control_df[col_sequence].str[15:285]
ref_interesting_region_control_df
# # write to fasta
write_fasta(ref_interesting_region_control_df, '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/results/control_sequences/blat_interesting_neuro_controls.fa', [col_name, 'no_adapter_sequence'])
# write_fasta(important_control_pre_metadata_df, '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/results/control_sequences/blat_interesting_neuro_controls.fa', [col_name, col_sequence])

True

In [ ]:
ref_interesting_region_control_df.head()

In [ ]:
# get from pre_metadata_df only controls which do not have 'alt' in allele
control_pre_metadata_df = pre_metadata_df[pre_metadata_df['tmp_label'] != 'cardiac_neuro_cava_random'] # 6275
ref_region_control_df = control_pre_metadata_df[control_pre_metadata_df[col_allele] != 'alt'] #  5622 
ref_region_control_df = ref_region_control_df[ref_region_control_df[col_category] != 'scrambled'] # 4955  
ref_region_control_df = ref_region_control_df[ref_region_control_df[col_category] != 'synthetic'] # 4755 
ref_region_control_df

In [17]:
# remove adapter (first 15 and last 15 characters of sequence)
ref_region_control_df['no_adapter_sequence'] = ref_region_control_df[col_sequence].str[15:285]
 
# write to fasta 
write_fasta(ref_region_control_df, '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/results/control_sequences/blat_controls_no_alt_scrambled_synthetic.fa', [col_name, 'no_adapter_sequence'])

True

#### Investigate control groups and check scrambled

In [23]:
# tmp: investigate number of control groups and sanity check scambled
design_control_df = design_df[design_df['label'] != "cardiac_neuro_cava_random"]
design_control_df['label'].value_counts()
all_control_groups = design_control_df['label'].unique()

# only dnase => all are scrambled
dnase_controls = [i for i in all_control_groups if "dnase" in i.lower()]
dnase_controls

# tmp: add functionality to read variant region map 
current_variant_map_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/variant_map_with_unified_headers_mendelian_focus.tsv'
current_variant_map = pd.read_csv(current_variant_map_path, sep="\t")
current_variant_map['label'] = current_variant_map['Variant'].apply(hf.get_label)
current_variant_map

,Variant,Region,REF_ID,ALT_ID,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
...,...,...,...,...,...
47022,C_positive_heart_CAD:rs7865618,NaN,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618,C_positive_heart_CAD
47023,C_positive_heart_CAD:rs4977757,NaN,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757,C_positive_heart_CAD
47024,C_positive_heart_CAD:rs1537373,NaN,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373,C_positive_heart_CAD
47025,C_positive_heart_CAD:rs10811656,NaN,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656,C_positive_heart_CAD


In [26]:
current_variant_map['label'] = current_variant_map['Variant'].apply(hf.get_label)
current_variant_map

reference_sequences = current_variant_map['REF_ID'].to_list()
alternative_sequences = current_variant_map['ALT_ID'].to_list()

# if sequence header in REF_ID return "REF" if sequence header in ALT_ID return "ALT" else return "region"
# testable example: GC_Atrial_fib: 1 region, 21 REF, 23 ALT => 45 sequences
atrial_fib = design_control_df[design_control_df['label'] == 'GC_Atrial_fib']
atrial_fib[col_category] = atrial_fib['header'].apply(get_category)
atrial_fib[col_allele] = atrial_fib['header'].apply(ref_alt_decision)
atrial_fib[col_class] = atrial_fib['header'].apply(get_class)
# GC_Mendelian_variants
mendelian_variants = design_control_df[design_control_df['label'] == 'GC_Mendelian_variants']
mendelian_variants[col_category] = mendelian_variants['header'].apply(get_category)
mendelian_variants[col_allele] = mendelian_variants['header'].apply(ref_alt_decision)
mendelian_variants[col_class] = mendelian_variants['header'].apply(get_class)

# atrial_fib['header'].to_list() 
# ref not recognized as ref
mendelian_variants

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-58/ipykernel_2083318/4216924870.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  atrial_fib['category'] = atrial_fib['header'].apply(get_category)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-58/ipykernel_2083318/4216924870.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  atrial_fib['allele'] = atrial_fib['header'].apply(ref_alt_decision)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-58/ipykernel_2083318/4216924870.py:12: SettingWithCopyWarning: 


,header,sequence,label,category,allele,class
74602,GC_Mendelian_variants:REF_chr1:21564170G*A|ALPL,AGGACCGGATCAACTCCCCAGGGAAATCTGTGGGCATTGTGACCAC...,GC_Mendelian_variants,variant,ref,variant negative control
74603,GC_Mendelian_variants:REF_chr1:209816133C*CA|IRF6,AGGACCGGATCAACTTTGAGCCCAGGGGCTGAATCTGGAGCTTTGG...,GC_Mendelian_variants,variant,ref,variant negative control
74604,GC_Mendelian_variants:REF_chr10:23219376A*C|PTF1A,AGGACCGGATCAACTATTTGGGTTTCTCCTGTGTTTCAGATACTGA...,GC_Mendelian_variants,variant,ref,variant negative control
74605,GC_Mendelian_variants:REF_chr10:23219434A*G|PTF1A,AGGACCGGATCAACTCACTTAAAGAGTCACTGTTACTTTGAGGTTT...,GC_Mendelian_variants,variant,ref,variant negative control
74606,GC_Mendelian_variants:REF_chr10:23219436A*G|PTF1A,AGGACCGGATCAACTCTTAAAGAGTCACTGTTACTTTGAGGTTTTA...,GC_Mendelian_variants,variant,ref,variant negative control
...,...,...,...,...,...,...
74806,GC_Mendelian_variants:ALT_chr9:101435912C*T|AL...,AGGACCGGATCAACTAGATCTTTGGTAGCACACAATTTTTATAGAC...,GC_Mendelian_variants,variant,alt,variant negative control
74807,GC_Mendelian_variants:ALT_chrX:38352331A*G|OTC...,AGGACCGGATCAACTGTGGAAAGACTGGCAATTAGAGGTAGAAAAG...,GC_Mendelian_variants,variant,alt,variant negative control
74808,GC_Mendelian_variants:ALT_chrX:55028202A*G|ALA...,AGGACCGGATCAACTAGGTAATTATGGCCACTTCACAAGGTAGGTA...,GC_Mendelian_variants,variant,alt,variant negative control
74809,GC_Mendelian_variants:ALT_chrX:55031184G*C|ALA...,AGGACCGGATCAACTGGCCTGGCCCTGCATTGGCCCCAAAGGTATC...,GC_Mendelian_variants,variant,alt,variant negative control


In [28]:
mendelian_variants[col_category].value_counts()
#  category
# variant    207
# element      2
mendelian_variants[col_allele].value_counts()
# allele
# alt    161
# ref     46
# NA       2

allele
alt    161
ref     46
NA       2
Name: count, dtype: int64

In [8]:
dnase_controls

['GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled']

#### Summary about metadata numbers

In [36]:
def metadata_summary_numbers(df):
    """
    Return summary statistics of the current version of the meta data file
    - number of unique names
    - number of synthetic and scrambled rows
    - number distribution of categroy from tested rows (class == "test")
    - number of controls variants and elements (class != "test")
    - number of distribution of controls between class: "variant positive control", "variant negative control", "element active control", "element inactive control"
    - number of ref and alt for tested and control variants
    expected values for the columns:
    df[col_class] = [test, variant positive control, variant negative control, element active control, element inactive control]
    df[col_category] = [variant, element, synthetic, scrambled]
    """
    print("-----------Summary statics of the current version of metadata file: -------------")
    print(f"The number of unique rows in the table is: {df[col_name].nunique()}")
    print(f"Number of tested ({df[df[col_class] == 'test'].shape[0]}) and control sequences ({df[df[col_class] != 'test'].shape[0]})")
    print(f"Number of class distribution:")
    print(pd.DataFrame(df[col_class].value_counts()).reset_index())
    print(f"Number distribution of category from tested rows:")
    print(pd.DataFrame(df[df[col_class] == 'test'][col_category].value_counts()).reset_index())
    print(f"Number of controls variants and elements:")
    print(pd.DataFrame(df[df[col_class] != 'test'][col_category].value_counts()).reset_index())
    print(f"Number distribution of controls between class:")
    print(df[df[col_class] != 'test'][col_class].value_counts())
    print(f"Number of ref and alt for tested:")
    print(df[df[col_class] == 'test'][col_allele].value_counts())
    print(f"Number of ref and alt for controls:")
    print(df[df[col_class] != 'test'][col_allele].value_counts())
    

In [23]:
# sanity check numbers to exect: 
ref_of_tested = 18582
alt_of_tested = 46458
elements_of_tested = 8900
all_of_tested = 73940
all_variants = 65040

all_controls = 6275
scramble_control = 506



In [46]:
metadata_summary_numbers(tested_pre_metadata_df)

-----------Summary statics of the current version of metadata file: -------------
The number of unique rows in the table is: 73940
Number of tested (73940) and control sequences (0)
Number of class distribution:
  class  count
0  test  73940
Number distribution of category from tested rows:
  category  count
0  variant  64946
1  element   8994
Number of controls variants and elements:
Empty DataFrame
Columns: [category, count]
Index: []
Number distribution of controls between class:
Series([], Name: count, dtype: int64)
Number of ref and alt for tested:
allele
alt    46374
ref    18572
NA      8994
Name: count, dtype: int64
Number of ref and alt for controls:
Series([], Name: count, dtype: int64)


In [45]:
metadata_summary_numbers(pre_metadata_df)

-----------Summary statics of the current version of metadata file: -------------
The number of unique rows in the table is: 80215
Number of tested (73940) and control sequences (6275)
Number of class distribution:
                      class  count
0                      test  73940
1  element inactive control   5986
2    element active control    289
Number distribution of category from tested rows:
  category  count
0  variant  64946
1  element   8994
Number of controls variants and elements:
    category  count
0    element   5408
1  scrambled    667
2  synthetic    200
Number distribution of controls between class:
class
element inactive control    5986
element active control       289
Name: count, dtype: int64
Number of ref and alt for tested:
allele
alt    46374
ref    18572
NA      8994
Name: count, dtype: int64
Number of ref and alt for controls:
allele
NA    6275
Name: count, dtype: int64


### Investigate gc_kircher controls and modify headers for new design
- GOAL: headers of all sequences have meaningful and parsable header (process: finish metadata file and take header from there)
- min and max length of headers from gc_kircher (4226 - 4252)
- make new design file with unified headers (~ instead , ; * instead of >), shortened headers for gc_kircher controls
- make new header match table: unified headers vs modified headers
- make new variant map for all variants
  - how many variants from the original design are there?

##### GC_Kircher header

In [65]:
gc_kircher = design_df[design_df['label'] == "GC_Kircher"]
gc_kircher['header_length'] = gc_kircher['header'].apply(len)
gc_kircher
gc_kircher['header_length'].max() # min header length: 4226 max header length: 4252

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-69/ipykernel_2175194/3245189294.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gc_kircher['header_length'] = gc_kircher['header'].apply(len)


4252

#### Create new files:

In [29]:
# useful functions
def is_variant_control(row):
    """Returns if row is variant control"""
    return True if (row[col_class] == 'variant positive control' or row[col_class] == 'variant negative control') else False


def shorten_kircher_headers(header):
    """
    Shorten kircher headers
    Make an informative new header which is unique and refers to the old longer header
    """
    if 'GC_Kircher'.lower() == hf.get_label(header).lower():
        return header
    # removed because assignment has still the long headers (Need to run MPRAsnakeflow again with the shorter ones)
        # tiling_info = header.split('KircherControls_')[-1]
        # if "REF_" in header:
        #     return 'GC_Kircher:REF_' + tiling_info
        # elif "ALT_" in header:
        #     return 'GC_Kircher:ALT_' + tiling_info
        # else:
        #     return 'GC_Kircher:oligo_' + tiling_info
    else:
        return header


def shorten_and_unify_header(header):
    """
    Function returns unified header and shortens the header if it is from GC_Kircher (expects it to be in the raw state (long))
    """
    modified_header = shorten_kircher_headers(header)
    if "," in modified_header:
        modified_header = modified_header.replace(",", "~")
    if ">" in modified_header:
        modified_header = modified_header.replace(">", "*")
    return modified_header


def make_gc_kircher_variant_unique(row):
    """make the variant id of GC_Kircher unique (input row of variant region map)
    expected row['ALT_ID']
    # TODO: fix Variant non uniqueness of control files
    """
    if 'GC_Kircher'.lower() == hf.get_label(row['ALT_ID']).lower():
        return row['ALT_ID']
    if 'GC_Selvarajan'.lower() == hf.get_label(row['REF_ID']).lower():
        tiling_info = get_tiling_for_controls(row['REF_ID'])
        return row['Variant'] + tiling_info
    if 'GC_Mendelian_variants'.lower() == hf.get_label(row['REF_ID']).lower():
        return row['ALT_ID']
    else:
        return row['Variant']
    

def get_tiling_for_controls(REF_ID):
    """Returns tiling information for controls"""
    if 'GC_Selvarajan'.lower() == hf.get_label(REF_ID).lower():
        # GC_Selvarajan:REF_rs11660614|STARR-seq-HepG2~rs11664887|STARR-seq-HepG2_fwd_tile1-1
        return '_'.join(REF_ID.split('_')[-2:])
    else:
        return ""
    
def change_to_unified_header(ID):
    """
    Changes deprecated headers to the design style of naming headers/IDs
    "," => "~"
    ">" => "*"
    """
    if "," in ID:
        ID = ID.replace(",","~")
    if ">" in ID:
        ID = ID.replace('>', '*')
    return ID
    

def change_to_deprecated_header(ID):
    """
    Changes headers to the deprecated design style of naming headers/IDs
    "~" => ","
    "*" => ">"
    """
    if "~" in ID:
        ID = ID.replace("~",",")
    if "*" in ID:
        ID = ID.replace('*', '>')
    return ID

In [21]:
design_fasta = config['files']['final_design']['design_fasta']
design_df = create_fasta_df_from_one_line_sequence_fasta(design_fasta, False)
design_df

,header,sequence
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...
...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...


In [22]:
# check design_df
design_df[design_df['header'].str.startswith('GC_Kircher')]

,header,sequence
74399,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTAAAGCCCTGTCCGGTGAGGGGGCAGAAGGAC...
74400,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTAGCCCCTCTCCTTTTCCTGGACTCTGGCCGT...
74401,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTCAGTCATGTGTTAAGTTGCGCTTCTTTGCTG...
74402,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTTCTGGGTTCTGGTGTCCACTCACCCACCCCA...
74403,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTATTGGCTCTCTTCTTCAAAGGACCAGGTCCT...
...,...,...
74597,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTTCTGGGTTCTGGTGTCCACTCACCCACCCCA...
74598,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTATTGGCTCTCTTCTTCAAAGGACCAGGTCCT...
74599,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTTCTGGGTTCTGGTGTCCACTCACCCACCCCA...
74600,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTATTGGCTCTCTTCTTCAAAGGACCAGGTCCT...


In [19]:
# # new design file
# design_df['deprecated_header'] = design_df['header'].apply(change_to_deprecated_header)
# design_df['design_style_header'] = design_df['header'].apply(shorten_kircher_headers)
# design_df

# # write file in fasta format
# design_df_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/design_file_improved_header.fa"
# with open(design_df_path, 'w') as f:
#     for index, row in design_df.iterrows():
#         f.write('>' + row['design_style_header'] + '\n' + row[col_sequence] + '\n')

# # write shortened GC_Kircher match table
# # store in `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/GC_Kircher_shortened_headers_match_table.tsv`
# gc_kircher_match_table_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/GC_Kircher_shortened_headers_match_table.tsv"
# design_df[design_df['header'].str.startswith('GC_Kircher')][['design_style_header', 'deprecated_header']].to_csv(gc_kircher_match_table_path, sep='\t', index=False)

# # # new design_header_match_table
# design_match_table_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/design_header_match_table.tsv"
# design_match_file = design_df[['design_style_header', 'deprecated_header']]
# design_match_file.to_csv(design_match_table_path, sep="\t", index=False)

In [23]:
# check GC_Kircher headers
design_df[design_df['header'].str.startswith('GC_Kircher')]['design_style_header'].nunique() # shape and nunique = 203 => unique

KeyError: 'design_style_header'

In [24]:
# get groups and numbers from the design file
design_df['label'] = design_df['header'].apply(hf.get_label)
design_df

design_df['label'].value_counts()

label
cardiac_neuro_cava_random            73940
MK                                    2397
C_positive_heart_AB                    909
GC_Selvarajan                          364
GC_Vista                               256
C_negative_heart_MK                    243
C_negative_neuron_MK                   222
C_negative_neuron_NP                   217
GC_Mendelian_variants                  209
GC_Kircher                             203
C_SLEA                                 200
GC_Cort_Chengyu                        185
C_positive_neuron_NP                    99
C_positive_heart_MK                     97
C_positive_heart_CAD                    97
C_positive_neuron_MK                    96
C_positive_neuron_CD                    94
GC_GABA_Chengyu                         85
GC_DNase_positive_shuffeled             55
GC_Atrial_fib                           45
GC_DNase_positive                       41
GC_Glut_Chengyu                         40
GC_Mohlke                               34
GC_DN

##### Sanity check variant map
- for tested variants: (check if Variant column is unique)
  - shape: 46374  unique ids: 45737
- for controls: shape 670 unique ids: 420
- how many variant map mentioned sequences are in the final design
- Numbers according to variant region map:
  - label
    - GC_Selvarajan            198
    - GC_Kircher               198
    - GC_Mendelian_variants    174
    - C_positive_heart_CAD      49
    - GC_Atrial_fib             23
    - GC_Mohlke                 20
    - GC_Liang                   8

  - check which headers of alt and ref could not be matched
    - ref: 
      - "GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3" - tiling what does this mean?
      - GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791413A>C|SHH (no GC_Mendelian_variants:ALT_chr7:156791472C>T)
      - GC_Mohlke:ALT_NC000010.11|100315721|G|A|MohlkeNonHepControl_fwd_tile1-1_NC000010_11_100315721_G_A
  
  - matchable 
    - ref 
      - label
        - GC_Selvarajan            198
        - GC_Kircher               198
        - GC_Mendelian_variants    163
        - C_positive_heart_CAD      48
        - GC_Atrial_fib             23
        - GC_Mohlke                 18
        - GC_Liang                   8
    - alt 
      - label
        - GC_Selvarajan            198
        - GC_Kircher               198
        - GC_Mendelian_variants    161
        - C_positive_heart_CAD      48
        - GC_Atrial_fib             23
        - GC_Mohlke                 17
        - GC_Liang                   8

In [25]:
# get headers from design + load headers from variant region map (references) and match with modified header "~" => ","
variant_region_map_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/variant_region_map.tsv'
variant_region_map = pd.read_csv(variant_region_map_path, sep="\t")
variant_region_map
variant_region_map['label'] = variant_region_map['Variant'].apply(hf.get_label)

#### Create variant region map for controls and merge with deduplicated variant region map of cardiac
- question: do I need the region column?
- is it required that the variant Id is unique (I guess yes)


In [30]:
control_variant_map = variant_region_map[variant_region_map['label'] != "cardiac_neuro_cava_random"]
control_variant_map # 670 

# add column with tile information for the variant controls with the non unique ids 
# control_variant_map['tile']

all_var_controls_ref = set(control_variant_map['REF_ID'])
all_var_controls_alt = set(control_variant_map['ALT_ID'])


# filter for matchable headers in design
design_match_table_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/design_header_match_table.tsv"
design_match_table = pd.read_csv(design_match_table_path, sep="\t")
design_match_table
# REF_ID: first variant and ref is one table 
ref_variant_map = control_variant_map[['Variant', 'REF_ID', 'label']]
pre_ref_merged_design_match_table = design_match_table.merge(ref_variant_map, left_on='deprecated_header', right_on='REF_ID', how='left')
ref_merged_design_match_table = pre_ref_merged_design_match_table[~pre_ref_merged_design_match_table['Variant'].isna()]
ref_merged_design_match_table # 656 rows 
merged_ref_controls = set(ref_merged_design_match_table['REF_ID'])

# ALT_ID: variant and alt_id is another table 
alt_variant_map = control_variant_map[['Variant', 'ALT_ID', 'label']]
pre_alt_merged_design_match_table = design_match_table.merge(alt_variant_map, left_on='deprecated_header', right_on='ALT_ID', how='left')
alt_merged_design_match_table = pre_alt_merged_design_match_table[~pre_alt_merged_design_match_table['Variant'].isna()]
alt_merged_design_match_table # 653 
merged_alt_controls = set(alt_merged_design_match_table['ALT_ID'])
# print(len(merged_ref_controls))
# print(len(merged_alt_controls))

not_matchable_ref = all_var_controls_ref - merged_ref_controls
not_matchable_alt = all_var_controls_alt - merged_alt_controls
print("Number of not matched ref and alt headers:")
print(len(not_matchable_ref))
print(len(not_matchable_alt))

# filter matched header
control_variant_map = control_variant_map[control_variant_map['REF_ID'].isin(merged_ref_controls)]
control_variant_map = control_variant_map[control_variant_map['ALT_ID'].isin(merged_alt_controls)]


# change header based on matching with design_df 
control_variant_map['design_style_REF_ID'] = control_variant_map['REF_ID'].apply(shorten_and_unify_header)
control_variant_map['design_style_ALT_ID'] = control_variant_map['ALT_ID'].apply(shorten_and_unify_header)


control_variant_map = control_variant_map[['Variant', 'Region', 'design_style_REF_ID', 'design_style_ALT_ID']]
control_variant_map['Region'] = "NA"
control_variant_map.columns = ['Variant', 'Region', 'REF_ID', 'ALT_ID']
control_variant_map['Variant'] = control_variant_map.apply(make_gc_kircher_variant_unique, axis=1)
control_variant_map # 653 

# TODO: Check if the Region column is useful






Number of not matched ref and alt headers:
4
17


,Variant,Region,REF_ID,ALT_ID
46374,GC_Atrial_fib:rs74541936,NA,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...
46375,GC_Atrial_fib:rs34292822,NA,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...
46376,GC_Atrial_fib:rs12754189,NA,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...
46377,GC_Atrial_fib:rs36088503,NA,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...
46378,GC_Atrial_fib:rs76749863,NA,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...
...,...,...,...,...
47039,C_positive_heart_CAD:rs7865618,NA,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
47040,C_positive_heart_CAD:rs4977757,NA,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
47041,C_positive_heart_CAD:rs1537373,NA,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373
47042,C_positive_heart_CAD:rs10811656,NA,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656


In [31]:
control_variant_map[control_variant_map['REF_ID'].isna()]

,Variant,Region,REF_ID,ALT_ID


In [32]:
control_variant_map[control_variant_map['REF_ID'].str.startswith('GC_Mendelian_variants')]
# control_variant_map[control_variant_map['REF_ID'].str.startswith('GC_Kircher')]

,Variant,Region,REF_ID,ALT_ID
46821,GC_Mendelian_variants:ALT_chr1:21564170G*A|ALP...,NA,GC_Mendelian_variants:REF_chr1:21564170G*A|ALPL,GC_Mendelian_variants:ALT_chr1:21564170G*A|ALP...
46822,GC_Mendelian_variants:ALT_chr1:209816133C*CA|I...,NA,GC_Mendelian_variants:REF_chr1:209816133C*CA|IRF6,GC_Mendelian_variants:ALT_chr1:209816133C*CA|I...
46823,GC_Mendelian_variants:ALT_chr10:23219376A*C|PT...,NA,GC_Mendelian_variants:REF_chr10:23219376A*C|PTF1A,GC_Mendelian_variants:ALT_chr10:23219376A*C|PT...
46824,GC_Mendelian_variants:ALT_chr10:23219434A*G|PT...,NA,GC_Mendelian_variants:REF_chr10:23219434A*G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219434A*G|PT...
46825,GC_Mendelian_variants:ALT_chr10:23219436A*G|PT...,NA,GC_Mendelian_variants:REF_chr10:23219436A*G|PTF1A,GC_Mendelian_variants:ALT_chr10:23219436A*G|PT...
...,...,...,...,...
46990,GC_Mendelian_variants:ALT_chr9:101435912C*T|AL...,NA,GC_Mendelian_variants:REF_chr9:101435912C*T|ALDOB,GC_Mendelian_variants:ALT_chr9:101435912C*T|AL...
46991,GC_Mendelian_variants:ALT_chrX:38352331A*G|OTC...,NA,GC_Mendelian_variants:REF_chrX:38352331A*G|OTC,GC_Mendelian_variants:ALT_chrX:38352331A*G|OTC...
46992,GC_Mendelian_variants:ALT_chrX:55028202A*G|ALA...,NA,GC_Mendelian_variants:REF_chrX:55028202A*G|ALAS2,GC_Mendelian_variants:ALT_chrX:55028202A*G|ALA...
46993,GC_Mendelian_variants:ALT_chrX:55031184G*C|ALA...,NA,GC_Mendelian_variants:REF_chrX:55031184G*C|ALAS2,GC_Mendelian_variants:ALT_chrX:55031184G*C|ALA...


In [33]:
def compare_shape_unique_variant_column(df, group):
    """compare with a function the shape and the number of unique variant rows"""
    groups_df = df[df['REF_ID'].str.startswith(group)]
    if groups_df.shape[0] != groups_df['Variant'].nunique():
        print(group, "Group has non unique variant ids: (all, unique)", groups_df.shape[0], groups_df['Variant'].nunique())
    else:
        print(group, ' has unique variant ids')
        

group_list = ['GC_Selvarajan', 'GC_Kircher', 'GC_Mendelian_variants', 'C_positive_heart_CAD', 'GC_Atrial_fib', 'GC_Mohlke', 'GC_Liang'] 

for group in group_list:
    compare_shape_unique_variant_column(control_variant_map, group)

GC_Selvarajan  has unique variant ids
GC_Kircher  has unique variant ids
GC_Mendelian_variants  has unique variant ids
C_positive_heart_CAD  has unique variant ids
GC_Atrial_fib  has unique variant ids
GC_Mohlke Group has non unique variant ids: (all, unique) 17 16
GC_Liang  has unique variant ids


In [26]:
control_variant_map[control_variant_map['REF_ID'].str.startswith('GC_Kircher')]
control_variant_map[control_variant_map['REF_ID'].str.startswith('GC_Selvarajan')]
# check if the variant IDs of the other labels are unique


,Variant,Region,REF_ID,ALT_ID
46405,GC_Selvarajan:rs9660819fwd_tile1-1,NA,GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs...,GC_Selvarajan:ALT_rs9660819|STARR-seq-HepG2~rs...
46406,GC_Selvarajan:rs9661525fwd_tile1-1,NA,GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs...,GC_Selvarajan:ALT_rs9660819|STARR-seq-HepG2~rs...
46407,GC_Selvarajan:rs72856440fwd_tile1-3,NA,GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs...,GC_Selvarajan:ALT_rs2993510|STARR-seq-HepG2~rs...
46408,GC_Selvarajan:rs72856440fwd_tile2-3,NA,GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs...,GC_Selvarajan:ALT_rs2993510|STARR-seq-HepG2~rs...
46409,GC_Selvarajan:rs72856440fwd_tile3-3,NA,GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs...,GC_Selvarajan:ALT_rs2993510|STARR-seq-HepG2~rs...
...,...,...,...,...
46598,GC_Selvarajan:rs10121140fwd_tile2-3,NA,GC_Selvarajan:REF_rs10121140|STARR-seq-HepG2~r...,GC_Selvarajan:ALT_rs10121140|STARR-seq-HepG2~r...
46599,GC_Selvarajan:rs10121140fwd_tile3-3,NA,GC_Selvarajan:REF_rs10121140|STARR-seq-HepG2~r...,GC_Selvarajan:ALT_rs10121140|STARR-seq-HepG2~r...
46600,GC_Selvarajan:rs6475604fwd_tile1-1,NA,GC_Selvarajan:REF_rs6475604|STARR-seq-HepG2_fw...,GC_Selvarajan:ALT_rs6475604|STARR-seq-HepG2_fw...
46601,GC_Selvarajan:rs1333045fwd_tile1-1,NA,GC_Selvarajan:REF_rs1333045|STARR-seq-HepG2_fw...,GC_Selvarajan:ALT_rs1333045|STARR-seq-HepG2_fw...


In [230]:
control_variant_map['Variant'].nunique() # 513: 17.24

513

In [34]:
tested_variant_map = variant_region_map[variant_region_map['label'] == "cardiac_neuro_cava_random"]
tested_variant_map # 46374  

# change header based on matching with design_df 
tested_variant_map['design_style_REF_ID'] = tested_variant_map['REF_ID'].apply(shorten_and_unify_header)
tested_variant_map['design_style_ALT_ID'] = tested_variant_map['ALT_ID'].apply(shorten_and_unify_header)

tested_variant_map = tested_variant_map[['Variant', 'Region', 'design_style_REF_ID', 'design_style_ALT_ID']]
tested_variant_map.columns = ['Variant', 'Region', 'REF_ID', 'ALT_ID']
tested_variant_map

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-187/ipykernel_2594444/397280161.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_variant_map['design_style_REF_ID'] = tested_variant_map['REF_ID'].apply(shorten_and_unify_header)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-187/ipykernel_2594444/397280161.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_variant_map['design_style_ALT_ID'] = tested_variant_map['ALT_ID'].apply(shorten_and_unify_header)


,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
...,...,...,...,...
46369,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...
46370,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...
46371,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...
46372,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...


In [114]:
new_variant_region_map

,Variant,Region,REF_ID,ALT_ID,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random
...,...,...,...,...,...
47039,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618,C_positive_heart_CAD
47040,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757,C_positive_heart_CAD
47041,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373,C_positive_heart_CAD
47042,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656,C_positive_heart_CAD


In [35]:
# # concatenate tested_variant_map and control_variant_map
new_variant_region_map = pd.concat([tested_variant_map, control_variant_map])#
new_variant_region_map # 47027  

# # write new variant map
# new_variant_map_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/variant_map_with_unified_headers_mendelian_focus.tsv'
# new_variant_region_map.to_csv(new_variant_map_path, sep="\t", index=None)

In [137]:
# summary statistics
new_variant_region_map['label'] = new_variant_region_map['REF_ID'].str.split(':').str[0]
new_variant_region_map['label'].value_counts()

label
cardiac_neuro_cava_random    46374
GC_Selvarajan                  198
GC_Kircher                     198
GC_Mendelian_variants          174
C_positive_heart_CAD            49
GC_Atrial_fib                   23
GC_Mohlke                       20
GC_Liang                         8
Name: count, dtype: int64

In [88]:
# test to change the header of the GC_Kircher rows
gc_kircher_variant_map = variant_region_map[variant_region_map['label'] == 'GC_Kircher']

# shorten kircher headers
gc_kircher_variant_map['design_style_REF_ID'] = gc_kircher_variant_map['REF_ID'].apply(shorten_kircher_headers)
gc_kircher_variant_map['design_style_ALT_ID'] = gc_kircher_variant_map['ALT_ID'].apply(shorten_kircher_headers)

gc_kircher_variant_map = gc_kircher_variant_map[['Variant', 'Region', 'design_style_REF_ID', 'design_style_ALT_ID']]
gc_kircher_variant_map.columns = ['Variant', 'Region', 'REF_ID', 'ALT_ID']

# gc_kircher_variant_map.to_csv('testing.test', sep="\t", index=None)

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-139/ipykernel_3900740/1434345348.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gc_kircher_variant_map['design_style_REF_ID'] = gc_kircher_variant_map['REF_ID'].apply(shorten_kircher_headers)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-139/ipykernel_3900740/1434345348.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gc_kircher_variant_map['design_style_ALT_ID'] = gc_kircher_variant_map['ALT_ID'].apply(shorten_kircher_headers)


In [33]:
# # number of groups which are not cardiac_neuro_cava_random
control_variant_map = variant_region_map[variant_region_map['label'] != "cardiac_neuro_cava_random"]
control_variant_map # 670 
control_variant_map['label'].value_counts()


# how many of them are matchable with the new matching headers
design_match_table_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/design_header_match_table.tsv"
design_match_table = pd.read_csv(design_match_table_path, sep="\t")
design_match_table

# über ref_id mergen 


# REF_ID: first variant and ref is one table 
ref_variant_map = control_variant_map[['Variant', 'REF_ID', 'label']]
pre_ref_merged_design_match_table = design_match_table.merge(ref_variant_map, left_on='deprecated_header', right_on='REF_ID', how='left')
ref_merged_design_match_table = pre_ref_merged_design_match_table[~pre_ref_merged_design_match_table['Variant'].isna()]
ref_merged_design_match_table # 656 rows 
merged_ref_controls = set(ref_merged_design_match_table['REF_ID'])


# ALT_ID: variant and alt_id is another table 
alt_variant_map = control_variant_map[['Variant', 'ALT_ID', 'label']]
pre_alt_merged_design_match_table = design_match_table.merge(alt_variant_map, left_on='deprecated_header', right_on='ALT_ID', how='left')
alt_merged_design_match_table = pre_alt_merged_design_match_table[~pre_alt_merged_design_match_table['Variant'].isna()]
alt_merged_design_match_table # 653 
merged_alt_controls = set(alt_merged_design_match_table['ALT_ID'])
# could only match 653 alternative and 656 references
# check which references and alternatives could not be matched
# make set of all variant control references and all variant control alternatives and the matchable references and alternatives respectively
# all control variants
all_var_controls_ref = set(control_variant_map['REF_ID'])
all_var_controls_alt = set(control_variant_map['ALT_ID'])

# not matchable references and controls
not_matchable_ref = all_var_controls_ref - merged_ref_controls
not_matchable_alt = all_var_controls_alt - merged_alt_controls


# 1-2 sentences about these control groups

##### Investigate not matchable headers 

In [34]:
ref_merged_design_match_table['label'].value_counts()
# label
# GC_Selvarajan            198
# GC_Kircher               198
# GC_Mendelian_variants    163
# C_positive_heart_CAD      48
# GC_Atrial_fib             23
# GC_Mohlke                 18
# GC_Liang                   8

label
GC_Selvarajan            198
GC_Kircher               198
GC_Mendelian_variants    163
C_positive_heart_CAD      48
GC_Atrial_fib             23
GC_Mohlke                 18
GC_Liang                   8
Name: count, dtype: int64

In [35]:
alt_merged_design_match_table['label'].value_counts()
# label
# GC_Selvarajan            198
# GC_Kircher               198
# GC_Mendelian_variants    161
# C_positive_heart_CAD      48
# GC_Atrial_fib             23
# GC_Mohlke                 17
# GC_Liang                   8

label
GC_Selvarajan            198
GC_Kircher               198
GC_Mendelian_variants    161
C_positive_heart_CAD      48
GC_Atrial_fib             23
GC_Mohlke                 17
GC_Liang                   8
Name: count, dtype: int64

##### Compare not matchable headers

In [36]:
not_matchable_ref

{'C_positive_heart_CAD:REF_rs12721051',
 'GC_Mendelian_variants:REF_chr7:156791472C>T|SHH',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3',
 'GC_Mohlke:REF_NC000010.11|100315721|G|A|MohlkeNonHepControl_fwd_tile1-1'}

In [37]:
not_matchable_alt

{'C_positive_heart_CAD:ALT_rs12721051_rs12721051',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791413A>C|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791459T>C|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791472C>G|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791472C>T|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791474G>A|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791480G>A|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791542A>C|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791547A>G|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791571T>A|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791579C>T|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791472C>T|SHH_chr7:156791581A>G|SHH',
 'GC_Mendelian_variants:ALT_chr8:11703860G>T|GATA4_chr8:11703890AG>A|GATA4',
 'GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:1

In [38]:
control_variant_map

,Variant,Region,REF_ID,ALT_ID,label
46374,GC_Atrial_fib:rs74541936,GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib
46375,GC_Atrial_fib:rs34292822,GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib
46376,GC_Atrial_fib:rs12754189,GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib
46377,GC_Atrial_fib:rs36088503,GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib
46378,GC_Atrial_fib:rs76749863,"GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs7...",GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib
...,...,...,...,...,...
47039,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618,C_positive_heart_CAD
47040,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757,C_positive_heart_CAD
47041,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373,C_positive_heart_CAD
47042,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656,C_positive_heart_CAD


In [116]:
# # new design file
# design_df['deprecated_header'] = design_df['header'].apply(change_to_deprecated_header)
# design_df['header'] = design_df['header'].apply(shorten_kircher_headers)
# design_df
# design_df_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/design_file_improved_header.fa"
# design_file = design_df[['header', col_sequence]]
# design_file.to_csv(design_df_path, sep='\t', index=False)
# new design_header_match_table
# design_match_table_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/design_header_match_table.tsv"
# design_match_file = design_df[['header', 'deprecated_header']]
# design_match_file.to_csv(design_match_table_path, sep="\t", index=False)

In [64]:
# # Shorten kircher header and store matching table
gc_kircher = design_df[design_df['label'] == "GC_Kircher"]
gc_kircher = gc_kircher[['header']]
gc_kircher['short_GC_Kircher_headers'] = gc_kircher['header'].apply(shorten_kircher_headers)

# unique short_GC_Kircher_headers
# gc_kircher['short_GC_Kircher_headers'].nunique()
# store in `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/GC_Kircher_shortened_headers_match_table.tsv`
# gc_kircher_match_table_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/GC_Kircher_shortened_headers_match_table.tsv"
# gc_kircher.to_csv(gc_kircher_match_table_path, sep='\t', index=False)

##### Sanity check if matching works with gc_kircher

In [65]:
gc_variant_map = variant_region_map[variant_region_map['Variant'].str.startswith('GC_Kircher')]
gc_variant_map # 198 is expected, because in design 198 Kircher:ALT_
# is variant column unique
# gc_variant_map['Variant'].nunique() # 100 unique Variant header
gc_variant_map[['REF_ID', 'Variant']]
gc_kircher['modified_header'] = gc_kircher['header'].apply(lambda x: x.replace('~', ','))

# show that all columns are unique
gc_kircher['header'].nunique()

gc_kircher['short_GC_Kircher_headers'].nunique()

gc_kircher['modified_header'].nunique()

# # all 203 => all unique
# gc_kircher_match_table_path = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/GC_Kircher_shortened_headers_match_table.tsv"
# gc_kircher.to_csv(gc_kircher_match_table_path, sep='\t', index=False)

203

In [66]:
gc_variant_map['REF_ID'].unique()

array(['GC_Kircher:REF_NC000001.11|109274794|C|T|KircherControls,NC000001.11|109274836|C|T|KircherControls,NC000001.11|109274840|A|C|KircherControls,NC000001.11|109274845|C|A|KircherControls,NC000001.11|109274846|T|G|KircherControls,NC000001.11|109274852|T|G|KircherControls,NC000001.11|109274857|T|C|KircherControls,NC000001.11|109274860|C|G|KircherControls,NC000001.11|109274865|C|A|KircherControls,NC000001.11|109274869|G|C|KircherControls,NC000001.11|109274884|G|C|KircherControls,NC000001.11|109274885|T|C|KircherControls,NC000001.11|109274886|C|A|KircherControls,NC000001.11|109274887|A|G|KircherControls,NC000001.11|109274888|T|G|KircherControls,NC000001.11|109274892|T|A|KircherControls,NC000001.11|109274908|T|G|KircherControls,NC000001.11|109274910|C|A|KircherControls,NC000001.11|109274912|G|T|KircherControls,NC000001.11|109274917|G|T|KircherControls,NC000001.11|109274922|T|C|KircherControls,NC000001.11|109274923|G|C|KircherControls,NC000001.11|109274924|G|C|KircherControls,NC000001.11

In [67]:
# merge on REF_ID
merge_on_ref = gc_kircher.merge(gc_variant_map, left_on="modified_header", right_on="REF_ID", how="left")
merge_on_ref[~merge_on_ref['Variant'].isna()]['short_GC_Kircher_headers'].unique()

# merge_on_ref['short_GC_Kircher_headers']

array([], dtype=object)

In [68]:
# merge design to variant map
merge_on_ref = gc_variant_map.merge(gc_kircher, left_on="ALT_ID", right_on="modified_header", how="left")
merge_on_ref[~merge_on_ref['short_GC_Kircher_headers'].isna()]

,Variant,Region,REF_ID,ALT_ID,label,header,short_GC_Kircher_headers,modified_header


In [ ]:
# Shorten headers in the variant region map

# /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/variant_region_map.tsv

# filter for GC_Kircher

# shorten headers as well and try to find in 

In [39]:
old_variant_region_map_with_controls = config['files']['final_design']['variant_table']
old_variant_region_map_w_controls = pd.read_csv(old_variant_region_map_with_controls, sep="\t")
old_variant_region_map_w_controls['label'] = old_variant_region_map_w_controls['REF_ID'].apply(get_label)

In [40]:
old_variant_region_map_w_controls
# change the header names of the ALT and REF id

# variant column unique ids
old_variant_region_map_w_controls['Variant'].nunique() # only 46157 => not all varaints have a unique variant ID

46157

In [41]:
# check how many only controls
only_controls_old_variant_map = old_variant_region_map_w_controls[old_variant_region_map_w_controls['label'] != 'cardiac_neuro_cava_random']
print(only_controls_old_variant_map['Variant'].shape[0])
only_controls_old_variant_map['Variant'].nunique()
print("REF_ID unique: ", only_controls_old_variant_map['REF_ID'].nunique())
print("ALT_ID unique: ", only_controls_old_variant_map['ALT_ID'].nunique())

# change 

670
REF_ID unique:  311
ALT_ID unique:  670


In [42]:
design_df

,header,sequence,modified_header,header_new
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK:tile_2240|chr1-116244322+116244591|scramble...,MK:tile_2240|chr1-116244322+116244591|scramble...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,MK:tile_6675|chr11-2374617+2374886|scramble_ne...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK:tile_18415|chr17-71181691+71181960|scramble...,MK:tile_18415|chr17-71181691+71181960|scramble...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK:tile_14356|chr15-67031618+67031887|scramble...,MK:tile_14356|chr15-67031618+67031887|scramble...


In [43]:
# change REF_ and ALT_ID from controls to new style and merge with modified version of duplicated variant match file
only_controls_old_variant_map['modified_REF_ID'] = only_controls_old_variant_map['REF_ID'].apply(change_to_unified_header)
only_controls_old_variant_map['modified_ALT_ID'] = only_controls_old_variant_map['ALT_ID'].apply(change_to_unified_header)
# verify by checking how many can be matched to the design numbers
merged_controls_alt = only_controls_old_variant_map.merge(design_df[['header', col_sequence]], left_on="modified_ALT_ID", right_on="header", how="left")
merged_controls_alt = merged_controls_alt[~merged_controls_alt[col_sequence].isna()] # from ALT_ID: 670 only 205 matched
merged_controls_ref = only_controls_old_variant_map.merge(design_df[['header', col_sequence]], left_on="modified_REF_ID", right_on="header", how="left")
merged_controls_ref = merged_controls_ref[~merged_controls_ref[col_sequence].isna()] # from ALT_ID: 670 only 205 matched

# check which headers could not be matched and look for them manually
# sets of matched ref and alt_ids and set of all old_variant_ref and alt ids
old_variant_map_alt = set(only_controls_old_variant_map['ALT_ID'])
old_variant_map_ref = set(only_controls_old_variant_map['REF_ID'])
matched_controls_ref = set(merged_controls_ref['REF_ID'])
matched_variant_alt = set(merged_controls_alt['ALT_ID'])

unmatched_alt = old_variant_map_alt - matched_variant_alt
unmatched_ref = old_variant_map_ref - matched_controls_ref

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-139/ipykernel_3900740/520970034.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  only_controls_old_variant_map['modified_REF_ID'] = only_controls_old_variant_map['REF_ID'].apply(change_to_unified_header)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-139/ipykernel_3900740/520970034.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  only_controls_old_variant_map['modified_ALT_ID'] = only_controls_old_variant_map['ALT_ID'].apply(change_to_unified_header)


In [44]:
print(len(unmatched_alt)) # 17
print(len(unmatched_ref)) # 4
unmatched_alt
unmatched_ref

17
4


{'C_positive_heart_CAD:REF_rs12721051',
 'GC_Mendelian_variants:REF_chr7:156791472C>T|SHH',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3',
 'GC_Mohlke:REF_NC000010.11|100315721|G|A|MohlkeNonHepControl_fwd_tile1-1'}

In [45]:
# new variant region map: 
# take all rows which alt_id is not in unmatched_alt and ref_id not in unmatched_ref
only_control_var_map_new = only_controls_old_variant_map[~only_controls_old_variant_map['REF_ID'].isin(list(unmatched_ref))]
# new unique = 311 - 4
only_control_var_map_new['REF_ID'].nunique()

307

In [45]:
# take all rows which alt_id is not in unmatched_alt and ref_id not in unmatched_ref
only_control_var_map_new = only_control_var_map_new[~only_control_var_map_new['ALT_ID'].isin(list(unmatched_alt))]
# new unique = 670 - 17
only_control_var_map_new['ALT_ID'].nunique()

653

In [47]:
design_df

,header,sequence,modified_header,header_new
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....
...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK:tile_2240|chr1-116244322+116244591|scramble...,MK:tile_2240|chr1-116244322+116244591|scramble...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,MK:tile_6675|chr11-2374617+2374886|scramble_ne...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK:tile_18415|chr17-71181691+71181960|scramble...,MK:tile_18415|chr17-71181691+71181960|scramble...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK:tile_14356|chr15-67031618+67031887|scramble...,MK:tile_14356|chr15-67031618+67031887|scramble...


In [49]:
only_control_var_map_new

,Variant,Region,REF_ID,ALT_ID,label,modified_REF_ID,modified_ALT_ID
46374,GC_Atrial_fib:rs74541936,GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...
46375,GC_Atrial_fib:rs34292822,GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...
46376,GC_Atrial_fib:rs12754189,GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...
46377,GC_Atrial_fib:rs36088503,GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...
46378,GC_Atrial_fib:rs76749863,"GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs7...",GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...
...,...,...,...,...,...,...,...
47039,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618,C_positive_heart_CAD,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
47040,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757,C_positive_heart_CAD,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
47041,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373,C_positive_heart_CAD,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373
47042,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656,C_positive_heart_CAD,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656


#### Sanity check the deduplicated variant region map
- change variant column: "," -> "~"
- check if all ref and alt ids can be matched with the design file
- what about the region column

In [58]:
# only_control_var_map_new
# # check how many regions in the variant region map can be found in the design
# only_control_var_map_new['Region'].nunique()
# only_control_var_map_new[only_control_var_map_new['Region'].str.contains(',')]
# # merge with design_df with modified header
# merged_with_design = only_control_var_map_new.merge(design_df, left_on="Region", right_on="deprecated_header", how="left")

# merged_with_design[~merged_with_design['design_style_header'].isna()] # no entry in the region file matched the design (why are there here?)

# # look for them manually
# only_control_var_map_new['Region'].to_list()
# # ask mohan if you dont understand what the entries mean (done)


# load new variant region map

# variant region map deduplicated does not include control variants
# name changes observed => sanity checking to build trust about the file and its ability to match information to our designed sequences
variant_region_map_tested_deduplicated = "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/cardiac_neuro_cava_random/variant_region_map_deduplicated.tsv"
variant_map_deduplicated_tested = pd.read_csv(variant_region_map_tested_deduplicated, sep="\t")
variant_map_deduplicated_tested

# modify header to match the design headers
# change REF_ and ALT_ID from controls to new style and merge with modified version of duplicated variant match file
variant_map_deduplicated_tested['modified_Variant'] = variant_map_deduplicated_tested['Variant'].apply(change_to_unified_header)
variant_map_deduplicated_tested['modified_REF_ID'] = variant_map_deduplicated_tested['REF_ID'].apply(change_to_unified_header)
variant_map_deduplicated_tested['modified_ALT_ID'] = variant_map_deduplicated_tested['ALT_ID'].apply(change_to_unified_header)
# Variant: "," REF and ALT_id: "~"
# merge with design_df one of the headers which also holds 

# check how many are matchable
# verify by checking how many can be matched to the design numbers
merged_tested_deduplicated_alt = variant_map_deduplicated_tested.merge(design_df[['header', col_sequence]], left_on="modified_ALT_ID", right_on="header", how="left")
merged_tested_deduplicated_alt = merged_tested_deduplicated_alt[~merged_tested_deduplicated_alt[col_sequence].isna()]
merged_tested_deduplicated_ref = variant_map_deduplicated_tested.merge(design_df[['header', col_sequence]], left_on="modified_REF_ID", right_on="header", how="left")
merged_tested_deduplicated_ref = merged_tested_deduplicated_ref[~merged_tested_deduplicated_ref[col_sequence].isna()]

# check which headers could not be matched and look for them manually
# sets of matched ref and alt_ids and set of all old_variant_ref and alt ids
old_tested_variant_map_alt = set(variant_map_deduplicated_tested['ALT_ID'])
old_tested_variant_map_ref = set(variant_map_deduplicated_tested['REF_ID'])
matched_tested_variant_ref = set(merged_tested_deduplicated_ref['REF_ID'])
matched_tested_variant_alt = set(merged_tested_deduplicated_alt['ALT_ID'])
# old_tested_variant_map_alt
# old_tested_variant_map_ref
# unmatched_tested_alt = old_tested_variant_map_alt - matched_tested_variant_alt # empty sets 
# unmatched_tested_ref = old_tested_variant_map_ref - matched_tested_variant_ref # empty sets
# => all elements could be matched and found => already 
# unmatched_tested_alt
# unmatched_tested_ref


{'cardiac_neuro_cava_random:ALT_DSP|ENSG00000096696.15|EH38E3689290_fwd_tile1-1_DSP|ENSG00000096696.15|EH38E3689290|6-7574303-T-C',
 'cardiac_neuro_cava_random:ALT_CFAP91|ENSG00000183833.16|EH38E2230889_fwd_tile1-1_CFAP91|ENSG00000183833.16|EH38E2230889|3-119680232-T-C',
 'cardiac_neuro_cava_random:ALT_ARHGAP31|ENSG00000031081.11|EH38E3533530_fwd_tile1-1_ARHGAP31|ENSG00000031081.11|EH38E3533530|3-119332996-C-T',
 'cardiac_neuro_cava_random:ALT_CDKN2A|ENSG00000147889.18|EH38E3877989~CDKN2B|ENSG00000147883.12|EH38E3877989_rev_tile1-1_CDKN2B|ENSG00000147883.12|EH38E3877989|9-21983787-G-A',
 'cardiac_neuro_cava_random:ALT_TRIO|ENSG00000038382.23|EH38E2357988_fwd_tile1-1_TRIO|ENSG00000038382.23|EH38E2357988|5-14100592-A-G',
 'cardiac_neuro_cava_random:ALT_TFAP2B|ENSG00000008196.13|EH38E3711999_fwd_tile1-1_TFAP2B|ENSG00000008196.13|EH38E3711999|6-50846016-T-C',
 'cardiac_neuro_cava_random:ALT_SLC22A5|ENSG00000197375.14|EH38E3660515_fwd_tile1-1_SLC22A5|ENSG00000197375.14|EH38E3660515|5-132411

In [67]:
design_df['deprecated_header'].str.contains(',').sum()

2089

In [64]:
merged_region_with_design = variant_map_deduplicated_tested.merge(design_df, left_on="Region", right_on="", how="left")

2089

In [59]:
len(old_tested_variant_map_alt)

46374

In [60]:
len(old_tested_variant_map_ref)

18572

set()

In [46]:
only_control_var_map_new['label'].value_counts()

label
GC_Selvarajan            198
GC_Kircher               198
GC_Mendelian_variants    161
C_positive_heart_CAD      48
GC_Atrial_fib             23
GC_Mohlke                 17
GC_Liang                   8
Name: count, dtype: int64

In [6]:
# investigate variant controls
pre_metadata_df[col_class].value_counts()

# class
# test                        73940
# element inactive control     5019
# variant negative control      967
# element active control        198
# variant positive control       91

class
test                        73940
element inactive control     5019
variant negative control      967
element active control        198
variant positive control       91
Name: count, dtype: int64

### Add variants to their reference
- read variant region map
- for each reference get the number of variants
- then get the variants coresponding to the reference

In [18]:
var_reg_map = '/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/variant_region_map/variant_region_map.tsv.gz'

variant_region_map = pd.read_csv(var_reg_map, sep='\t')
print('Number of rows in variant region map: ', variant_region_map.shape[0])
print('Number of rows in variant region map of cardiac_neuro_cava_random: ', variant_region_map.loc[variant_region_map['Variant'].str.startswith('cardiac_neuro_cava_random')].shape[0])
print('Number of rows in variant region map of controls: ', variant_region_map.loc[~variant_region_map['Variant'].str.startswith('cardiac_neuro_cava_random')].shape[0])
# get number of unique REF_ID rows: 18883
ref_ids = variant_region_map['REF_ID'].unique()
len(ref_ids)
# get number of unique ALT_ID rows: 47044
alt_ids = variant_region_map['ALT_ID'].unique()
len(alt_ids)

## go through all references and get list of variants for each reference
tested_sequences_map = variant_region_map[variant_region_map['REF_ID'].str.startswith('cardiac_neuro_cava_random')]
tested_sequences_map.groupby('REF_ID')['ALT_ID'].apply(list).reset_index(name='alternatives')
tested_sequences_map.groupby('REF_ID')['ALT_ID'].apply(list).to_dict()

# get maximal number of alternatives per reference: 41 (cardiac_neuro_cava_random:REF_H1-3|ENSG00000124575.7|EH38E3697654_rev_tile1-1)
count_tbl = pd.DataFrame(tested_sequences_map.groupby('REF_ID')['ALT_ID'].count())
count_tbl[count_tbl['ALT_ID'] == 41]

Number of rows in variant region map:  47044
Number of rows in variant region map of cardiac_neuro_cava_random:  46374
Number of rows in variant region map of controls:  670


,ALT_ID
REF_ID,
cardiac_neuro_cava_random:REF_H1-3|ENSG00000124575.7|EH38E3697654_rev_tile1-1,41


In [19]:
variant_region_map.head()

,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...


#### Add seq_chrom, seq_strand, seq_start and seq_end to the table

In [20]:
# add chrom, seq_start, seq_end for tested sequences with bed file (/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/input/regions_5K.bed)

# load bed file
bed_file_test_seqs = pd.read_csv(config['files']['final_design']['test_seqs_region_file'], sep='\t', header=None)
bed_file_test_seqs.columns = ['seq_chr', 'seq_start', 'seq_end', 'tmp_bed_name', 'tmp_score', 'seq_strand']
data_types = {'seq_chr': str, 'seq_start': str, 'seq_end': str, 'tmp_bed_name': str, 'tmp_score': str, 'seq_strand': str}
bed_file_test_seqs = bed_file_test_seqs.astype(data_types) # seq_start + seq_end not int or float

# generate correct name for the pre_metadata_df (tmp_bed_name) (TODO: change functionality of get_region_match_name (error in matching * => >))
pre_metadata_df['tmp_bed_name'] = pre_metadata_df['header'].apply(get_region_match_name)

# all rows with cardiac_neuro_cava_random: 73940 all rows: 80215

# adding this information to pre_metadata_df (problem: wrong number of rows => 80215 (correct) vs 83751 (wrong)) 
pre_metadata_df = pre_metadata_df.merge(bed_file_test_seqs, on='tmp_bed_name', how='left')
# checking_chromposrefalt.fillna('NA', inplace=True)

pre_metadata_df['seq_chr'].isna().sum() # 49123 # 6275

# checking_chromposrefalt[checking_chromposrefalt['seq_chr'].isna()].tmp_label.value_counts() # 6275 


6275

##### Add for controls:

In [21]:
pre_metadata_df[pre_metadata_df['seq_chr'].isna()]['tmp_label'].value_counts()

tmp_label
MK                                   2397
C_positive_heart_AB                   909
GC_Selvarajan                         364
GC_Vista                              256
C_negative_heart_MK                   243
C_negative_neuron_MK                  222
C_negative_neuron_NP                  217
GC_Mendelian_variants                 209
GC_Kircher                            203
C_SLEA                                200
GC_Cort_Chengyu                       185
C_positive_neuron_NP                   99
C_positive_heart_CAD                   97
C_positive_heart_MK                    97
C_positive_neuron_MK                   96
C_positive_neuron_CD                   94
GC_GABA_Chengyu                        85
GC_DNase_positive_shuffeled            55
GC_Atrial_fib                          45
GC_DNase_positive                      41
GC_Glut_Chengyu                        40
GC_Mohlke                              34
GC_DNase_negative_blood_shuffeled      19
GC_Liang                

In [22]:
pre_metadata_df[pre_metadata_df[col_name] == 'cardiac_neuro_cava_random:ALT_RIT1|ENSG00000143622.12|EH38E1387405_rev_tile1-1_RIT1|ENSG00000143622.12|EH38E1387405|1-155892969-T-C']

,header,sequence,tmp_label,name,category,class,source,variant_class,variant_pos,SPDI,allele,info,ref_seq,tmp_bed_name,seq_chr,seq_start,seq_end,tmp_score,seq_strand
31092,cardiac_neuro_cava_random:ALT_RIT1|ENSG0000014...,AGGACCGGATCAACTGATTTGGATTGAATGGTGGGAGAATGAGAAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_RIT1|ENSG0000014...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,RIT1|ENSG00000143622.12|EH38E1387405,chr1,155892803,155893150,.,-


In [23]:
pre_metadata_df[pre_metadata_df[col_class] != 'test']

,header,sequence,tmp_label,name,category,class,source,variant_class,variant_pos,SPDI,allele,info,ref_seq,tmp_bed_name,seq_chr,seq_start,seq_end,tmp_score,seq_strand
73940,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,AGGACCGGATCAACTTCATTTCATTATAATCAAAAAGGATTTTTAA...,GC_Atrial_fib,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,element,element inactive control,IGVF general controls (GC_Atrial_fib),NA,NA,NA,NA,GC_Atrial_fib,GRCh38,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,NaN,NaN,NaN,NaN,NaN
73941,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,GC_Atrial_fib,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
73942,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,GC_Atrial_fib,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
73943,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,GC_Atrial_fib,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
73944,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,GC_Atrial_fib,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),SNP,NA,NA,ref,GC_Atrial_fib,GRCh38,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK,MK:tile_2240|chr1-116244322+116244591|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_2240|chr1-116244322+116244591|scramble...,NaN,NaN,NaN,NaN,NaN
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,NaN,NaN,NaN,NaN,NaN
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK,MK:tile_18415|chr17-71181691+71181960|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_18415|chr17-71181691+71181960|scramble...,NaN,NaN,NaN,NaN,NaN
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK,MK:tile_14356|chr15-67031618+67031887|scramble...,scrambled,element inactive control,NA,NA,NA,NA,NA,MK,GRCh38,MK:tile_14356|chr15-67031618+67031887|scramble...,NaN,NaN,NaN,NaN,NaN


In [24]:
# get the name and header for the metadata file
name_header_match = pre_metadata_df[[col_name, 'header']]

name_header_match_path = config['files']['creating']['name_header_match'] 
# write the match table to a tsv
# create output path
create_path(name_header_match_path)
name_header_match.to_csv(name_header_match_path, sep='\t', index=False)

# order the columns as in the metadata file
pre_metadata_df_final = pre_metadata_df[[col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]]

pre_metadata_df_final

# # write the metadata file
metadata_path = config['files']['creating']['metadata_table']
# create_path(metadata_path)
# metadata_path = 'metadata_example_2703.tsv'
pre_metadata_df_final.to_csv(metadata_path, sep='\t', index=False)
# pre_metadata_df.to_csv(metadata_path, sep='\t', index=False)

# in the end check NA distribution/number of each column 


KeyboardInterrupt: 

In [ ]:
pre_metadata_df_final

,name,sequence,category,class,source,ref_seq,seq_chr,seq_start,seq_end,seq_strand,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2181818,2182138,+,NA,NA,NA,NA,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182410,2182738,+,NA,NA,NA,NA,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182832,2183099,+,NA,NA,NA,NA,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2184994,2185331,+,NA,NA,NA,NA,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2188356,2188693,+,NA,NA,NA,NA,cardiac_neuro_cava_random
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,scrambled,element inactive control,NA,GRCh38,NaN,NaN,NaN,NaN,NA,NA,NA,NA,MK


### Sanity check seq_start, seq_end
- get the length of the regions and compare

In [ ]:
sub_df = pre_metadata_df[pre_metadata_df['tmp_label'] == 'cardiac_neuro_cava_random']
sub_df.loc[:,'tmp_seq_length'] = sub_df['seq_end'].astype(int) - sub_df['seq_start'].astype(int)
sub_df['tmp_seq_length'].value_counts()
sub_df

/tmp/ipykernel_1887/3271522864.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df.loc[:,'tmp_seq_length'] = sub_df['seq_end'].astype(int) - sub_df['seq_start'].astype(int)


,header,sequence,tmp_label,name,category,class,source,variant_class,variant_pos,SPDI,allele,info,ref_seq,tmp_bed_name,seq_chr,seq_start,seq_end,tmp_score,seq_strand,tmp_seq_length
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778476,chr1,2181818,2182138,.,+,320
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778477,chr1,2182410,2182738,.,+,328
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778478,chr1,2182832,2183099,.,+,267
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778480,chr1,2184994,2185331,.,+,337
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",NA,NA,NA,NA,cardiac_neuro_cava_random,GRCh38,SKI|ENSG00000157933.11|EH38E2778484,chr1,2188356,2188693,.,+,337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E3949733,chrX,154544972,154545298,.,-,326
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E2774396,chrX,154549862,154550013,.,-,151
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,.,-,325
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",SNP,NA,NA,alt,cardiac_neuro_cava_random,GRCh38,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,.,-,325


#### Investigates similarity of sequences

In [ ]:
AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCC
AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCC

In [ ]:
AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCCCATTGCGTGAACCGA

In [ ]:
def get_seq_length(row):
    return len(row[col_sequence])

sub_df['tmp_seq_length'] = sub_df.apply(get_seq_length, axis=1)
sub_df['tmp_seq_without_adapter'] = sub_df[col_sequence].str.slice(15, 285)

# exclude ALT_ sequences
sub_df = sub_df[~sub_df['header'].str.contains('ALT_')]
sub_df.shape # 27482

# revert sequences on - strand

# # write fasta of tmp_seq_without_adapter and name as header 
# file_path = 'tested_ref_and_elements.fasta'

# for index, row in sub_df.iterrows():
#     with open(file_path, 'a') as f:
#         f.write('>' + row[col_name] + '\n' + row['tmp_seq_without_adapter'] + '\n')





# sub_df['tmp_seq_length'].value_counts()

/tmp/ipykernel_1887/3550457708.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['tmp_seq_length'] = sub_df.apply(get_seq_length, axis=1)
/tmp/ipykernel_1887/3550457708.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['tmp_seq_without_adapter'] = sub_df['sequence'].str.slice(15, 285)


(27482, 21)

In [ ]:
# check strand information of the sequences: 
sub_df['seq_strand'].value_counts() # all are "+"

seq_strand
+    14306
-    13176
Name: count, dtype: int64

#### Investigate blat results: 
<!-- - expect 27482 hits with 100% identity and 270 matching bases -->
- for 27120 we have exactly one match with 270 matching bases
- interest in T_name (=> chromosome), T_start and T_end, strand

In [ ]:
import pandas as pd

In [ ]:
# bash commands to generate a tsv out of the psl file
file = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_result_without_header.tsv'
# remove header
# tail -n +6 blat_matched_file.psl > blat_result_without_header.tsv

blat_result_df = pd.read_csv(file, sep='\t', header=None)

# add column names
blat_result_columns = ['match', 'mismatch', 'rep_match', 'Ns', 'Q_gap_count', 'Q_gap_bases', 'T_gap_count', 'T_gap_bases', 'strand', 'Q_name', 'Q_size', 'Q_start', 'Q_end', 'T_name', 'T_size', 'T_start', 'T_end', 'block_count', 'blockSizes', 'qStarts', 'tStarts']

blat_result_df.columns = blat_result_columns

In [ ]:
blat_result_df
# filter: number of matches=270, 
perfect_matches = blat_result_df[blat_result_df['match'] == 270]
perfect_matches.shape # 28276
# filter for blockSizes = 270
perfect_matches = perfect_matches[perfect_matches['blockSizes'] == '270,']
perfect_matches.shape # 28236
# check all columns for unique values and did not find any suspecious values
perfect_matches.T_name.value_counts() # 0 
# filter for matches startwith "NC_" in T_name
perfect_matches = perfect_matches[perfect_matches['T_name'].str.startswith('NC_')]
perfect_matches.shape # 27482
perfect_matches.Q_name.nunique() # 27482

# get subset of interesting columns
interesting_blat_results = ['Q_name', 'match', 'strand', 'T_name', 'T_start', 'T_end']
interesting_blat_results

perfect_matches = perfect_matches[interesting_blat_results]
perfect_matches

,Q_name,match,strand,T_name,T_start,T_end
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2181843,2182113
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182439,2182709
64,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182830,2183100
122,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2185027,2185297
124,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2188389,2188659
...,...,...,...,...,...,...
369505,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154531979,154532249
369506,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154539055,154539325
369507,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154545000,154545270
369562,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154549802,154550072


In [ ]:
perfect_matches

,Q_name,match,strand,T_name,T_start,T_end
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2181843,2182113
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182439,2182709
64,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2182830,2183100
122,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2185027,2185297
124,cardiac_neuro_cava_random:SKI|ENSG00000157933....,270,+,NC_000001.11,2188389,2188659
...,...,...,...,...,...,...
369505,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154531979,154532249
369506,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154539055,154539325
369507,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154545000,154545270
369562,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,270,-,NC_000023.11,154549802,154550072


In [ ]:
perfect_matches.T_name.value_counts()

T_name
NC_000001.11    3119
NC_000003.12    1899
NC_000007.14    1816
NC_000002.12    1613
NC_000011.10    1504
NC_000010.11    1448
NC_000016.10    1389
NC_000006.12    1361
NC_000017.11    1331
NC_000009.12    1285
NC_000023.11    1189
NC_000015.10    1162
NC_000012.12    1049
NC_000005.10    1000
NC_000014.9      985
NC_000004.12     923
NC_000018.10     860
NC_000019.10     844
NC_000020.11     804
NC_000008.11     733
NC_000022.11     691
NC_000021.9      286
NC_000013.11     191
Name: count, dtype: int64

##### Add to the table
- check if all + have a + in the new search results
- check if the new positions are close to the old ones
  - same chromosome
  - start - start + end - end 

In [ ]:
def set_modified_chromosome(row):
    """i.e. from NC_000001.11 to chr1, ..."""
    if '23' in row['T_name']:
        return 'chrX'
    if '24' in row['T_name']:
        return 'chrY'
    chr_number = int(row['T_name'].split('_')[1].split('.')[0])
    return 'chr%s'%(chr_number) 

In [ ]:
mapped_blat_results = pre_metadata_df_final.merge(perfect_matches, left_on=col_name, right_on='Q_name', how='left')

mapped_blat_results.shape[0] - mapped_blat_results['Q_name'].isna().sum()

suc_mapped_blat_results = mapped_blat_results[~mapped_blat_results['Q_name'].isna()]

# check quality of new results: 
# 1. strand same? 
suc_mapped_blat_results['tmp_same_strand_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_strand'] == x['strand'] else False, axis=1)
if suc_mapped_blat_results['tmp_same_strand_info'].sum() == suc_mapped_blat_results.shape[0]: # passed the test
    print('Passed test: Strand information is the same')

# 2. Same chromosome?
# convert NC_* chromosomes to chr*
suc_mapped_blat_results['tmp_modified_chr'] = suc_mapped_blat_results.apply(set_modified_chromosome, axis=1)
suc_mapped_blat_results['tmp_same_chr_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_chr'] == x['tmp_modified_chr'] else False, axis=1)
if suc_mapped_blat_results['tmp_same_chr_info'].sum() == suc_mapped_blat_results.shape[0]: # passed the test
    print('All results have the same chromosome information')

# Check if positions are nearby 
suc_mapped_blat_results['T_start'] = suc_mapped_blat_results['T_start'].astype(int)
suc_mapped_blat_results['T_end'] = suc_mapped_blat_results['T_end'].astype(int)
suc_mapped_blat_results['seq_start'] = suc_mapped_blat_results['seq_start'].astype(int)
suc_mapped_blat_results['seq_end'] = suc_mapped_blat_results['seq_end'].astype(int)
suc_mapped_blat_results['tmp_start_difference'] = suc_mapped_blat_results.apply(lambda x: abs(x['T_start'] - x['seq_start']), axis=1)
suc_mapped_blat_results['tmp_start_difference'].value_counts()


# print suc_mapped_blat_results value counts as dataframe 
pd.DataFrame(suc_mapped_blat_results['tmp_start_difference'].value_counts())



# suc_mapped_blat_results['tmp_end_difference'] = suc_mapped_blat_results.apply(lambda x: abs(x['T_end'] - x['seq_end']), axis=1)
# start positions have difference from over 1kb this is sus 
# 39    1955
# 38    1508
# 35    1380
# 40    1364
# 37    1231

/tmp/ipykernel_1887/2245177199.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_same_strand_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_strand'] == x['strand'] else False, axis=1)
/tmp/ipykernel_1887/2245177199.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_modified_chr'] = suc_mapped_blat_results.apply(set_modified_chromosome, axis=1)


Passed test: Strand information is the same
All results have the same chromosome information


/tmp/ipykernel_1887/2245177199.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_same_chr_info'] = suc_mapped_blat_results.apply(lambda x: True if x['seq_chr'] == x['tmp_modified_chr'] else False, axis=1)
/tmp/ipykernel_1887/2245177199.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['T_start'] = suc_mapped_blat_results['T_start'].astype(int)
/tmp/ipykernel_1887/2245177199.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s

,count
tmp_start_difference,
39,1955
38,1508
35,1380
40,1364
37,1231
...,...
47,152
48,152
59,150


In [81]:
# write seq_chr, seq_start, seq_end, name, ., seq_strand
suc_mapped_blat_results['score'] = '.'
region_a_table = suc_mapped_blat_results[['seq_chr', 'seq_start', 'seq_end', 'name', 'score', 'seq_strand']]
# # write tmp_modified_chr, T_start, T_end, Q_name, ., strand
region_b_table = suc_mapped_blat_results[['tmp_modified_chr', 'T_start', 'T_end', 'Q_name', 'score', 'strand']]

# logic: check if seq_start < T_start and T_end < seq_end
# if not: check if seq_start < T_start and T_end > seq_end
# print the name and count the number of cases which are printed at the end 
def is_inbetween(row):
    """
    Return True if seq_start < T_start and T_end < seq_end
    Can only be done like that because all are on the same strand (?)
    """
    return row['seq_start'] < row['T_start'] and row['T_end'] < row['seq_end']

suc_mapped_blat_results['tmp_is_inbetween'] = suc_mapped_blat_results.apply(is_inbetween, axis=1)
pd.DataFrame(suc_mapped_blat_results['tmp_is_inbetween'].value_counts())
# 11538 are not inbetween 



/tmp/ipykernel_1887/3931396714.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['score'] = '.'
/tmp/ipykernel_1887/3931396714.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_is_inbetween'] = suc_mapped_blat_results.apply(is_inbetween, axis=1)


,count
tmp_is_inbetween,
True,15944
False,11538


##### Investigate the ucsc genome browser of these regions and find the cCREs


In [84]:
suc_mapped_blat_results['tmp_region5k_length'] = suc_mapped_blat_results['seq_end'] - suc_mapped_blat_results['seq_start']

## not inbetween
suc_mapped_blat_results[suc_mapped_blat_results['tmp_is_inbetween'] == False]['tmp_region5k_length'].max()

# ### only smaller length:
suc_mapped_blat_results[suc_mapped_blat_results['tmp_region5k_length'] <= 271].shape[0] # 11336 
suc_mapped_blat_results[suc_mapped_blat_results['tmp_region5k_length'] <= 271]['name'].to_list()
# example1: cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778609_fwd_tile1-1
# example2: cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779779_fwd_tile1-1
# example3: cardiac_neuro_cava_random:FGF12|ENSG00000114279.15|EH38E2269809_rev_tile1-1
suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778609_fwd_tile1-1']
# # on the last 15bp on the right a distal enhancer like sequence could be found: 
# # investigated with ucsc genome browser: https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A2250503%2D2250773&hgsid=2082689898_VAczbUehrymUlkaWiiYiCorrQZsL
suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779779_fwd_tile1-1']
# # Didn't find regulatory element at this position
# # https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A3210194%2D3210464&hgsid=2082689898_VAczbUehrymUlkaWiiYiCorrQZsL 

# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:FGF12|ENSG00000114279.15|EH38E2269809_rev_tile1-1']
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:FGF12|ENSG00000114279.15|EH38E2269809_rev_tile1-1'][col_sequence].values[0][15:]
# - strand hit: go to right side end and use complement bases: blat coordinates: NC_000003.12	192748482	192748752
# # found ELS from encode (matching accessions: EH38E2269809): chr3:192,748,528-192,748,708 => we are out of bounds at the right side
# # https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr3%3A192748528%2D192748752&hgsid=2082689898_VAczbUehrymUlkaWiiYiCorrQZsL

# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778717_fwd_tile1-1']
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778717_fwd_tile1-1'][col_sequence].values[0][15:]
# # + strand hit: found cCRE from encode accession:EH38E1311783: chr1:2309913-2310122 our position (region5K): chr1	2309925	2310126 blat: NC_000001.11	2309890	2310160
# # not matching accession: our: EH38E2778717 found: EH38E1311783
# # https://genome.ucsc.edu/cgi-bin/hgc?hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq&db=hg38&c=chr1&l=2309924&r=2310126&o=2309912&t=2310122&g=encodeCcreCombined&i=EH38E1311783

# # example chr X: cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1']
# suc_mapped_blat_results[suc_mapped_blat_results['name'] == 'cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1'][col_sequence].values[0][15:]
# # found CRE from encode but it is larger than the region and encode accession does not match (chrX:154515117-154515465; EH38E2774357 (https://genome.ucsc.edu/cgi-bin/hgc?hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq&db=hg38&c=chrX&l=154515132&r=154515183&o=154515116&t=154515465&g=encodeCcreCombined&i=EH38E2774357))
# # region5K: https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chrX%3A154515118%2D154515418&hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq
# # blat region of sequence (NC_000023.11	154515133	154515403): https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chrX%3A154515133%2D154515403&hgsid=2080448280_bDXQ4noYZFZPU5Nj0a4cUrr9C1wq


/tmp/ipykernel_1887/655454006.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  suc_mapped_blat_results['tmp_region5k_length'] = suc_mapped_blat_results['seq_end'] - suc_mapped_blat_results['seq_start']


,name,sequence,category,class,source,ref_seq,seq_chr,seq_start,seq_end,seq_strand,...,T_name,T_start,T_end,tmp_same_strand_info,tmp_modified_chr,tmp_same_chr_info,tmp_start_difference,score,tmp_is_inbetween,tmp_region5k_length
52,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,AGGACCGGATCAACTAGCCAGCCTGCACTCACAGGCAGGTGGGCTC...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,3210229,3210430,+,...,NC_000001.11,3210194,3210464,True,chr1,True,35,.,False,201


In [62]:
suc_mapped_blat_results[suc_mapped_blat_results['seq_start'] == 154515118]['name'].to_list()
# suc_mapped_blat_results[suc_mapped_blat_results['seq_start'] == 154515118]['name'].to_list()

['cardiac_neuro_cava_random:REF_G6PD|ENSG00000160211.20|EH38E3949700_rev_tile1-1']

In [46]:
suc_mapped_blat_results[suc_mapped_blat_results['tmp_start_difference'] == 39].iloc[23,:]
## sanity check: 
# - check inbetween
# suc_mapped_blat
# - check for 2 regions

name                    cardiac_neuro_cava_random:RERE|ENSG00000142599...
sequence                AGGACCGGATCAACTGATTAAGCTCCTATATGATATATGTAACAAA...
category                                                          element
class                                                                test
source                  candidate CRE nearby cardiac, neuro, cava and ...
ref_seq                                                            GRCh38
seq_chr                                                              chr1
seq_start                                                         8733505
seq_end                                                           8733853
seq_strand                                                              -
variant_class                                                          NA
variant_pos                                                            NA
SPDI                                                                   NA
allele                                

In [69]:
perfect_matches

,match,mismatch,rep_match,Ns,Q_gap_count,Q_gap_bases,T_gap_count,T_gap_bases,strand,Q_name,...,Q_start,Q_end,T_name,T_size,T_start,T_end,block_count,blockSizes,qStarts,tStarts
0,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2181843,2182113,1,"270,","0,","2181843,"
1,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2182439,2182709,1,"270,","0,","2182439,"
64,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2182830,2183100,1,"270,","0,","2182830,"
122,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2185027,2185297,1,"270,","0,","2185027,"
124,270,0,0,0,0,0,0,0,+,cardiac_neuro_cava_random:SKI|ENSG00000157933....,...,0,270,NC_000001.11,248956422,2188389,2188659,1,"270,","0,","2188389,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369505,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154531979,154532249,1,"270,","0,","154531979,"
369506,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154539055,154539325,1,"270,","0,","154539055,"
369507,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154545000,154545270,1,"270,","0,","154545000,"
369562,270,0,0,0,0,0,0,0,-,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,...,0,270,NC_000023.11,156040895,154549802,154550072,1,"270,","0,","154549802,"


In [47]:
perfect_matches.groupby('Q_name').size().value_counts()
# group by Q_name
# 1     27120
# 2       221
# 3        83
# 10       32
# 4        17
# 9         8
# 5         1
perfect_matches.groupby(['Q_name', 'strand']).size().value_counts()
# groupby 'Q_name', 'strand'
# 1     27147
# 2       246
# 3        67
# 10       32
# 4        11
# 9         8

# which sequences have multiple matches
multiple_matches = perfect_matches.groupby('Q_name').size()
multiple_matches = multiple_matches[multiple_matches > 1]

In [48]:
multiple_matches

Q_name
cardiac_neuro_cava_random:AP3B2|ENSG00000103723.17|EH38E3150740_rev_tile1-1     2
cardiac_neuro_cava_random:AP3B2|ENSG00000103723.17|EH38E3150741_rev_tile1-1     2
cardiac_neuro_cava_random:ASH1L|ENSG00000116539.14|EH38E2839971_rev_tile1-1     2
cardiac_neuro_cava_random:CNOT3|ENSG00000088038.20|EH38E1963947_fwd_tile1-1    10
cardiac_neuro_cava_random:CNOT3|ENSG00000088038.20|EH38E3316435_fwd_tile1-1    10
                                                                               ..
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484239_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484241_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484245_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484247_rev_tile1-1     3
cardiac_neuro_cava_random:TCF20|ENSG00000100207.21|EH38E3484250_rev_tile1-1     3
Length: 362, dtype: int64

In [104]:
sub_df['tmp_seq_without_adapter'].to_list()

['...',
 ...]

In [ ]:
# download hg38 reference genome

In [107]:
! curl -L 'https://api.genome.ucsc.edu/search?search=AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCACAATCCTGCTGTGCGCACACCACAGGTGAGCCGTTCTGCCTAAGGGAACAGCCCCGGCCCCTCCCTCCGGCTCCTCCCCAGCACCGTCTCCTCCACCCAGTGGCCTGGCCGTGGATGCTGCCTGTGGCCCAGCTTTGAGACACCGCCCTGACACGTGTCCAGCCTTACGTGGAAGGATTTGTCTGTTTTGTGGCATCCTAGTAGATGCCACGTTAGTAGATGCC&genome=hg38'


{ "downloadTime": "2024:03:27T19:22:03Z", "downloadTimeStamp": 1711567323, "genome": "hg38"} 

#### Underscore counts in the headers: We want to combine these headers with the header in the bed file [here](/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/input/regions_5K.bed)
- still having problems with the matching
- n_underscore
  - 7:    46458
  - 6:    18582

In [30]:
variant_list_test_seqs = pre_metadata_df_final[pre_metadata_df_final[col_category] == 'variant'][col_name].to_list() # e.g. 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'

def get_region_match_name(header):
    """Get the region match name from the header
    1. split by ':' and take the second part [1]
    2. split by '_' and take the second part [1]
    """
    if header.split(':')[0] == 'cardiac_neuro_cava_random':
        if 'ALT_' in header or 'REF_' in header:
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
                header = header.replace('~', ',')
            # easy types: 
            # cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            # cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'
            return ':'.join(header.split(':')[1:]).split('_')[1]
        else: # cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778480_fwd_tile1-1
            if '~' in header:
                # replace ~ to ,: cardiac_neuro_cava_random:APOL2|ENSG00000128335.14|EH38E3478577~APOL4|ENSG00000100336.18|EH38E3478577_rev_tile1-1
                header = header.replace('~', ',')
            return ':'.join(header.split(':')[1:]).split('_')[0]
    else:
        return header

def get_variant_position(header):
    """Check the vcf file, prepare ID and merge with the tested sequences"""
    # /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/input/variants_5K.vcf
    # G6PD|ENSG00000160211.20|EH38E3949659|X-154478369-G-A

# investigate different groups of headers depending on the nuumber of "_" used. So count the number of "_" in name
tested_variants = pre_metadata_df_final[pre_metadata_df_final[col_name].str.split(':').str[0] == 'cardiac_neuro_cava_random']
# tested_variants = tested_variants[tested_variants[col_category] == 'variant']
tested_variants['n_underscore'] = tested_variants[col_name].str.count('_')
tested_variants['n_underscore'].value_counts()

# # show one case for each number of underscores
# for i in [6,7]:
#     print(i)
#     print(tested_variants[tested_variants['n_underscore'] == i][col_name].to_list())



# ## now make from all cardiac_neuro_cava_random headers the following mutations and call it bed_match_name
tested_variants['region_match_name'] = tested_variants[col_name].apply(get_region_match_name)
tested_variants[tested_variants[col_name] == 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1']

# 3. test how many sequences can be matched with the bed file
bed_file_test_seqs = pd.read_csv(config['files']['final_design']['test_seqs_region_file'], sep='\t', header=None)
bed_file_test_seqs.columns = ['tmp_chrom', 'tmp_chromStart', 'tmp_chromEnd', 'tmp_name', 'tmp_score', 'tmp_strand']
bed_file_test_seqs

# left join tested_variants and bed_file both files and count number of NAs (not found in bed file)
checking_chromposrefalt = tested_variants.merge(bed_file_test_seqs, left_on='region_match_name', right_on='tmp_name', how='left')
checking_chromposrefalt # left: 73940 outer: 86376
# checking_chromposrefalt['tmp_chrom'].isna().sum() # 8900 => all elements are not in the bed file?

# # # checking sequences with NA values (alle haben so eine ~)
# checking_chromposrefalt[checking_chromposrefalt['tmp_chrom'].isna()][col_name].to_list()

# replaced ~ with , and now no NA in tested sequences matching with the bed file
# next step: add this step earlier to the generation script

/tmp/ipykernel_45845/2932931991.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_variants['n_underscore'] = tested_variants['name'].str.count('_')
/tmp/ipykernel_45845/2932931991.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_variants['region_match_name'] = tested_variants['name'].apply(get_region_match_name)


,name,sequence,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,...,allele,info,n_underscore,region_match_name,tmp_chrom,tmp_chromStart,tmp_chromEnd,tmp_name,tmp_score,tmp_strand
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778476,chr1,2181818,2182138,SKI|ENSG00000157933.11|EH38E2778476,.,+
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778477,chr1,2182410,2182738,SKI|ENSG00000157933.11|EH38E2778477,.,+
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778478,chr1,2182832,2183099,SKI|ENSG00000157933.11|EH38E2778478,.,+
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778480,chr1,2184994,2185331,SKI|ENSG00000157933.11|EH38E2778480,.,+
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,...,NA,cardiac_neuro_cava_random,5,SKI|ENSG00000157933.11|EH38E2778484,chr1,2188356,2188693,SKI|ENSG00000157933.11|EH38E2778484,.,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E3949733,chrX,154544972,154545298,G6PD|ENSG00000160211.20|EH38E3949733,.,-
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E2774396,chrX,154549862,154550013,G6PD|ENSG00000160211.20|EH38E2774396,.,-
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,G6PD|ENSG00000160211.20|EH38E3949745,.,-
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,SNP,...,alt,cardiac_neuro_cava_random,7,G6PD|ENSG00000160211.20|EH38E3949745,chrX,154552118,154552443,G6PD|ENSG00000160211.20|EH38E3949745,.,-


In [19]:
tested_variants[tested_variants[col_category] == 'variant']['region_match_name'].to_list()

['SKI|ENSG00000157933.11|EH38E2778471',
 'SKI|ENSG00000157933.11|EH38E2778490',
 'SKI|ENSG00000157933.11|EH38E2778492',
 'SKI|ENSG00000157933.11|EH38E1311587',
 'SKI|ENSG00000157933.11|EH38E2778494',
 'SKI|ENSG00000157933.11|EH38E2778498',
 'SKI|ENSG00000157933.11|EH38E2778506',
 'SKI|ENSG00000157933.11|EH38E2778513',
 'SKI|ENSG00000157933.11|EH38E2778524',
 'SKI|ENSG00000157933.11|EH38E2778530',
 'SKI|ENSG00000157933.11|EH38E2778534',
 'SKI|ENSG00000157933.11|EH38E2778535',
 'SKI|ENSG00000157933.11|EH38E2778544',
 'SKI|ENSG00000157933.11|EH38E2778546',
 'SKI|ENSG00000157933.11|EH38E2778564',
 'SKI|ENSG00000157933.11|EH38E2778568',
 'SKI|ENSG00000157933.11|EH38E2778574',
 'SKI|ENSG00000157933.11|EH38E2778579',
 'SKI|ENSG00000157933.11|EH38E2778583',
 'SKI|ENSG00000157933.11|EH38E2778585',
 'SKI|ENSG00000157933.11|EH38E2778586',
 'SKI|ENSG00000157933.11|EH38E2778588',
 'SKI|ENSG00000157933.11|EH38E2778590',
 'SKI|ENSG00000157933.11|EH38E2778592',
 'SKI|ENSG00000157933.11|EH38E1311678',


In [11]:
variant_list_test_seqs = pre_metadata_df_final[pre_metadata_df_final[col_category] == 'variant'][col_name].to_list() # e.g. 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1'

def get_region_match_name(header):
    """Get the region match name from the header
    1. split by ':' and take the second part [1]
    2. split by '_' and take the second part [1]
    """
    if header.str.('cardiac_neuro_cava_random'):
        return header.split(':')[1].split('_')[1]
    else:
        return header

# investigate different groups of headers depending on the nuumber of "_" used. So count the number of "_" in name
tested_variants = pre_metadata_df_final[pre_metadata_df_final[col_name].str.split(':').str[0] == 'cardiac_neuro_cava_random']
tested_variants = tested_variants[tested_variants[col_category] == 'variant']
tested_variants['n_underscore'] = tested_variants[col_name].str.count('_')
tested_variants['n_underscore'].value_counts()

# show one case for each number of underscores
for i in [6,7]:
    print(i)
    print(tested_variants[tested_variants['n_underscore'] == i].head(2))



# ## now make from all cardiac_neuro_cava_random headers the following mutations and call it bed_match_name
# # 1. split by ':' and take the second part [1]
# # 2. split by '_' and take the second part [1]
# pre_metadata_df_final[col_name].apply(get_region_match_name)

# 3. test how many sequences can be matched with the bed file

SyntaxError: invalid syntax (2013555655.py, line 8)

In [ ]:
variant_list_test_seqs

#### Investigating controls: 
- check the blat results for the control variants
  - `/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26`
  - `blat_variant_control_sequences_matched_file.tsv`
- compare with the region files of the controls you have

In [ ]:
# read blat results in 

# for which results do I have a perfect match?

In [6]:
## old code: 
# get positive controls
positive_controls = pre_metadata_df_final[pre_metadata_df_final[col_info].isin(positive_control_groups)]
positive_controls.head()


def is_control(row):
    """True if control false if not"""
    return row[col_class] != 'test'

def get_control_variant(row):
    """Get the control variant"""
    if row[col_class] == 'variant positive control':
        return True
    elif row[col_class] == 'variant negative control':
        return True
    else:
        return False
    
def is_dnase_control(row):
    """True if control is a dnase control"""
    return 'dnase' in row[col_info].lower()


# get number of controls with variants
control_vars = pre_metadata_df_final[pre_metadata_df_final.apply(is_control, axis=1)] # 1058 
# control_vars[col_info].nunique() # 8 controls with variants # but CD is wrong
control_vars.head()

In [8]:
control_vars[col_info].unique()

array(['GC_Atrial_fib', 'GC_Liang', 'GC_Selvarajan', 'GC_Mohlke',
       'GC_Kircher', 'GC_Mendelian_variants', 'C_positive_heart_CAD',
       'GC_Cort_Chengyu', 'GC_GABA_Chengyu', 'GC_Glut_Chengyu', 'GC_Hon',
       'GC_Vista',
       'GC_DNase_positive;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_brain;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_blood;wrong genome build (GRCh37) used for these sequences',
       'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'C_positive_heart_AB', 'C_SLEA',
       'GC_DNase_positive_shuffeled;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_brain_shuffeled;wrong genome build (GRCh37) used for these sequences',
       'GC_DNase_negative_blood_shuffeled;wrong genome build (GRCh37) used for these sequences',
    

In [ ]:
# GC_Atrial_fib', 'GC_Liang', 'GC_Selvarajan', 'GC_Mohlke',
#        'GC_Kircher', 
#        'GC_Cort_Chengyu', 
#        'GC_GABA_Chengyu', 'GC_Glut_Chengyu', 'GC_Hon',
#        'GC_Mendelian_variants', 'C_positive_heart_CAD',
#        'GC_Vista',
#        'GC_DNase_positive;wrong genome build (GRCh37) used for these sequences',
#        'GC_DNase_negative_brain;wrong genome build (GRCh37) used for these sequences',
#        'GC_DNase_negative_blood;wrong genome build (GRCh37) used for these sequences',
#        'MK'
       
#        'C_negative_heart_MK', 'C_negative_neuron_MK',
#        'C_negative_neuron_NP', 'C_positive_heart_MK',
#        'C_positive_neuron_CD', 'C_positive_neuron_MK',
#        'C_positive_neuron_NP', 'C_positive_heart_AB', 'C_SLEA',
#        'GC_DNase_positive_shuffeled;wrong genome build (GRCh37) used for these sequences',
#        'GC_DNase_negative_brain_shuffeled;wrong genome build (GRCh37) used for these sequences',
#        'GC_DNase_negative_blood_shuffeled;wrong genome build (GRCh37) used for these sequences',


In [20]:
pre_metadata_df_final.apply(is_dnase_control, axis=1).sum()

161

In [16]:
6275 - 289

5986

In [7]:
pre_metadata_df_final.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref_sequence',
       'chrom', 'chrom_start', 'chrom_end', 'variant_class', 'variant_pos',
       'SPDI', 'allele', 'info'],
      dtype='object')

### Make REF ALT map (variant map) for controls
- Find all REF and ALT headers and match them for each group (because of not matching patterns)
- the following control groups have variants (positive or negative) (ref_ or alt_ in header.lower())
  - C_positive_heart_CAD
  - C_positive_neuron_CD
  - GC_Atrial_fib
  - GC_Kircher
  - GC_Liang
  - GC_Mendelian_variants
  - GC_Mohlke
  - GC_Selvarajan

In [5]:
old_metadata = config['files']['creating']['old_metadata_table']
pre_metadata_df_final = pd.read_csv(old_metadata, sep="\t")


In [6]:
variant_control_groups = ['C_positive_heart_CAD', 'C_positive_neuron_CD', 'GC_Atrial_fib', 'GC_Kircher', 'GC_Liang', 'GC_Mendelian_variants', 'GC_Mohlke', 'GC_Selvarajan']
# get all entries of pre_metadata_df_final with the labels in the info column with variant in the name
variant_control_seqs = pre_metadata_df_final[pre_metadata_df_final[col_info].isin(variant_control_groups)]
variant_control_vars = variant_control_seqs[variant_control_seqs[col_category] == 'variant']
variant_control_vars # 1058 

variant_control_vars[col_class].value_counts()

class
variant negative control    967
variant positive control     91
Name: count, dtype: int64

In [7]:
variant_control_vars[col_info].value_counts()

info
GC_Selvarajan            364
GC_Mendelian_variants    209
GC_Kircher               203
C_positive_heart_CAD      97
C_positive_neuron_CD      91
GC_Atrial_fib             44
GC_Mohlke                 34
GC_Liang                  16
Name: count, dtype: int64

In [ ]:
# info                   class                     number of variants   after assignment (with elements)
# C_positive_heart_CAD   variant negative control     97    92
# C_positive_neuron_CD   variant positive control     91    88
# GC_Atrial_fib          variant negative control     44    45
# GC_Kircher             variant negative control    203    185
# GC_Liang               variant negative control     16    16
# GC_Mendelian_variants  variant negative control    209    185
# GC_Mohlke              variant negative control     34    31
# GC_Selvarajan          variant negative control    364    356

#### C_positive_heart_CAD (97)
- REF: C_positive_heart_CAD:REF_rs17114036 
- ALT: C_positive_heart_CAD:ALT_rs17114036_rs17114036	

In [10]:
def get_matching_element_between_ref_alt(header):
    """Get the rsid from the header"""
    if 'C_positive_heart_CAD' in header:
        return header.split('_')[-1] # C_positive_heart_CAD:ALT_rs17114036_rs17114036
    elif 'GC_Selvarajan' in header:
        # think about tiling: add the tile number to the rsid with an _tile<number>
        tile_number = header.split('tile')[-1].split('-')[0]
        return '%s_tile%s'%(header.split('_')[2].split('|')[0], tile_number) # GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile1-3
    elif 'GC_Liang' in header:
        return header.split('_')[2].split('|')[0] # GC_Liang:REF_rs2125358|Liang_fwd_tile1-1
    else:
        print('unexpected')

def get_rsID_from_GC_Selvarajan(header, number=0):
    """Get the rsid from the header"""
    if number == 0:
        return get_matching_element_between_ref_alt(header)
    else:
        substr_header = header.split('~')[number]
    return substr_header.split('|')[0]

In [12]:
# example for C_positive_heart_CAD
from collections import defaultdict
 
C_positive_heart_CAD = variant_control_vars[variant_control_vars[col_info].isin(['C_positive_heart_CAD'])]
header_list = C_positive_heart_CAD[col_name].to_list()


# Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
# each ref should have at least 1 element in dict
header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
ref_dict = defaultdict(list)
for header in header_list:
    if 'REF_' in header: # refs need to be unique
        ref_unique_element = get_matching_element_between_ref_alt(header)
        for alt_header in header_list:
            if 'ALT_' in alt_header:
                alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                if ref_unique_element == alt_unique_element:
                    ref_dict[header].append(alt_header)
    # break
ref_dict

# # investigate number of elements in ref_dict values
# for ref, alt in ref_dict.items():
#     if len(alt) != 1:
#         print(ref)

# prepare dataframe with ref and alt from ref_dict
ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
id_list = []
ref_list = []
alt_list = []
for ref, alt in ref_dict.items():
    for elem in alt:
        unique_id = get_matching_element_between_ref_alt(ref)
        unique_header = '%s:%s'%(get_label(ref), unique_id)
        id_list.append(unique_header)
        ref_list.append(ref)
        alt_list.append(elem)
ref_alt_df['ID'] = id_list
ref_alt_df['REF'] = ref_list
ref_alt_df['ALT'] = alt_list
ref_alt_df
outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%('C_positive_heart_CAD')
ref_alt_df.to_csv(outpath, sep='\t', index=False)
    

#### For the easy ones: C_positive_heart_CAD, GC_Liang

In [13]:
from collections import defaultdict

for group in ['C_positive_heart_CAD', 'GC_Liang']:
    group_vars = variant_control_vars[variant_control_vars[col_info] == group]
    header_list = group_vars[col_name].to_list()


    # Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
    # each ref should have at least 1 element in dict
    header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
    ref_dict = defaultdict(list)
    for header in header_list:
        if 'REF_' in header: # refs need to be unique
            ref_unique_element = get_matching_element_between_ref_alt(header)
            for alt_header in header_list:
                if 'ALT_' in alt_header:
                    alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                    if ref_unique_element == alt_unique_element:
                        ref_dict[header].append(alt_header)


    # # investigate number of elements in ref_dict values
    # for ref, alt in ref_dict.items():
    #     if len(alt) != 1:
    #         print(ref)

    # prepare dataframe with ref and alt from ref_dict
    ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
    id_list = []
    ref_list = []
    alt_list = []
    for ref, alt in ref_dict.items():
        for elem in alt:
            unique_id = get_matching_element_between_ref_alt(ref)
            unique_header = '%s:%s'%(get_label(ref), unique_id)
            id_list.append(unique_header)
            ref_list.append(ref)
            alt_list.append(elem)
    ref_alt_df['ID'] = id_list
    ref_alt_df['REF'] = ref_list
    ref_alt_df['ALT'] = alt_list
    ref_alt_df
    outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%(group)
    ref_alt_df.to_csv(outpath, sep='\t', index=False)

In [ ]:
from collections import defaultdict
 
C_positive_heart_CAD = variant_control_vars[variant_control_vars[col_info].isin(['C_positive_heart_CAD'])]
header_list = C_positive_heart_CAD[col_name].to_list()


# Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
# each ref should have at least 1 element in dict
header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
ref_dict = defaultdict(list)
for header in header_list:
    if 'REF_' in header: # refs need to be unique
        ref_unique_element = get_matching_element_between_ref_alt(header)
        for alt_header in header_list:
            if 'ALT_' in alt_header:
                alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                if ref_unique_element == alt_unique_element:
                    ref_dict[header].append(alt_header)
    # break
ref_dict

# # investigate number of elements in ref_dict values
# for ref, alt in ref_dict.items():
#     if len(alt) != 1:
#         print(ref)

# prepare dataframe with ref and alt from ref_dict
ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
id_list = []
ref_list = []
alt_list = []
for ref, alt in ref_dict.items():
    for elem in alt:
        unique_id = get_matching_element_between_ref_alt(ref)
        unique_header = '%s:%s'%(get_label(ref), unique_id)
        id_list.append(unique_header)
        ref_list.append(ref)
        alt_list.append(elem)
ref_alt_df['ID'] = id_list
ref_alt_df['REF'] = ref_list
ref_alt_df['ALT'] = alt_list
ref_alt_df
outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%('C_positive_heart_CAD')
ref_alt_df.to_csv(outpath, sep='\t', index=False)

#### GC_Selvarajan variant negative control 364 (try to generalize)
- GC_Selvarajan:REF_rs4848980|STARR-seq-HepG2_fwd_tile1-1	GC_Selvarajan:ALT_rs4848980|STARR-seq-HepG2_fwd_tile1-1_rs4848980 
- Merged headers: GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1


In [45]:
# how many alt in set of variant controls

selvarajan = variant_control_vars[variant_control_vars[col_info] == 'GC_Selvarajan']
selvarajan_alt = selvarajan[selvarajan[col_allele] == 'alt']
selvarajan_alt # => 199 but in subsequent table are only 198 entries (Hypothesis: are duplicated alts in there? OR One alternative does not have a reference)

# duplicates:
selvarajan_alt[col_name].duplicated().sum()

# check in ref_dict number of alternative
alt_count = 0
for ref, alt in ref_dict.items():
    alt_count += len(alt)
print(alt_count) # 198

# concat all alternative from ref_dict and check which is missing from selvarajan_alt list
all_alt = []
for alt in ref_dict.values():
    all_alt += alt

all_alt_in_df = selvarajan_alt[col_name].to_list()
all_alt_in_df_set = set(all_alt_in_df)
print(len(all_alt_in_df_set))
all_matched_alt_set = set(all_alt)
print(len(all_matched_alt_set))

# symmetric_difference
all_alt_in_df_set.symmetric_difference(all_matched_alt_set)

198
199
198


AttributeError: 'list' object has no attribute 'symmetric_difference'

In [31]:
from collections import defaultdict

# for group in ['C_positive_heart_CAD', 'GC_Selvarajan']:
for group in ['GC_Selvarajan']:
    group_variants = variant_control_vars[variant_control_vars[col_info] == group]
    header_list = group_variants[col_name].to_list()

    # Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
    # each ref should have at least 1 element in dict
    header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
    ref_dict = defaultdict(list)
    for header in header_list:
        if 'REF_' in header: # refs need to be unique
            # check tiling
            ref_unique_element = get_matching_element_between_ref_alt(header)
            for alt_header in header_list:
                if 'ALT_' in alt_header:
                    alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                    if ref_unique_element == alt_unique_element:
                        ref_dict[header].append(alt_header)
        # break
    # print(ref_dict)

    # # investigate number of elements in ref_dict values
    for ref, alt in ref_dict.items():
        if len(alt) != 1:
            # throw error
            print('WARNING: more than one alt for ref')
            print(ref)

    # prepare dataframe with ref and alt from ref_dict
    ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
    id_list = []
    ref_list = []
    alt_list = []
    for ref, alt in ref_dict.items():
        alt_count = 0
        print(ref)
        print(alt)
        for elem in alt:
            # if len(alt) > 1: headers are merged => we need different unique ids => need a function which get the alt_counts rsID from the ref
            unique_id = get_rsID_from_GC_Selvarajan(ref, alt_count)
            unique_header = '%s:%s'%(get_label(ref), unique_id)
            id_list.append(unique_header)
            ref_list.append(ref)
            alt_list.append(elem)
            alt_count += 1
    ref_alt_df['ID'] = id_list
    ref_alt_df['REF'] = ref_list
    ref_alt_df['ALT'] = alt_list
    ref_alt_df
    # outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv.gz'%(group)
    # ref_alt_df.to_csv(outpath, sep='\t', index=False, compression='gzip')
    print(ref_alt_df.shape)
    outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%(group)
    ref_alt_df.to_csv(outpath, sep='\t', index=False)

GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs7280278|STARR-seq-HepG2~rs7282405|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs7139492|STARR-seq-HepG2~rs7316984|STARR-seq-HepG2_fwd_tile2-3
GC_Selvarajan:REF_rs58217496|STARR-seq-HepG2~rs58324269|STARR-seq-HepG2~rs60089052|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs494207|STARR-seq-HepG2~rs649192|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs4845619|STARR-seq-HepG2~rs56383622|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs4537545|STARR-seq-HepG2~rs4576655|STARR-seq-HepG2_fwd_tile2-3
GC_Selvarajan:REF_rs3781780|STARR-seq-HepG2~rs3781781|STARR-seq-HepG2_fwd_tile1-1
GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile3-3
GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile2-3
GC_Selvarajan:REF_rs2993510|STARR-seq-HepG2~rs72856440|STARR-seq-HepG2_fwd_tile1-3
GC_Selvarajan:REF_rs2820321|STARR-seq-HepG2~rs2820322|STARR-seq-Hep

#### GC_Liang  variant negative control  16
- GC_Liang:REF_rs2125358|Liang_fwd_tile1-1 GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs2125358

In [15]:
from collections import defaultdict
GC_Liang = variant_control_vars[variant_control_vars[col_info] == 'GC_Liang']
header_list = GC_Liang[col_name].to_list()
header_list

# Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
# each ref should have at least 1 element in dict
header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
ref_dict = defaultdict(list)
for header in header_list:
    if 'REF_' in header: # refs need to be unique
        ref_unique_element = get_matching_element_between_ref_alt(header)
        for alt_header in header_list:
            if 'ALT_' in alt_header:
                alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                if ref_unique_element == alt_unique_element:
                    ref_dict[header].append(alt_header)
    # break
ref_dict

# # investigate number of elements in ref_dict values
# for ref, alt in ref_dict.items():
#     if len(alt) != 1:
#         print(ref)

# prepare dataframe with ref and alt from ref_dict
ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
id_list = []
ref_list = []
alt_list = []
for ref, alt in ref_dict.items():
    for elem in alt:
        unique_id = get_matching_element_between_ref_alt(ref)
        unique_header = '%s:%s'%(get_label(ref), unique_id)
        id_list.append(unique_header)
        ref_list.append(ref)
        alt_list.append(elem)
ref_alt_df['ID'] = id_list
ref_alt_df['REF'] = ref_list
ref_alt_df['ALT'] = alt_list
ref_alt_df

,ID,REF,ALT
0,GC_Liang:rs2838227,GC_Liang:REF_rs2838227|Liang_fwd_tile1-1,GC_Liang:ALT_rs2838227|Liang_fwd_tile1-1_rs283...
1,GC_Liang:rs2530731,GC_Liang:REF_rs2530731|Liang_fwd_tile1-1,GC_Liang:ALT_rs2530731|Liang_fwd_tile1-1_rs253...
2,GC_Liang:rs2125358,GC_Liang:REF_rs2125358|Liang_fwd_tile1-1,GC_Liang:ALT_rs2125358|Liang_fwd_tile1-1_rs212...
3,GC_Liang:rs17882077,GC_Liang:REF_rs17882077|Liang_fwd_tile1-1,GC_Liang:ALT_rs17882077|Liang_fwd_tile1-1_rs17...
4,GC_Liang:rs17603855,GC_Liang:REF_rs17603855|Liang_fwd_tile1-1,GC_Liang:ALT_rs17603855|Liang_fwd_tile1-1_rs17...
5,GC_Liang:rs10939614,GC_Liang:REF_rs10939614|Liang_fwd_tile1-1,GC_Liang:ALT_rs10939614|Liang_fwd_tile1-1_rs10...
6,GC_Liang:rs10502466,GC_Liang:REF_rs10502466|Liang_fwd_tile1-1,GC_Liang:ALT_rs10502466|Liang_fwd_tile1-1_rs10...
7,GC_Liang:rs1036014,GC_Liang:REF_rs1036014|Liang_fwd_tile1-1,GC_Liang:ALT_rs1036014|Liang_fwd_tile1-1_rs103...


#### C_positive_neuron_CD   variant positive control     91
- according to mohan not really variant controls
- C_positive_neuron_CD:p1_rs7115714_G_A_ref_50_A::chr11:120424017-120424287-mean_ratio2.22  C_positive_neuron_CD:p1_rs7115714_G_A_alt_50_A::chr11:120424017-120424287-mean_ratio2.13

In [16]:
from collections import defaultdict

for group in ['']
    C_positive_neuron_CD = variant_control_vars[variant_control_vars[col_info] == 'C_positive_neuron_CD']
    header_list = C_positive_neuron_CD[col_name].to_list()
    header_list

    # Go over each REF: put sequence into dict and if rsID is matching (rsID=ALT_<rsID>) then put into dict and compare
    # each ref should have at least 1 element in dict
    header_list.sort(reverse=True) # first REF then ALT (headers are not different until "ALT_" or "REF_")
    ref_dict = defaultdict(list)
    for header in header_list:
        if '_ref_' in header: # refs need to be unique
            ref_unique_element = get_matching_element_between_ref_alt(header)
            # print(header)
            for alt_header in header_list:
                if '_alt_' in alt_header:
                    alt_unique_element = get_matching_element_between_ref_alt(alt_header)
                    # print(alt_unique_element)
                    if ref_unique_element == alt_unique_element:
                        ref_dict[header].append(alt_header)
                        # print("-------- FOUND ---------")
        # break
    ref_dict

    # # investigate number of elements in ref_dict values
    # for ref, alt in ref_dict.items():
    #     if len(alt) != 1:
    #         print(ref)

    # prepare dataframe with ref and alt from ref_dict
    ref_alt_df = pd.DataFrame(columns=['ID', 'REF', 'ALT'])
    id_list = []
    ref_list = []
    alt_list = []
    for ref, alt in ref_dict.items():
        for elem in alt:
            unique_id = get_matching_element_between_ref_alt(ref)
            unique_header = '%s:%s'%(get_label(ref), unique_id)
            id_list.append(unique_header)
            ref_list.append(ref)
            alt_list.append(elem)
    ref_alt_df['ID'] = id_list
    ref_alt_df['REF'] = ref_list
    ref_alt_df['ALT'] = alt_list
    ref_alt_df

    outpath = 'results/variant_control_map/variant_control_ref_alt_%s.tsv'%(group)
    ref_alt_df.to_csv(outpath, sep='\t', index=False)

SyntaxError: invalid syntax (605436688.py, line 3)

In [5]:
pre_metadata_df_final

,name,sequence,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,scrambled,element inactive control,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,MK


In [21]:
pre_metadata_df[col_info].value_counts()

info
cardiac_neuro_cava_random                                                                  73940
MK                                                                                          2397
C_positive_heart_AB                                                                          909
GC_Selvarajan                                                                                364
GC_Vista                                                                                     256
C_negative_heart_MK                                                                          243
C_negative_neuron_MK                                                                         222
C_negative_neuron_NP                                                                         217
GC_Mendelian_variants                                                                        209
GC_Kircher                                                                                   203
C_SLEA                   

In [148]:
# test category and variant_class
pre_metadata_df[pre_metadata_df[col_category] == 'variant']

,header,sequence,id,category,class,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info
8900,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,oligo_17b5d1f079f92f165ae6b760da616314,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8901,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,oligo_2e53f6e8c61ef7cd84a0c218519435a9,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8902,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,oligo_fa6b9ab66a2c9d8a83e0b807968d16de,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8903,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,oligo_9be9e91086b215fc8a407ca5505bf01d,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
8904,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,oligo_512f507b4c554969462b4ce5da071afb,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,ref,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73935,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,oligo_6e74a25bd02b34708ccbf8093673ef62,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73936,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,oligo_87305b21de48ce90a3caf03cd0265009,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73937,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,oligo_07249343e8d909bfd63abcc7ff28b50d,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA
73938,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTCCTCTGCCCTCCCTGGCTTCTTCCCCTGTCC...,oligo_e35878121d9479168994f561fc4a4028,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,SNV,NA,NA,alt,NA


#### Correlation between chengyu and log2ratios


### Add knowledge on controls


In [22]:
# checking manually: found many ALT_REF pairs
# - Weird: GC_Vista have ";" do not follow similar pattern (no ALT_ or REF_)
pre_metadata_df.loc[:,'tmp_label'] = pre_metadata_df['header'].apply(get_label)


# check which controls are element and variant 

# check which controlls are positive and negative (positive: for neuro positive, negative: for neuro negative)

# store headers to a tsv file:
header_path = 'controll_header.tsv'

controls = pre_metadata_df[pre_metadata_df['tmp_label'] != 'cardiac_neuro_cava_random']
controls['header'].to_csv(header_path, sep='\t', index=False)

In [9]:
controls['header']

73940    GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...
73941    GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...
73942    GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...
73943    GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...
73944    GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...
                               ...                        
80210    MK:tile_2240|chr1-116244322+116244591|scramble...
80211    MK:tile_6675|chr11-2374617+2374886|scramble_ne...
80212    MK:tile_18415|chr17-71181691+71181960|scramble...
80213    MK:tile_14356|chr15-67031618+67031887|scramble...
80214    MK:tile_22033|chr2-71506006+71506275|scramble_...
Name: header, Length: 6275, dtype: object

In [10]:
controls['tmp_label'].value_counts()


tmp_label
MK                                   2397
C_positive_heart_AB                   909
GC_Selvarajan                         364
GC_Vista                              256
C_negative_heart_MK                   243
C_negative_neuron_MK                  222
C_negative_neuron_NP                  217
GC_Mendelian_variants                 209
GC_Kircher                            203
C_SLEA                                200
GC_Cort_Chengyu                       185
C_positive_neuron_NP                   99
C_positive_heart_CAD                   97
C_positive_heart_MK                    97
C_positive_neuron_MK                   96
C_positive_neuron_CD                   94
GC_GABA_Chengyu                        85
GC_DNase_positive_shuffeled            55
GC_Atrial_fib                          45
GC_DNase_positive                      41
GC_Glut_Chengyu                        40
GC_Mohlke                              34
GC_DNase_negative_blood_shuffeled      19
GC_Liang                

In [ ]:
# interesting for NGN2 experiment: 
positive_controls = ['C_positive_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_CD'] # 99, 96, 94
negative_controls = ['C_negative_neuron_NP', 'C_negative_neuron_MK'] # 217, 222
# # maybe
# GC_Liang    16
# GC_Mohlke   34
# MK
# GC_Hon

In [11]:
def ref_or_alt_in_header(header):
    """Checks if "ref_" or "_alt" is in the header"""
    if 'ref_' in header.lower() or 'alt_' in header.lower():
        return True
    return False

def set_negative_class(header):
    if ref_or_alt_in_header(header):
        return 'variant negative control'
    return 'element inactive control'

def set_positive_class(header):
    if ref_or_alt_in_header(header):
        return 'variant positive control'
    return 'element active control'

#### Plan:
0. Look only on controls
1. get the positive control groups and the synthetic groups
2. iterate all groups (except for the positive control groups) (because you want to skipp the positive control groups)
  - when their headers have "alt_" or "ref_" included: 
    - 'variant negative control'
  - otherwise: 
    - 'element inactive control'

In [16]:
# get all unique values of the tmp_label column 
all_control_groups = controls['tmp_label'].unique()
# todo: add additional information; 
# todo: add label to source
### please put in the positive list the controls you want to be positive controls
positive_control_groups = ['C_positive_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_CD']
synthetic_control_groups = ['C_SLEA']
additional_information_groups = [group for group in all_control_groups if "dnase" in group.lower()]
additional_information_groups
negative_control_groups = [group for group in all_control_groups if not group in positive_control_groups]
negative_control_groups


['GC_Atrial_fib',
 'GC_Liang',
 'GC_Selvarajan',
 'GC_Mohlke',
 'GC_Kircher',
 'GC_Mendelian_variants',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_GABA_Chengyu',
 'GC_Glut_Chengyu',
 'GC_Hon',
 'GC_Vista',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'C_negative_heart_MK',
 'C_negative_neuron_MK',
 'C_negative_neuron_NP',
 'C_positive_heart_MK',
 'C_positive_heart_AB',
 'C_SLEA',
 'GC_DNase_positive_shuffeled',
 'GC_DNase_negative_brain_shuffeled',
 'GC_DNase_negative_blood_shuffeled',
 'MK']

In [13]:
group_count = 0

updated_neg_control = pd.DataFrame(columns=["header", "class"])
length_counter = 0
for group in negative_control_groups:
    print(group)
    group_control = controls[controls['tmp_label'] == group]
    # check if some MK_control header have REF or ALT included included
    group_control.loc[:,col_class] = group_control['header'].apply(set_negative_class)
    group_control_sub = group_control[['header', col_class]]
    # print(group_control_sub[col_class].value_counts())
    # join this information to the controls pandas dataframe
    updated_neg_control = pd.concat([updated_neg_control, group_control_sub], ignore_index=True, sort=False) 
    length_counter += group_control_sub.shape[0]
    # print(updated_neg_control[col_class].value_counts())


updated_neg_control[col_class].value_counts()



GC_Atrial_fib
GC_Liang
GC_Selvarajan
GC_Mohlke
GC_Kircher
GC_Mendelian_variants
C_positive_heart_CAD
GC_Cort_Chengyu
GC_GABA_Chengyu
GC_Glut_Chengyu
GC_Hon
GC_Vista
GC_DNase_positive
GC_DNase_negative_brain
GC_DNase_negative_blood
C_negative_heart_MK
C_negative_neuron_MK
C_negative_neuron_NP
C_positive_heart_MK
C_positive_heart_AB
GC_DNase_positive_shuffeled
GC_DNase_negative_brain_shuffeled
GC_DNase_negative_blood_shuffeled
MK


class
element inactive control    4819
variant negative control     967
Name: count, dtype: int64

In [14]:
print('In this dataset are %s negative controls'%(updated_neg_control['header'].shape[0])) # 5986

In this dataset are 5786 negative controls


#### Do same with positive controls

In [163]:

updated_pos_control = pd.DataFrame(columns=["header", "class"])
length_counter = 0
for group in positive_control_groups:
    print(group)
    group_control = controls[controls['tmp_label'] == group]
    # check if some MK_control header have REF or ALT included included
    group_control.loc[:,col_class] = group_control['header'].apply(set_positive_class)
    group_control_sub = group_control[['header', col_class]]
    # print(group_control_sub[col_class].value_counts())
    # join this information to the controls pandas dataframe
    updated_pos_control = pd.concat([updated_pos_control, group_control_sub], ignore_index=True, sort=False) 
    length_counter += group_control_sub.shape[0]
    # print(updated_pos_control[col_class].value_counts())


updated_pos_control[col_class].value_counts()

C_positive_neuron_NP
C_positive_neuron_MK
C_positive_neuron_CD


class
element active control      198
variant positive control     91
Name: count, dtype: int64

In [164]:
length_counter

289

In [165]:
updated_pos_control

,header,class
0,C_positive_neuron_NP:GW18_PFC_ABC_chr5_6873074...,element active control
1,C_positive_neuron_NP:GW18_PFC_ABC_chr5_6873083...,element active control
2,C_positive_neuron_NP:NGN2_iPSC_ABC_chrX_743252...,element active control
3,C_positive_neuron_NP:GW18_PFC_ABC_NGN2_iPSC_AB...,element active control
4,C_positive_neuron_NP:NGN2_iPSC_ABC_chr10_88387...,element active control
...,...,...
284,C_positive_neuron_CD:p1_rs8049948_A_G_ref_50_A...,variant positive control
285,C_positive_neuron_CD:p1_rs6791336_T_C_ref_50_T...,variant positive control
286,C_positive_neuron_CD:p1_rs4982404_C_T_ref_50_C...,variant positive control
287,C_positive_neuron_CD:p2_rs9926320_G_A_ref_25_A...,variant positive control


In [166]:
# merge with metadata df 
pre_metadata_df_test = pre_metadata_df.merge(updated_neg_control, on='header', how='left')

pre_metadata_df_test[col_class] = pre_metadata_df_test['class_y'].fillna(pre_metadata_df_test['class_x'])
pre_metadata_df_test = pre_metadata_df_test.drop(['class_x', 'class_y'], axis=1)


pre_metadata_df_test = pre_metadata_df_test.merge(updated_pos_control, on='header', how='left')
pre_metadata_df_test[col_class] = pre_metadata_df_test['class_y'].fillna(pre_metadata_df_test['class_x'])
pre_metadata_df_test = pre_metadata_df_test.drop(['class_x', 'class_y'], axis=1)

pre_metadata_df_test[col_class].value_counts()

class
test                        73940
element inactive control     5019
variant negative control      967
element active control        198
variant positive control       91
Name: count, dtype: int64

In [167]:
# check SLEA controlls
pre_metadata_df_test[pre_metadata_df_test['tmp_label'] == 'C_SLEA']

,header,sequence,id,category,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info,tmp_label,class
77528,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,oligo_e2a545a9d984399d75f1829e31a1a8b6,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77529,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,oligo_efa28f0df5e5feed32a5fcdf423e9f0b,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77530,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,AGGACCGGATCAACTTAGGCTTCTCACCCCCTGACCTTTGCCCCCT...,oligo_d901ad4b07a7c993cb92b33f3f267a64,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77531,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,AGGACCGGATCAACTTAGGCTTCTCATGTTTGCTTTGTAACAAAAT...,oligo_7bd5f3c9c84842756445d26d59214da4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77532,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,AGGACCGGATCAACTTAGGCTTCTCAAAGGTCCAGTTTGGGGATCG...,oligo_52e5854cec759d31e5c2dcd1d01e76a4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77723,C_SLEA:SLEA_hg18:chr9:82902419-82902586|6:V_Rx...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGGGCCGTGACCCCGT...,oligo_97208d39b8763704a889856eb6bc5b9b,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77724,C_SLEA:SLEA_hg18:chr9:82902419-82902586|7:V_AH...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCGGGGATCGCGTGC...,oligo_1b6d517fd936db6af97c8b06328f191e,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77725,C_SLEA:SLEA_hg18:chr9:82902419-82902586|80:V_H...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAGGCAAGAAGTG...,oligo_6377a5a1eab86d0c18d6e3f3267a949d,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control
77726,C_SLEA:SLEA_hg18:chr9:82902419-82902586|8:V_HN...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAAGGTCCAGGTG...,oligo_49c1874afef2c81f8e56fe36231a1f65,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,C_SLEA,element inactive control


In [1]:


def get_control_category_from_class(row):
    """Set category of controls from class"""
    if row['tmp_label'] == "cardiac_neuro_cava_random":
        return row['catgegory']
    
    category_conversion_dict = {
        'variant positive control': 'variant', 
        'variant negative control': 'variant', 
        'element active control': 'element', 
        'element inactive control': 'element'
    }
    
    # C_SLEA
    if "C_SLEA" in row['tmp_label']:
        return 'synthetic'
    
    # scramble in name
    if "scramble" in row['header']:
        return 'scrambled'
    
    return category_conversion_dict[row[col_class]]

In [144]:
# set category
pre_metadata_df_test.apply(set_control_category_from_class)

,header,sequence,id,category,source,ref_sequence,chrom,chrom_start,chrom_end,variant_class,variant_pos,SPDI,allele,info,tmp_label,class
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,oligo_c32acb98ad2a851ab46621b1c3af8b44,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,oligo_8deb96fab75f4f41dbc05b2d163ad89b,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,oligo_d08942ae2ac12327ebaad04b395b7dc5,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,oligo_d0ac046887b1f97d9c494e6a2bae69f7,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,oligo_328edd51262a4c1c0ea79c9043176d20,element,"candidate CRE nearby cardiac, neuro, cava and ...",hg38,NA,NA,NA,NA,NA,NA,NA,NA,cardiac_neuro_cava_random,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,oligo_23474e71ce5c02846892d2e5f40bdfc4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,oligo_7cf2325b7ef95359164092b47379b3d5,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,oligo_db734997542b4f69c38d4d3c6e88603e,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,oligo_095855115099aa9e709dc0452fbc4edd,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,MK,element inactive control


##### Set negative controls to 

#### positive controls

## Start creating the Variant region lists:

In [5]:
# is there a header without "tile"? => No
design_df[(design_df['header'].str.contains("cardiac_neuro_cava_random") == True) & (design_df['header'].str.contains('tile') == False)]

,header,sequence,label


In [6]:
# filter for the rows with "cardiac_neuro_cava_random" in label column
cardiac_neuro_cava_random = design_df[design_df['label'] == 'cardiac_neuro_cava_random']

# do all of these rows have 2 pipes in the header?
cardiac_neuro_cava_random['header'].str.count(r'\|').value_counts()


# header
# 5     45082
# 2     27141
# 8       596
# 7       471
# 4       341
# 10      30


header
5     45082
2     27141
8       596
7       471
4       341
10      309
Name: count, dtype: int64

#### 2 pipe example
- 18340 REF
- 27141 all (if no REF_ or ALT_ then it is a region)
- example 
    - cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1
    - cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
- format: <label> : <sequence_type> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>
- final columns: ['header', 'sequence', 'label', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
- pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>[a-z]+\d+[-]+\d+)'



In [17]:
# show me an example with threshold pipes in the header
num_pipes = 2
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1' # gene name | ensembl id | enhancer/encodeID_tile id

# does a row with 2 pipes in the header have ALT in the header?
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].str.contains('ALT_').value_counts() # no alt there
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].str.contains('REF_').value_counts() # no ref there # 18340 REF

# investigate the rows with 2 pipes in the header and ALT in the header
card_cava_2_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
print(card_cava_2_pipes.shape) # (27141, 3)
# card_cava_2_pipes['header'].str.contains('ALT')]['header'].tolist()[0]
card_cava_2_pipes[card_cava_2_pipes['header'].str.contains('REF_')]['header'].tolist()[10] # 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

(27141, 3)


'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1'

##### split the dataframe 2 pipe 


In [18]:
# 1: pre_header column: header without the label
# 2: tile info column: tile info (split by "_" and take the last element) + remove this from the pre_header column
# 3: strand column: strand info (split by "_" and take again the last element) + remove this from the pre_header column
# 4: gene_name + ensembl_id + enhancer_id column: split the pre_header column by "|" and take the first 3 elements (expand = True)
# format: <label> : <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>

# short way using regex
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>[a-z]+\d+[-]+\d+)'
df_extracted = card_cava_2_pipes['header'].str.extract(pattern)
df_extracted
# concatenate df_extracted and card_cava_2_pipes
card_cava_2_pipes = pd.concat([card_cava_2_pipes, df_extracted], axis = 1)
card_cava_2_pipes

# long way: splitting manually
# # split the header column by ":" and take the second element
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['header'].str.split(':').str[1:].str.join(':')
# # tile info: split the pre_header column by "_" and take the last element
# card_cava_2_pipes['tile_info'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# # remove the tile info from the pre_header column
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# # strand info: split the pre_header column by "_" and take the last element
# card_cava_2_pipes['strand'] = card_cava_2_pipes['pre_header'].str.split('_').str[-1]
# # remove the strand info from the pre_header column
# card_cava_2_pipes['pre_header'] = card_cava_2_pipes['pre_header'].str.split('_').str[:-1].str.join('_')
# # gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# card_cava_2_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_2_pipes['pre_header'].str.split('|', expand = True)
# # remove the pre_header column
# card_cava_2_pipes.drop(columns = ['pre_header'], inplace = True)
# # category: element or variant
# card_cava_2_pipes[col_category] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: 'element' if '_' not in x else 'variant')
# # put alt or ref if "_"
# card_cava_2_pipes[col_allele] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: 'NA' if '_' not in x else x.split('_')[0].lower())
# # add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
# card_cava_2_pipes['gene_name'] = card_cava_2_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# # drop the pre_gene_name column
# card_cava_2_pipes.drop(columns = ['pre_gene_name'], inplace = True)

# print(card_cava_2_pipes.columns)
# # ['header', 'sequence', 'label', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
card_cava_2_pipes

,header,sequence,label,label,allel,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778476,fwd,tile1-1
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778477,fwd,tile1-1
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778478,fwd,tile1-1
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778480,fwd,tile1-1
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,NaN,SKI,ENSG00000157933.11,EH38E2778484,fwd,tile1-1
...,...,...,...,...,...,...,...,...,...,...
27477,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTACCCCACTGCTGCACCAGATTGAGCTGGAGA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949715,rev,tile1-1
27478,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTTTGCTGAGTAGTATCCGTTGTATGAATGCAC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949725,rev,tile1-1
27479,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E3949733,rev,tile1-1
27480,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random,REF_,G6PD,ENSG00000160211.20,EH38E2774396,rev,tile1-1


#### 4 pipe example (what group is this, mohan?)
- 242 REF / 99 (region)
- example: 
    - cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1
    - cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
- format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
- assumption reference is also category variant
- final columns: ['header', 'sequence', 'label', 'additional_gene_association',
       'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category',
       'allele', 'gene_name'] => additional gene association

In [13]:
num_pipes = 4
# Examples for the header string
# cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1
# cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
# format: `<label> : [<allele>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
# Problem: allele is somethimes given (e.g. REF_ or "") but not everytime 
# solution regex: 
# pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.+)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*)'
card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
# Extract values into new columns
df_extracted = card_cava_4_pipes['header'].str.extract(pattern)
df_extracted.head()

341


allel
REF_    242
Name: count, dtype: int64

In [14]:
# show me an example with threshold pipes in the header
num_pipes = 4
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[0]
# 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] # 341 rows
print(len(card_cava_4_pipes)) # 341
# do all these rows have ALT in the header?
card_cava_4_pipes['header'].str.contains('ALT_').value_counts() # no, no alt 
card_cava_4_pipes['header'].str.contains('REF_').value_counts() # True, 242 

# check the header of something containing REF_
card_cava_4_pipes[card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1'

# check the header of something not containing REF_
card_cava_4_pipes[~card_cava_4_pipes['header'].str.contains('REF_')]['header'].tolist()[0] # 'cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1'

# check how many rows have ~ in the header
card_cava_4_pipes['header'].str.count('~').value_counts() # all have just one "~" 341

# check if all rows have 3 "_" in the header
card_cava_4_pipes['header'].str.count('_').value_counts() # all have 3 "_" 341

341


header
6    242
5     99
Name: count, dtype: int64

In [15]:
# format: `<label> : [<sequence_type>_]*<gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

# fast way using regex: 
pattern = r'(?P<label>.*):(?P<allel>[A-Z]*_)?(?P<gene_name>.+)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.+)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*)'
card_cava_4_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]
df_extracted = card_cava_4_pipes['header'].str.extract(pattern)
print(df_extracted.head())

# # Concatenate the extracted columns with the original DataFrame
card_cava_4_pipes = pd.concat([card_cava_4_pipes, df_extracted], axis=1)
print(card_cava_4_pipes.head())

# long way: splitting manually
# # split the header column by ":" and take the second element
# card_cava_4_pipes['pre_header'] = card_cava_4_pipes['header'].str.split(':').str[1:].str.join(':')
# # additional_gene_association: split the pre_header column by "~" and take the second element
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['pre_header'].str.split('~').str[1]
# # tile info: split the additional_gene_association column by "_" and take the last element
# card_cava_4_pipes['tile_info'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# # remove the tile info from the additional_gene_association column
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# # strand info: split the additional_gene_association column by "_" and take the last element
# card_cava_4_pipes['strand'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[-1]
# # remove the strand info from the additional_gene_association column
# card_cava_4_pipes['additional_gene_association'] = card_cava_4_pipes['additional_gene_association'].str.split('_').str[:-1].str.join('_')
# # remove the additional_gene_association column from the pre_header column
# card_cava_4_pipes['pre_header'] = card_cava_4_pipes['pre_header'].str.split('~').str[0]
# # pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# card_cava_4_pipes[['pre_gene_name', 'ensembl_id', 'enhancer_id']] = card_cava_4_pipes['pre_header'].str.split('|', expand = True)
# # remove the pre_header column
# card_cava_4_pipes.drop(columns = ['pre_header'], inplace = True)
# # category: element or variant
# card_cava_4_pipes[col_category] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: 'element' if '_' not in x else 'variant')
# # sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element
# card_cava_4_pipes['allele'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: 'NA' if '_' not in x else x.split('_')[0].lower())

# # add gene_name column: split the pre_gene_name column by "_" and take the second element but only if "_" exists in the pre_gene_name column
# card_cava_4_pipes['gene_name'] = card_cava_4_pipes['pre_gene_name'].apply(lambda x: x.split('_')[1] if '_' in x else x)
# # drop the pre_gene_name column
# card_cava_4_pipes.drop(columns = ['pre_gene_name'], inplace = True)

# print(card_cava_4_pipes.columns)
# # ['header', 'sequence', 'label', 'additional_gene_association', 'tile_info', 'strand', 'ensembl_id', 'enhancer_id', 'category', 'allele', 'gene_name']
# card_cava_4_pipes


                         label allel gene_name          ensembl_id  \
708  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
709  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
710  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
711  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   
712  cardiac_neuro_cava_random   NaN     CSDE1  ENSG00000009307.17   

      enhancer_id gene_name2        ensembl_id2  enhancer_id2 fwd_rev  \
708  EH38E1378377       NRAS  ENSG00000213281.5  EH38E1378377     rev   
709  EH38E2832502       NRAS  ENSG00000213281.5  EH38E2832502     rev   
710  EH38E2832508       NRAS  ENSG00000213281.5  EH38E2832508     rev   
711  EH38E2832513       NRAS  ENSG00000213281.5  EH38E2832513     rev   
712  EH38E1378387       NRAS  ENSG00000213281.5  EH38E1378387     rev   

    tile_info  
708   tile1-1  
709   tile1-1  
710   tile1-1  
711   tile1-1  
712   tile1-1  
                                            

enhancer id column; other genes with same enhancer
- mohan idea: 2D array or 3D array (gene column will have multiple genes)
- EH38E1378377~NRAS|ensemble id ... 
- fwd: (+ strand)
- rev: (- strand)
- gene name does not implicitly tell you the enhancer id (e.g. JUP|ENSG00000173801.17|EH38E3223379)

#### 5 pipe examples are all alt / variants (45082)
- 45082 alt
- example: 
    - cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
    - cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A
    - cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C
- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'

In [43]:
num_pipes = 5
# show me an example with 5 pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[2323]
# idx 0: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
# idx 10: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778513_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778513|1-2203222-G-A
# idx 24: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778546_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778546|1-2215527-T-C
# idx 2400: cardiac_neuro_cava_random:ALT_POMGNT1|ENSG00000085998.15|EH38E2809115_rev_tile1-1_POMGNT1|ENSG00000085998.15|EH38E2809115|1-46191391-T-C
# idx 2323: cardiac_neuro_cava_random:ALT_ST3GAL3|ENSG00000126091.21|EH38E1342813_fwd_tile1-1_ST3GAL3|ENSG00000126091.21|EH38E1342813|1-43835787-G-A


# investigate and find pattern
card_cava_5_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_5_pipes['header'].str.split(':').str[1].str.contains('ALT_').value_counts() # yes => all with 5 pipes have ALT_ in name (45082)
# card_cava_5_pipes['header'].str.split(':').str[1].str.contains('REF_').value_counts() #

# # do all of these rows end with the regex /-[A-Z]*-[A-Z]*/
# card_cava_5_pipes['header'].str.extract(r'(-[A-Z]*-[A-Z]*)$')[0].value_counts() # yes => all with 5 pipes have ALT_ in name

# # check if all rows have the same number of "-"
# card_cava_5_pipes['header'].str.count('-').value_counts() # 44237 have 4 and 845 have 6
# # check what the 6 "-" rows look like
# card_cava_5_pipes[card_cava_5_pipes['header'].str.count('-') == 6]['header'].tolist()[0] # cardiac_neuro_cava_random:ALT_IGLV3-25|ENSG00000211659.2|EH38E3470175_fwd_tile1-1_IGLV3-25|ENSG00000211659.2|EH38E3470175|22-22639049-T-C

# # investigate the different number of "-" cases
# print("header with 4 '-': ", card_cava_5_pipes[card_cava_5_pipes['header'].str.count(r'-') == 4]['header'].to_list()[4:8]) 
# print("header with 6 '-': ", card_cava_5_pipes[card_cava_5_pipes['header'].str.count(r'-') == 6]['header'].to_list()[10:14]) # some genes do have "-" in there name

# do all of them have 3 "_" between 2 and 3
# list(card_cava_5_pipes['header'].str.split('|'))


header
True    45082
Name: count, dtype: int64

In [33]:
# format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`
# 1: pre_header column: header without the label
# 2: "additional_gene_association" column: split the pre_header column by "~" and take the second element
# 3: tile info column: tile info (split "additional_gene_association" by "_" and take the last element) + remove this from the column
# 4: strand info column: strand info (split "additional_gene_association" by "_" and take again the last element) + remove this from the column
# 5: remove the "additional_gene_association" column from the pre_header column (split by "~" and take the first element)
# 6: pre_gene_name + ensembl_id + enhancer_id: split the pre_header column by "|" and take the first 3 elements (expand = True)
# 7: remove the pre_header column
# 8: sequence_type: put "region" if pre_gene_name does not contain "_" otherwise split the pre_gene_name column by "_" and take the first element

# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)'

# Extract values into new columns
df_extracted = card_cava_5_pipes['header'].str.extract(pattern)

# # Concatenate the extracted columns with the original DataFrame
card_cava_5_pipes = pd.concat([card_cava_5_pipes, df_extracted], axis=1)
print(card_cava_5_pipes.head())
# gene_name == gene_name2
## checking if same gene name or enhancer or ensembl id for all sequences => no, but for the majority
#! Might be interesting for the resulting table
# df_extracted["same_gene_names"] = df_extracted["gene_name"] == df_extracted["gene_name2"]
# print(len(df_extracted) - df_extracted["same_gene_names"].sum())
# df_extracted["same_enhancer"] = df_extracted["enhancer_id"] == df_extracted["enhancer_id2"] 
# print(len(df_extracted) - df_extracted["same_enhancer"].sum())
# df_extracted["same_ensemble"] = df_extracted["ensembl_id"] == df_extracted["ensembl_id2"]
# print(len(df_extracted) - df_extracted["same_ensemble"].sum())
# df_extracted[df_extracted["gene_name"] != df_extracted["gene_name2"]]

,label,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info,gene_name2,ensembl_id2,enhancer_id2,chrom2,pos2,ref2,alt2
27482,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778471,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778471,1,2179591,T,C
27483,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778490,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778490,1,2191444,G,A
27484,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778492,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778492,1,2192015,G,T
27485,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E1311587,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E1311587,1,2192366,T,G
27486,cardiac_neuro_cava_random,SKI,ENSG00000157933.11,EH38E2778494,fwd,tile1-1,SKI,ENSG00000157933.11,EH38E2778494,1,2193142,G,A


#### 7 pipe examples (471) 
- all alt 471
- example: 
    - cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G
    - cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G
- "~" differenciates same region for different gene
- "_" differenciates same region for region and variant id
- variant is indicated by chr-pos-ref-alt

- format: `<label> : <sequence_type (all ALT)> _ <gene name> | <ensembl id> | <enhancer/encode id> ~ <gene name (additional gene association) | <ensembl id> | <enhancer/encode id> _ <fwd/rev> _ <tile info> _ <gene name> | <ensembl id> | <enhancer/encode id> | <variant info>`

- `pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'` 

In [31]:
num_pipes = 7
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[23]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832509~NRAS|ENSG00000213281.5|EH38E2832509_rev_tile1-1_NRAS|ENSG00000213281.5|EH38E2832509|1-114691158-A-G'
# 'cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937979~SLC25A22|ENSG00000177542.11|EH38E2937979_rev_tile1-1_SLC25A22|ENSG00000177542.11|EH38E2937979|11-745581-A-G'


# # investigate and find pattern
card_cava_7_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 471
# # card_cava_7_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# does each row has one ~
print("Check if all have one ~:", sum(card_cava_7_pipes['header'].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"
print("Check if all have one ~ between second and third '|': ", sum(card_cava_7_pipes['header'].str.split(r'\|').str[2].str.count('~') == 1)) # 471 all have one ~ between 2 and third "|"

# how many contain "rev" and how many contain "fwd"
print("All headers with rev: ", sum(card_cava_7_pipes['header'].str.count('rev'))) # 392


# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'

# Extract values into new columns
df_extracted = card_cava_7_pipes['header'].str.extract(pattern)
df_extracted.head()


Check if all have one ~: 471
Check if all have one ~ between second and third '|':  471
All headers with rev:  392


,label,gene_name,ensembl_id,enhancer_id,gene_name2,ensembl_id2,enhancer_id2,fwd_rev,tile_info,gene_name3,ensembl_id3,enhancer_id3,chrom3,pos3,ref3,alt3
30703,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832509,NRAS,ENSG00000213281.5,EH38E2832509,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832509,1,114691158,A,G
30705,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832510,NRAS,ENSG00000213281.5,EH38E2832510,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832510,1,114691688,G,A
30706,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832510,NRAS,ENSG00000213281.5,EH38E2832510,rev,tile1-1,CSDE1,ENSG00000009307.17,EH38E2832510,1,114691802,G,C
30707,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832511,NRAS,ENSG00000213281.5,EH38E2832511,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832511,1,114692408,G,A
30708,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832511,NRAS,ENSG00000213281.5,EH38E2832511,rev,tile1-1,NRAS,ENSG00000213281.5,EH38E2832511,1,114692413,C,T


#### 8 pipe example 596
- 596 alt
-'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'
- 
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)~(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'


In [22]:
num_pipes = 8
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[200]
# 'cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C'

# investigate and find pattern
card_cava_8_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# # does all of these rows have ALT_ after the first ":" in the header?
# card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 596
# # card_cava_8_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# check if all have one ~
print("Check if all have one ~:", sum(card_cava_8_pipes['header'].str.count('~') == 1)) # 596 all have one ~ between 2 and third "|"
print("Check if all have one ~ between 5 and 6 '|': ", sum(card_cava_8_pipes['header'].str.split(r'\|').str[5].str.count('~') == 1)) # 596 all have one ~ 
# check if all have same number of "_"
print("Check if all have three _ between 2 and 3 '|': ", sum(card_cava_8_pipes['header'].str.split(r'\|').str[2].str.count('_') == 3))
print("Number of tile: ", sum(card_cava_8_pipes['header'].str.count('tile'))) # expected: 596 given: 596

# Define the regular expression pattern
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)_(?P<fwd_rev>.*?)_(?P<tile_info>.*?)_(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)\|(?P<chrom2>.*?)-(?P<pos2>.*?)-(?P<ref2>.*?)-(?P<alt2>.*)~(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)'
# Extract values into new columns
df_extracted = card_cava_8_pipes['header'].str.extract(pattern)
df_extracted.head()

Check if all have one ~: 596
Check if all have one ~ between 5 and 6 '|':  596
Check if all have three _ between 2 and 3 '|':  596
Number of tile:  596


,label,gene_name,ensembl_id,enhancer_id,fwd_rev,tile_info,gene_name2,ensembl_id2,enhancer_id2,chrom2,pos2,ref2,alt2,gene_name3,ensembl_id3,enhancer_id3,chrom3,pos3,ref3,alt3
35409,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596480,T,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596480,T,C
35410,cardiac_neuro_cava_random,DEAF1,ENSG00000177030.19,EH38E2937745,rev,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596480,T,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596480,T,C
35411,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596499,G,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596499,G,C
35412,cardiac_neuro_cava_random,DEAF1,ENSG00000177030.19,EH38E2937745,rev,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596499,G,C,DRD4,ENSG00000069696.7,EH38E2937745,11,596499,G,C
35415,cardiac_neuro_cava_random,DRD4,ENSG00000069696.7,EH38E2937745,fwd,tile1-1,DEAF1,ENSG00000177030.19,EH38E2937745,11,596672,G,T,DRD4,ENSG00000069696.7,EH38E2937745,11,596672,G,T


#### 10 pipes in example 309
- 309 alt
- example: 
  - 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'
  - 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832521~NRAS|ENSG00000213281.5|EH38E2832521_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832521|1-114716340-T-C~NRAS|ENSG00000213281.5|EH38E2832521|1-114716340-T-C'
- do all have "~" between "|" 2 and 3? yes
- do all have "_" between "|" 4 and 5? yes 
- do all have "~" between "|" 7 and 8? yes
- pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev2>.*?)_(?P<tile_info2>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)~(?P<gene_name4>.*?)\|(?P<ensembl_id4>.*?)\|(?P<enhancer_id4>.*?)\|(?P<chrom4>.*?)-(?P<pos4>.*?)-(?P<ref4>.*?)-(?P<alt4>.*)'

In [30]:
num_pipes = 10
# show me an example with num_pipes pipes in the header
cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes]['header'].tolist()[20]
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G'
# 'cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832521~NRAS|ENSG00000213281.5|EH38E2832521_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832521|1-114716340-T-C~NRAS|ENSG00000213281.5|EH38E2832521|1-114716340-T-C'

# investigate and find pattern
card_cava_10_pipes = cardiac_neuro_cava_random[cardiac_neuro_cava_random['header'].str.count(r'\|') == num_pipes] 
# does all of these rows have ALT_ after the first ":" in the header?
card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('ALT_').value_counts() # 309
# card_cava_10_pipes['header'].str.split(':').str[1].str.startswith('REF_').value_counts() # no ref

# - do all have "~" between "|" 2 and 3?
print("Check if all have one ~ between 2 and 3 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[2].str.count('~') == 1)) # 309 all have one ~ 

# - do all have "_" between "|" 4 and 5?
print("Check if all have three _ between 4 and 5 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[4].str.count('_') == 3))

# - do all have "~" between "|" 7 and 8? 
print("Check if all have one ~ between 7 and 8 '|': ", sum(card_cava_10_pipes['header'].str.split(r'\|').str[7].str.count('~') == 1)) # 309 all have one ~ 

# define regex
pattern = r'(?P<label>.*):ALT_(?P<gene_name>.*?)\|(?P<ensembl_id>.*?)\|(?P<enhancer_id>.*?)~(?P<gene_name2>.*?)\|(?P<ensembl_id2>.*?)\|(?P<enhancer_id2>.*?)_(?P<fwd_rev2>.*?)_(?P<tile_info2>.*?)_(?P<gene_name3>.*?)\|(?P<ensembl_id3>.*?)\|(?P<enhancer_id3>.*?)\|(?P<chrom3>.*?)-(?P<pos3>.*?)-(?P<ref3>.*?)-(?P<alt3>.*)~(?P<gene_name4>.*?)\|(?P<ensembl_id4>.*?)\|(?P<enhancer_id4>.*?)\|(?P<chrom4>.*?)-(?P<pos4>.*?)-(?P<ref4>.*?)-(?P<alt4>.*)'

# Extract values into new columns
df_extracted = card_cava_10_pipes['header'].str.extract(pattern)
df_extracted.head()

Check if all have one ~ between 2 and 3 '|':  309
Check if all have three _ between 4 and 5 '|':  309
Check if all have one ~ between 7 and 8 '|':  309


,label,gene_name,ensembl_id,enhancer_id,gene_name2,ensembl_id2,enhancer_id2,fwd_rev2,tile_info2,gene_name3,...,pos3,ref3,alt3,gene_name4,ensembl_id4,enhancer_id4,chrom4,pos4,ref4,alt4
30689,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832494,NRAS,ENSG00000213281.5,EH38E2832494,rev,tile1-1,CSDE1,...,114668983,A,G,NRAS,ENSG00000213281.5,EH38E2832494,1,114668983,A,G
30690,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E1378368,NRAS,ENSG00000213281.5,EH38E1378368,rev,tile1-1,CSDE1,...,114669283,T,C,NRAS,ENSG00000213281.5,EH38E1378368,1,114669283,T,C
30691,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670761,G,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670761,G,C
30692,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670764,T,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670764,T,C
30693,cardiac_neuro_cava_random,CSDE1,ENSG00000009307.17,EH38E2832496,NRAS,ENSG00000213281.5,EH38E2832496,rev,tile1-1,CSDE1,...,114670766,G,C,NRAS,ENSG00000213281.5,EH38E2832496,1,114670766,G,C


### Conclusion:

cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1
cardiac_neuro_cava_random:ALT_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1_CSDE1|ENSG00000009307.17|EH38E2832494|1-114668983-A-G~NRAS|ENSG00000213281.5|EH38E2832494|1-114668983-A-G
cardiac_neuro_cava_random:ALT_DRD4|ENSG00000069696.7|EH38E2937745_fwd_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937745|11-596480-T-C~DRD4|ENSG00000069696.7|EH38E2937745|11-596480-T-C
cardiac_neuro_cava_random:REF_CSDE1|ENSG00000009307.17|EH38E2832494~NRAS|ENSG00000213281.5|EH38E2832494_rev_tile1-1
cardiac_neuro_cava_random:CSDE1|ENSG00000009307.17|EH38E1378377~NRAS|ENSG00000213281.5|EH38E1378377_rev_tile1-1

<label>:[ALT_/REF_]*<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>[][~<gene_name>|<ensembl_id>|<enhancer/encodeID_tile id>]*_<fwd/rev>_<tile_info>

### Prepare a tsv of all labels and get the number of sequences

In [18]:
# write sequences with same label in one file
# get list of all labels in dataframe
labels = design_df['label'].unique().tolist()
group_list_output_dir = config['general']['group_lists_directory']

for label_of_interest in labels: 
    # get the rows of the dataframe with the label of interest
    label_of_interest_df = design_df[design_df['label'] == label_of_interest]
    # write it to directory as tsv file
    label_of_interest_df.to_csv(os.path.join(group_list_output_dir, label_of_interest + f'_{len(label_of_interest_df)}.tsv'), sep='\t', index=False)


In [19]:
# get number of all rows with a label starting with "C_"
C_labels = design_df[design_df['label'].str.startswith('C_')]
C_labels

,header,sequence,label
74811,C_positive_heart_CAD:REF_rs17114036,AGGACCGGATCAACTAGGAAGCAGGTCATAATTAGTGATAGTCATT...,C_positive_heart_CAD
74812,C_positive_heart_CAD:REF_rs72664324,AGGACCGGATCAACTTCCTCTGCTGAACCCACAGCAATGGCAGCCG...,C_positive_heart_CAD
74813,C_positive_heart_CAD:REF_rs12740374,AGGACCGGATCAACTTGACCCAAAAGTGCTTCATTTTTCGTGCCCG...,C_positive_heart_CAD
74814,C_positive_heart_CAD:REF_rs4450010,AGGACCGGATCAACTTGAGGTCCAAGGATGTGAGAGTGACCACAGT...,C_positive_heart_CAD
74815,C_positive_heart_CAD:REF_rs34091558,AGGACCGGATCAACTCTTCTCGGCCAATGAAGGGTCAACTCCATTG...,C_positive_heart_CAD
...,...,...,...
77723,C_SLEA:SLEA_hg18:chr9:82902419-82902586|6:V_Rx...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGGGCCGTGACCCCGT...,C_SLEA
77724,C_SLEA:SLEA_hg18:chr9:82902419-82902586|7:V_AH...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCGGGGATCGCGTGC...,C_SLEA
77725,C_SLEA:SLEA_hg18:chr9:82902419-82902586|80:V_H...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAGGCAAGAAGTG...,C_SLEA
77726,C_SLEA:SLEA_hg18:chr9:82902419-82902586|8:V_HN...,AGGACCGGATCAACTTAACTTCCAAGAGGCAGCCAAGGTCCAGGTG...,C_SLEA


## Use vcf files to get the AF of the genes
- get the vcf files to the analysis (analyze_NGN2_feather.py)

## Use sequences of the deduplicated header and look for them in the duplicated header
- if you find a match take the header and make a list

In [20]:

# load duplicate and deduplicate fasta 
design_duplicates_fasta = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design.fa"
design_deduplicate_fasta = "/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/design_no_duplicates_sequence_and_header.fa"
# load the fasta: 
design_dup_df = create_fasta_df_from_one_line_sequence_fasta(design_duplicates_fasta)
design_dedup_df = create_fasta_df_from_one_line_sequence_fasta(design_deduplicate_fasta)

In [72]:
design_dup_df[col_sequence].astype(str)[15:285]

15     AGGACCGGATCAACTGGTTTGCTGTGGCCTGGCTCGATTGAGAATC...
16     AGGACCGGATCAACTCGGGATGGTGCCCGTGGCATCTTCTGCTCGG...
17     AGGACCGGATCAACTCTTGAACTCCTGACCTTGTGAGCTACCCACC...
18     AGGACCGGATCAACTGCAGTCTGCTTTTGGGCCTGTAGATTCGTTG...
19     AGGACCGGATCAACTCCACAGGCCAGGGCACTCCCAACAGCACTGC...
                             ...                        
280    AGGACCGGATCAACTGGGTCATCCAAGGTCCCAGGATCCAGCTCAT...
281    AGGACCGGATCAACTTGTGCGTGATCACCTGTGTAGCTCCTGGAGG...
282    AGGACCGGATCAACTTAAATTAAAATAAATAAACATTTAAAAATTA...
283    AGGACCGGATCAACTCATCAGAGTAGGTGAGACCACCTAGGGAAGA...
284    AGGACCGGATCAACTAGTACTTTGTGTTCTCATGTGACAATGGGCA...
Name: sequence, Length: 270, dtype: object

In [63]:
design_dup_df[col_sequence] = design_dup_df[col_sequence].astype(str)[15:285]
design_dedup_df[col_sequence] = design_dedup_df[col_sequence].astype(str)[15:285]

# # sort both files using their sequence
design_dup_df.sort_values(by=[col_sequence], inplace=True)
design_dedup_df.sort_values(by=[col_sequence], inplace=True)

In [64]:
design_dup_df

,header,sequence
227,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,AGGACCGGATCAACTAAAAATTTTTAAGGGAATTTTAAGTGTGAAA...
135,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,AGGACCGGATCAACTAAAAGAGAGACAACTACTGCTTTTACTTCAG...
211,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,AGGACCGGATCAACTAAACAGCCATTACTACTTTAATAGAGCAGAG...
265,cardiac_neuro_cava_random:PBX1|ENSG00000185630...,AGGACCGGATCAACTAAAGAAGACAGATTATTCCCCAGTTGCCAAC...
124,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,AGGACCGGATCAACTAAAGCTCTGGGGTGGGAAAAGGGCTCCCAAG...
...,...,...
80801,MK:tile_2240|chr1-116244322+116244591|scramble...,NaN
80802,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,NaN
80803,MK:tile_18415|chr17-71181691+71181960|scramble...,NaN
80804,MK:tile_14356|chr15-67031618+67031887|scramble...,NaN


In [49]:
idx_dup = 0
idx_dedup = 0
duplicated_sequences_list = []
duplicate = False
while idx_dup < len(design_dup_df) and idx_dedup < len(design_dedup_df):
    if  design_dup_df.iloc[idx_dup][col_sequence] == design_dedup_df.iloc[idx_dedup][col_sequence]:
        if (duplicate):
            # append to list
            duplicated_sequences_list[-1][2].append(design_dup_df.iloc[idx_dup]['header'])
        else:
            # create new list
            duplicated_sequences_list.append((design_dedup_df.iloc[idx_dedup]['header'], design_dedup_df.iloc[idx_dedup][col_sequence], [design_dup_df.iloc[idx_dup]['header']]))
            duplicate = True
        idx_dedup += 1
    elif design_dup_df.iloc[idx_dup][col_sequence] < design_dedup_df.iloc[idx_dedup][col_sequence]:
        idx_dup += 1
        duplicate = False
    else:
        idx_dedup += 1
    
if len(design_dedup_df) != len(duplicated_sequences_list):
    for i in range(len(design_dedup_df)):
        if design_dedup_df[i] != duplicated_sequences_list[i][0]:
            print("Missing: " + str(design_dedup_df[i]))
            exit(1)

TypeError: '<' not supported between instances of 'str' and 'float'

In [43]:
duplicated_sequences_list

[('cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T',
  'AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCAAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA',
  ['cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T']),
 ('cardiac_neuro_cava_random:REF_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1',
  'AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCTAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA',
  ['cardiac_neuro

In [46]:
print(len(duplicated_sequences_list))

count_merge = 0
for tuple in duplicated_sequences_list:
    print(tuple[0])
    print(tuple[1])
    print(tuple[2])
    break
    if len(tuple[2]) >= 2:
        count_merge += 1
        
print("Mergings in the design: ", count_merge)

80215
cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T
AGGACCGGATCAACTAAAAAAAAAACACAAAAAACAAAAAACAAAAAAAACCTCTTCCTTTGCTTCCATCCAAATACAGTCGTGCATCATTTAACAATGGTGATGCATTCTGAGAAATGTGTTGTTAGGCAGTTTCTTCGTGGTGTAAACATCACAGAGTGCACTCACACAAACCAAGAATGGTATGGCCTATTGCTCCAAGACTACAAACCTGTTCAGATGTTACTGTACTGACTTGTAGGCAACAGTAACACAATGGTATGTATTTGTGTATCTAAACATACACATTGCGTGAACCGA
['cardiac_neuro_cava_random:ALT_NCKAP1|ENSG00000061676.16|EH38E2057960_rev_tile1-1_NCKAP1|ENSG00000061676.16|EH38E2057960|2-183088247-A-T']
Mergings in the design:  0
